# Fitbit 데이터 분석 (삼성헬스 분석 파이프라인의 외적 재현성 점검)

- 데이터: 설문 응답 + Fitbit 웨어러블 원자료 (전처리 완료 후 `data/` 폴더의 최소 파일로 대체됨)
- 목표: 삼성헬스에서 진행한 자기보고–웨어러블 간극 분석 파이프라인을 독립 표본(Fitbit)에 적용해 외적 재현성을 점검

## 1~2단계: 데이터 로드 (전처리 완료본)

원본 설문과 Fitbit 웨어러블 원자료의 전처리 과정 — 설문·웨어러블 매칭, 주차별 [D-7,D-1] 윈도우 커버리지 계산, steps/heartrate/sleep person-week 집계, 완전중복행 제거, mainSleep 필터링 — 은 사전에 실행되어 다음 2개의 최종 데이터로 정리되어 있으며, 이 노트북은 원본 파싱·집계 코드 없이 이 데이터를 불러오는 것에서 시작한다.

- `data/fitbit_survey_person_week.csv`: 참여자×주차 단위 설문 응답(q1~q9, mq1~mq4) + gender + stress/mood 파생값 (id, survey_date 기준 386행, 42명)
- `data/fitbit_wearable_person_week.csv`: 참여자×주차 단위 웨어러블 파생 지표 (수면/걷기/취침·기상 시각 원자료 + canonical coverage_ratio)
- `data/fitbit_daily_mood.csv`: 참여자×일 단위 daily mood 원문항 응답 (Table 10 재현용)

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
warnings.showwarning = lambda *args, **kwargs: None  # statsmodels 내부에서 필터가 재설정되어도 경고 출력을 막음
pd.set_option('display.width', 200)

DATA = "data"
RESULT = "results"
os.makedirs(RESULT, exist_ok=True)

survey_pw = pd.read_csv(os.path.join(DATA, 'fitbit_survey_person_week.csv'))
wearable_pw = pd.read_csv(os.path.join(DATA, 'fitbit_wearable_person_week.csv'))
survey_pw['survey_date'] = pd.to_datetime(survey_pw['survey_date'])
wearable_pw['survey_date'] = pd.to_datetime(wearable_pw['survey_date'])

# 이후 단계가 읽는 d2_merged_person_week.csv 구성으로 병합
base_survey_cols = ['id', 'survey_date', 'q1-1', 'q1-2', 'q2', 'q3', 'q4', 'q5', 'q6',
                     'q6-1-1', 'q6-1-2', 'q6-1-3', 'q6-1-4',
                     'q6-2-1', 'q6-2-2', 'q6-2-3', 'q6-2-4',
                     'q6-3-1', 'q6-3-2', 'q6-3-3', 'q6-3-4',
                     'q7', 'q7-1', 'q7-2', 'q8-1', 'q8-2', 'q9',
                     'mq1', 'mq2', 'mq3', 'mq4']
base_wearable_cols = ['id', 'survey_date', 'wearable_coverage', 'w_mean_steps', 'w_mean_hr',
                       'w_mean_sleep_minutes', 'w_mean_sleep_score', 'w_sleep_nights_n']

mw = survey_pw[base_survey_cols].merge(wearable_pw[base_wearable_cols], on=['id', 'survey_date'], how='inner')
mw.to_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'), index=False)

both = sorted(mw['id'].unique())
print(f"대상(설문+웨어러블 둘 다 있음): {len(both)}명")
print(f"person-week: {len(mw)}건")


대상(설문+웨어러블 둘 다 있음): 42명
person-week: 386건


## 3단계: 수면시간 Gap / 수면질 근사 Gap 생성
- 수면시간: 삼성헬스의 D-1A 방식(|D|/SD(D), SD는 person-level 단순평균 기반 canonical 상수)
- 수면질(근사): 삼성헬스의 D-1B 방식(Zs-Zw). Fitbit에는 "수면질" 단일문항이 없어 불면 문항(q4+q5, 역채점) vs wearable `overall_score`로 근사 — 삼성헬스와 동일 문항이 아니므로 근사치이다

In [2]:
from scipy import stats

MIN_WEEKS = 3
m = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
print(f"입력 person-week: {len(m)}건, {m['id'].nunique()}명")

# ============================================================
# 1) 수면시간 Gap (삼성헬스의 D-1A 방식: |D|/SD(D), SD는 person-level 단순평균 기반 canonical 상수)
# ============================================================
m['survey_sleep_hr'] = pd.to_numeric(m['q1-1'], errors='coerce') + pd.to_numeric(m['q1-2'], errors='coerce') / 60
m['wearable_sleep_hr'] = m['w_mean_sleep_minutes'] / 60
m['D_sleep_hr'] = m['survey_sleep_hr'] - m['wearable_sleep_hr']

valid_hr = m.dropna(subset=['D_sleep_hr'])
person_mean_D = valid_hr.groupby('id')['D_sleep_hr'].agg(['mean', 'count'])
person_mean_D = person_mean_D[person_mean_D['count'] >= MIN_WEEKS]
sd_D = person_mean_D['mean'].std()
print(f"\n[수면시간] canonical SD(D) = {sd_D:.4f} (person-level 평균 기반, n_person={len(person_mean_D)}, MIN_WEEKS>={MIN_WEEKS})")

m['absGap_sleeptime'] = (m['D_sleep_hr'] / sd_D).abs()

# ============================================================
# 2) 수면질 근사 Gap (삼성헬스의 D-1B 방식: Zs-Zw, 불면문항 역방향 합산 vs wearable score)
# ============================================================
m['insomnia_sum'] = pd.to_numeric(m['q4'], errors='coerce') + pd.to_numeric(m['q5'], errors='coerce')
m['Squal'] = -m['insomnia_sum']  # 역채점: 값이 클수록 불면 적음(=주관적 수면질 좋음)

valid_q = m.dropna(subset=['Squal', 'w_mean_sleep_score'])
pcount = valid_q.groupby('id').size()
valid_ids = pcount[pcount >= MIN_WEEKS].index
person_S = valid_q[valid_q['id'].isin(valid_ids)].groupby('id')['Squal'].mean()
person_W = valid_q[valid_q['id'].isin(valid_ids)].groupby('id')['w_mean_sleep_score'].mean()
mean_S, sd_S = person_S.mean(), person_S.std()
mean_W, sd_W = person_W.mean(), person_W.std()
print(f"[수면질] canonical 상수 - S: mean={mean_S:.3f},sd={sd_S:.3f} / W: mean={mean_W:.3f},sd={sd_W:.3f} (n_person={len(valid_ids)})")

m['Zs_qual'] = (m['Squal'] - mean_S) / sd_S
m['Zw_qual'] = (m['w_mean_sleep_score'] - mean_W) / sd_W
m['R_qual'] = m['Zs_qual'] - m['Zw_qual']
m['absGap_sleepquality'] = m['R_qual'].abs()

# ============================================================
# 3) 기초 확인: Gap 분포 + 자기보고-웨어러블 상관관계(스피어만)
# ============================================================
print("\n=== 수면시간 Gap(absGap_sleeptime) 분포 ===")
print(m['absGap_sleeptime'].describe())
n_valid_hr = m['absGap_sleeptime'].notna().sum()
print(f"유효 person-week: {n_valid_hr}/{len(m)} ({n_valid_hr/len(m)*100:.1f}%)")

rho, p = stats.spearmanr(m['survey_sleep_hr'], m['wearable_sleep_hr'], nan_policy='omit')
print(f"survey_sleep_hr vs wearable_sleep_hr 스피어만 상관: rho={rho:.3f}, p={p:.4f}")

print("\n=== 수면질 근사 Gap(absGap_sleepquality) 분포 ===")
print(m['absGap_sleepquality'].describe())
n_valid_q = m['absGap_sleepquality'].notna().sum()
print(f"유효 person-week: {n_valid_q}/{len(m)} ({n_valid_q/len(m)*100:.1f}%)")

rho2, p2 = stats.spearmanr(m['Squal'], m['w_mean_sleep_score'], nan_policy='omit')
print(f"Squal(역채점 불면) vs wearable_sleep_score 스피어만 상관: rho={rho2:.3f}, p={p2:.4f}")

m.to_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'), index=False)
print("\n저장 완료: d2_sleep_gap_pw.csv")

입력 person-week: 386건, 42명

[수면시간] canonical SD(D) = 0.8377 (person-level 평균 기반, n_person=29, MIN_WEEKS>=3)
[수면질] canonical 상수 - S: mean=-2.733,sd=0.941 / W: mean=75.831,sd=4.124 (n_person=29)

=== 수면시간 Gap(absGap_sleeptime) 분포 ===
count    241.000000
mean       1.186578
std        1.020031
min        0.000000
25%        0.352419
50%        0.935047
75%        1.822348
max        5.331760
Name: absGap_sleeptime, dtype: float64
유효 person-week: 241/386 (62.4%)
survey_sleep_hr vs wearable_sleep_hr 스피어만 상관: rho=0.474, p=0.0000

=== 수면질 근사 Gap(absGap_sleepquality) 분포 ===
count    242.000000
mean       1.533179
std        1.202109
min        0.010510
25%        0.495504
50%        1.258760
75%        2.361068
max        5.180932
Name: absGap_sleepquality, dtype: float64
유효 person-week: 242/386 (62.7%)
Squal(역채점 불면) vs wearable_sleep_score 스피어만 상관: rho=-0.268, p=0.0000

저장 완료: d2_sleep_gap_pw.csv


## 4단계: 일치도분석 / 스피어만 / 반복측정 관계분석(FDR)
- 삼성헬스에서 진행한 방법론 그대로: 일치도분석(Bland-Altman) + 스피어만 상관 + 반복측정(mixedlm) 관계분석, family 단위 FDR 보정
- 지표마다 수집 비율이 다르므로, 각 지표는 그 지표에서 실제 유효한 인원/주만 사용(dropna 후 진행)

In [3]:
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

RNG = np.random.default_rng(42)
m = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))
print(f"입력 person-week: {len(m)}건, {m['id'].nunique()}명")
print("(참고: 모든 컬럼이 동일 수집률이 아니므로, 각 지표마다 그 지표에서 실제 유효한 인원/주만 사용)")

# ============================================================
# 0) 유틸: ICC(2,1) + 참여자단위 군집부트스트랩 95%CI (5·6단계에서도 동일하게 사용)
# ============================================================
def icc_2_1(x, y):
    data = np.column_stack([x, y])
    n, k = data.shape
    grand_mean = data.mean()
    row_means, col_means = data.mean(axis=1), data.mean(axis=0)
    SST = ((data - grand_mean) ** 2).sum()
    SSR = k * ((row_means - grand_mean) ** 2).sum()
    SSC = n * ((col_means - grand_mean) ** 2).sum()
    SSE = SST - SSR - SSC
    MSR, MSC, MSE = SSR/(n-1), SSC/(k-1), SSE/((n-1)*(k-1))
    return (MSR - MSE) / (MSR + (k-1)*MSE + k*(MSC-MSE)/n)

def cluster_bootstrap_icc(dd, scol, wcol, n_boot=1000):
    """참여자단위 군집부트스트랩으로 ICC·bias(scol-wcol) 95%CI를 함께 산출 (5·6단계에서도 동일하게 사용)."""
    ids = dd['id'].unique()
    iccs, biases = [], []
    for i in range(n_boot):
        sample_ids = RNG.choice(ids, size=len(ids), replace=True)
        boot = pd.concat([dd[dd['id'] == sid] for sid in sample_ids], ignore_index=True)
        if boot['id'].nunique() < 3:
            continue
        try:
            iccs.append(icc_2_1(boot[scol].values, boot[wcol].values))
        except Exception:
            pass
        biases.append((boot[scol] - boot[wcol]).mean())
    return (np.percentile(iccs, 2.5), np.percentile(iccs, 97.5)), \
           (np.percentile(biases, 2.5), np.percentile(biases, 97.5))

results = []

# ============================================================
# 1) 수면시간: survey_sleep_hr vs wearable_sleep_hr (동일 단위: 시간)
# ============================================================
d1 = m.dropna(subset=['survey_sleep_hr', 'wearable_sleep_hr']).copy()
print(f"\n=== [수면시간] 유효 n={len(d1)} ({d1['id'].nunique()}명) ===")

# 1-1. 일치도분석 (Bland-Altman: bias, 95% LOA, ±60분 목표일치율)
diff = d1['survey_sleep_hr'] - d1['wearable_sleep_hr']
bias, sd_diff = diff.mean(), diff.std()
loa_low, loa_high = bias - 1.96*sd_diff, bias + 1.96*sd_diff
within_60min = (diff.abs() <= 1.0).mean()  # 60분=1시간 이내
print(f"[일치도] bias(자기보고-웨어러블)={bias:.3f}시간, 95% LOA=[{loa_low:.3f}, {loa_high:.3f}], ±60분 이내 일치율={within_60min*100:.1f}%")

# 1-2. ICC(2,1) 절대일치도 + 군집부트스트랩 95%CI (bias CI도 함께 산출)
icc1 = icc_2_1(d1['survey_sleep_hr'].values, d1['wearable_sleep_hr'].values)
icc1_ci, bias1_ci = cluster_bootstrap_icc(d1, 'survey_sleep_hr', 'wearable_sleep_hr')
print(f"[ICC(2,1) 절대일치도] {icc1:.3f} [군집부트스트랩95%CI: {icc1_ci[0]:.3f}, {icc1_ci[1]:.3f}]")
print(f"[bias 군집부트스트랩95%CI] [{bias1_ci[0]:.3f}, {bias1_ci[1]:.3f}]")

# 1-3. 스피어만(보조지표, 반복측정 구조 미반영)
rho, p_rho = stats.spearmanr(d1['survey_sleep_hr'], d1['wearable_sleep_hr'])
print(f"[스피어만, 보조지표] rho={rho:.3f}, p={p_rho:.4f}")

# 1-4. 반복측정 관계분석 (mixedlm, 참여자 임의절편, 공식추론)
fit1 = smf.mixedlm('wearable_sleep_hr ~ survey_sleep_hr', d1, groups=d1['id']).fit(reml=True)
coef1, p1 = fit1.params['survey_sleep_hr'], fit1.pvalues['survey_sleep_hr']
print(f"[반복측정, 공식추론] wearable ~ survey: coef={coef1:.3f}, p={p1:.4f} (n={len(d1)}, n_id={d1['id'].nunique()})")
results.append({'지표': '수면시간', 'n': len(d1), 'n_id': d1['id'].nunique(), 'bias': bias,
                 'bias_ci_low': bias1_ci[0], 'bias_ci_high': bias1_ci[1],
                 'icc': icc1, 'icc_ci_low': icc1_ci[0], 'icc_ci_high': icc1_ci[1],
                 'rho': rho, 'p_rho': p_rho, 'coef': coef1, 'p': p1})

# ============================================================
# 2) 수면질 근사: Squal(역채점 불면) vs w_mean_sleep_score (척도 상이 -> Z score 기반)
# ============================================================
d2 = m.dropna(subset=['Squal', 'w_mean_sleep_score', 'R_qual']).copy()
print(f"\n=== [수면질 근사] 유효 n={len(d2)} ({d2['id'].nunique()}명) ===")
print("(척도가 다른 두 변수라 원단위(Squal/score) 일치도분석은 부적절 -> Z-score(Zs-Zw=R_qual) 기준으로 bias/ICC 산출)")

bias2 = d2['R_qual'].mean()  # R_qual = Zs_qual - Zw_qual (자기보고Z - 웨어러블Z, 방향 명시)
sd2 = d2['R_qual'].std()
print(f"[일치도, Z-score 기준] bias(자기보고Z-웨어러블Z)={bias2:.3f}, 95%LOA=[{bias2-1.96*sd2:.3f},{bias2+1.96*sd2:.3f}]")

# 주의: Squal(약 -7~-2)과 w_mean_sleep_score(약 62~87)는 척도가 완전히 달라
# 원척도 그대로 ICC를 계산하면 척도차이(평균차) 자체가 분산을 지배해 ICC가 항상 0 근처로
# 수학적으로 눌리는 인공물이 발생함(실측 확인: 원척도 ICC=-0.0003, 매우 좁은 CI로 고정).
# 그래서 Zs_qual/Zw_qual(각각 Z-score화)로 계산한다. 이 값은 표준화 척도에서의 상대적 대응 정도를 보는 탐색값이며 원척도 절대일치도를 의미하지 않는다.
icc2 = icc_2_1(d2['Zs_qual'].values, d2['Zw_qual'].values)
icc2_ci, bias2_ci = cluster_bootstrap_icc(d2, 'Zs_qual', 'Zw_qual')
print(f"[ICC(2,1), Z-score 기준(표준화 척도의 상대적 대응, 원척도 절대일치도 아님)] {icc2:.3f} [군집부트스트랩95%CI: {icc2_ci[0]:.3f}, {icc2_ci[1]:.3f}]")
print(f"[bias(Zs-Zw) 군집부트스트랩95%CI] [{bias2_ci[0]:.3f}, {bias2_ci[1]:.3f}]")

rho2, p_rho2 = stats.spearmanr(d2['Squal'], d2['w_mean_sleep_score'])
print(f"[스피어만, 보조지표] rho={rho2:.3f}, p={p_rho2:.4f}")

fit2 = smf.mixedlm('w_mean_sleep_score ~ Squal', d2, groups=d2['id']).fit(reml=True)
coef2, p2 = fit2.params['Squal'], fit2.pvalues['Squal']
print(f"[반복측정, 공식추론] wearable_score ~ Squal: coef={coef2:.3f}, p={p2:.4f} (n={len(d2)}, n_id={d2['id'].nunique()})")
results.append({'지표': '수면질(근사)', 'n': len(d2), 'n_id': d2['id'].nunique(), 'bias': bias2,
                 'bias_ci_low': bias2_ci[0], 'bias_ci_high': bias2_ci[1],
                 'icc': icc2, 'icc_ci_low': icc2_ci[0], 'icc_ci_high': icc2_ci[1],
                 'rho': rho2, 'p_rho': p_rho2, 'coef': coef2, 'p': p2})

# ============================================================
# 3) FDR 보정 (이 family: 수면시간, 수면질근사 반복측정 관계 2개 검정)
# ============================================================
res_df = pd.DataFrame(results)
rej, p_fdr, _, _ = multipletests(res_df['p'], method='fdr_bh')
res_df['p_fdr'] = p_fdr
res_df['reject_fdr'] = rej
print("\n=== 반복측정 관계분석 FDR 보정 결과 (family: 수면시간+수면질근사, 2개 검정) ===")
print(res_df.to_string(index=False))

# Spearman rho도 반복측정 관계와 동일하게 자체 family로 별도 FDR 보정을 적용한다.
rej_rho, p_fdr_rho, _, _ = multipletests(res_df['p_rho'], method='fdr_bh')
res_df['rho_p_fdr'] = p_fdr_rho
res_df['rho_reject_fdr'] = rej_rho
print("\n=== Spearman rho FDR 보정 결과 (family: 수면시간+수면질근사, 2개 검정) ===")
print(res_df[['지표', 'rho', 'p_rho', 'rho_p_fdr', 'rho_reject_fdr']].to_string(index=False))

res_df.to_csv(os.path.join(RESULT, 'd2_agreement_results.csv'), index=False)
print("\n저장 완료: d2_agreement_results.csv")


입력 person-week: 386건, 42명
(참고: 모든 컬럼이 동일 수집률이 아니므로, 각 지표마다 그 지표에서 실제 유효한 인원/주만 사용)

=== [수면시간] 유효 n=241 (34명) ===
[일치도] bias(자기보고-웨어러블)=0.491시간, 95% LOA=[-1.893, 2.876], ±60분 이내 일치율=58.9%
[ICC(2,1) 절대일치도] 0.394 [군집부트스트랩95%CI: 0.163, 0.563]
[bias 군집부트스트랩95%CI] [0.165, 0.772]
[스피어만, 보조지표] rho=0.474, p=0.0000
[반복측정, 공식추론] wearable ~ survey: coef=0.308, p=0.0000 (n=241, n_id=34)

=== [수면질 근사] 유효 n=242 (34명) ===
(척도가 다른 두 변수라 원단위(Squal/score) 일치도분석은 부적절 -> Z-score(Zs-Zw=R_qual) 기준으로 bias/ICC 산출)
[일치도, Z-score 기준] bias(자기보고Z-웨어러블Z)=0.002, 95%LOA=[-3.821,3.826]
[ICC(2,1), Z-score 기준(표준화 척도의 상대적 대응, 원척도 절대일치도 아님)] -0.243 [군집부트스트랩95%CI: -0.369, -0.069]
[bias(Zs-Zw) 군집부트스트랩95%CI] [-0.592, 0.625]
[스피어만, 보조지표] rho=-0.268, p=0.0000
[반복측정, 공식추론] wearable_score ~ Squal: coef=-0.148, p=0.6486 (n=242, n_id=34)

=== 반복측정 관계분석 FDR 보정 결과 (family: 수면시간+수면질근사, 2개 검정) ===
     지표   n  n_id     bias  bias_ci_low  bias_ci_high       icc  icc_ci_low  icc_ci_high       rho        p_rho      coef            p    

## 5단계 (2단계 심화): 취침시각/기상시각 Gap
- 웨어러블: sleep.csv 완전중복행 제거 후(다중세션은 전부 순수 중복행으로 확인됨) `startTime`/`endTime` 추출
- **원형시간(circular) 방식 사용**: 정오 기준 단순 shift는 취침시각엔 적절하지만 기상시각 분포(주로 오전 6시~정오)가 그 경계에 걸려 왜곡이 생길 수 있다(예: 실제 차이 5.7시간인 사례가 -18.3시간으로 계산됨). 특정 기준점에 의존하지 않는 **원형평균(circular mean)**과 **원형차이(circular diff, 시계상 최단거리)**를 사용한다
- **내적 일관성 필터**: q1(자기보고 수면시간)과 q3-q2(계산값)이 3시간 넘게 차이나는 행은 결측 처리. 원인은 확정할 수 없어 "12시간제 변환 오류"로 단정하지 않고 현상만 기술. 필터 임계값(없음/2h/3h/4h) 민감도 분석 및 제외행의 참여자·심리척도 쏠림 여부 확인 포함
- 삼성헬스와 동일 방법론(일치도분석+스피어만+반복측정+FDR) 적용, 회귀/상관은 모집단 기준시각 대비 편차로 인코딩(경계 왜곡 방지)

In [4]:
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

RNG = np.random.default_rng(42)

# ============================================================
# 0) 원형시간 유틸리티
#    - 정오/자정 등 특정 기준점을 고르는 대신, 어떤 분포에도 안전한 원형평균/원형차이를 사용
#    - to_raw_minutes: HH:MM -> 자정 기준 raw 분(0~1439), 별도 shift 없음
#    - circular_mean_minutes: 여러 raw 분 값의 "원형" 평균(벡터평균 후 각도 복원)
#    - circular_diff_minutes: a-b를 (-720,720]로 감아 반환(시계상 최단 차이)
# ============================================================
def to_raw_minutes(hhmm):
    try:
        h, mi = str(hhmm).split(':')
        return int(h) * 60 + int(mi)
    except Exception:
        return np.nan

def circular_mean_minutes(values):
    vals = np.array([v for v in values if pd.notna(v)], dtype=float)
    if len(vals) == 0:
        return np.nan
    theta = vals / 1440 * 2 * np.pi
    s, c = np.sin(theta).mean(), np.cos(theta).mean()
    mean_theta = np.arctan2(s, c)
    return (mean_theta / (2 * np.pi) * 1440) % 1440

def circular_diff_minutes(a, b):
    if pd.isna(a) or pd.isna(b):
        return np.nan
    return ((a - b + 720) % 1440) - 720

# ============================================================
# 1) 웨어러블 취침/기상시각 person-week 값 로드
#    (원형평균 집계는 data/fitbit_wearable_person_week.csv에 사전 계산되어 있음)
# ============================================================
mw = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
mw['survey_date'] = pd.to_datetime(mw['survey_date'])
both_ids = sorted(mw['id'].unique())

wearable_pw_bt = pd.read_csv(os.path.join(DATA, 'fitbit_wearable_person_week.csv'))
wearable_pw_bt['survey_date'] = pd.to_datetime(wearable_pw_bt['survey_date'])
bt = wearable_pw_bt[['id', 'survey_date', 'w_bedtime_raw', 'w_waketime_raw']]

m = mw.merge(bt, on=['id', 'survey_date'], how='left')

# ============================================================
# 2) 설문 취침(q2)/기상(q3)도 raw 분으로 변환, 내적 일관성 지표 계산
#    (원형 연산이라 raw 표현 그대로 사용 가능, 별도 shift 불필요)
# ============================================================
m['s_bedtime_raw'] = m['q2'].apply(to_raw_minutes)
m['s_waketime_raw'] = m['q3'].apply(to_raw_minutes)

# 자기보고 취침->기상 소요시간(원형, 자정 넘어가도 안전) vs q1(자기보고 수면시간) 비교
m['survey_dur_calc'] = ((m['s_waketime_raw'] - m['s_bedtime_raw']) % 1440) / 60
m['survey_dur_q1'] = pd.to_numeric(m['q1-1'], errors='coerce') + pd.to_numeric(m['q1-2'], errors='coerce') / 60
m['internal_inconsistency'] = (m['survey_dur_calc'] - m['survey_dur_q1']).abs()

# ============================================================
# 3) Gap 산출 함수 (특정 threshold로 필터링 후 계산) - 민감도 분석용으로 함수화
# ============================================================
def compute_gaps(m_in, threshold):
    d = m_in.copy()
    if threshold is not None:
        bad_mask = d['internal_inconsistency'] > threshold
        d.loc[bad_mask, ['s_bedtime_raw', 's_waketime_raw']] = np.nan
    else:
        bad_mask = pd.Series(False, index=d.index)

    out = {}
    for label, scol, wcol in [('bedtime', 's_bedtime_raw', 'w_bedtime_raw'),
                              ('waketime', 's_waketime_raw', 'w_waketime_raw')]:
        d[f'D_{label}_hr'] = d.apply(lambda row: circular_diff_minutes(row[scol], row[wcol]), axis=1) / 60
    return d, bad_mask.sum()

# ============================================================
# 3-1) 상관/회귀용 "레벨" 인코딩: 원형차이를 이용해 각 변수의 모집단 기준시각으로부터의
#      편차(분, -720~720)로 표현 -> 자정/정오 등 특정 경계에 걸리지 않는 안전한 선형 인코딩
#      (D = s_dev - w_dev 이므로 circular_diff 기반 D와 일관됨)
# ============================================================
REF = {}
for label in ['bedtime', 'waketime']:
    REF[label] = circular_mean_minutes(m[f'w_{label}_raw'])
    print(f"[{label}] 모집단 기준시각(원형평균) = {REF[label]/60:.2f}시(raw)")

def centered_hours(raw_series, ref):
    return raw_series.apply(lambda v: circular_diff_minutes(v, ref) / 60 if pd.notna(v) else np.nan)

# ============================================================
# 4) 민감도 분석: 내적 일관성 제외 기준(없음/2h/3h/4h)별 표본·결과 비교
# ============================================================
print("\n=== 내적 일관성 필터 민감도 분석 ===")
sens_rows = []
for thr in [None, 2.0, 3.0, 4.0]:
    d, n_excl = compute_gaps(m, thr)
    label_thr = 'None' if thr is None else f'{thr}h'
    for label in ['bedtime', 'waketime']:
        dd = d.dropna(subset=[f'D_{label}_hr']).copy()
        if len(dd) < 10:
            continue
        dd['s_hr'] = centered_hours(dd[f's_{label}_raw'], REF[label])
        dd['w_hr'] = centered_hours(dd[f'w_{label}_raw'], REF[label])
        try:
            fit = smf.mixedlm(f'w_hr ~ s_hr', dd, groups=dd['id']).fit(reml=True)
            coef, p = fit.params['s_hr'], fit.pvalues['s_hr']
        except Exception:
            coef, p = np.nan, np.nan
        bias = dd[f'D_{label}_hr'].mean()
        sens_rows.append({'threshold': label_thr, '지표': label, 'n_제외': n_excl, 'n': len(dd),
                           'n_id': dd['id'].nunique(), 'bias': bias, 'coef': coef, 'p': p})

sens_df = pd.DataFrame(sens_rows)
print(sens_df.to_string(index=False))
sens_df.to_csv(os.path.join(RESULT, 'd2_bedtime_sensitivity.csv'), index=False)

# ============================================================
# 5) 제외된 행이 특정 참여자/스트레스 수준에 쏠려있는지 확인 (3시간 기준)
# ============================================================
_, _ = compute_gaps(m, 3.0)
bad_mask_3h = m['internal_inconsistency'] > 3.0
print(f"\n=== 3시간 기준 제외 행({bad_mask_3h.sum()}건)의 참여자 분포 ===")
excl_by_id = m.loc[bad_mask_3h, 'id'].value_counts()
print(excl_by_id.head(10))
print(f"\n제외행의 평균 mq1(스트레스 관련) vs 전체 평균 비교:")
for col in ['mq1', 'mq2', 'mq3', 'mq4']:
    v_excl = pd.to_numeric(m.loc[bad_mask_3h, col], errors='coerce').mean()
    v_all = pd.to_numeric(m[col], errors='coerce').mean()
    print(f"  {col}: 제외행 평균={v_excl:.2f}, 전체 평균={v_all:.2f}")

# ============================================================
# 6) 최종 채택(3시간 기준)으로 본 분석 진행 - 삼성헬스와 동일 방법론
# ============================================================
THRESHOLD = 3.0
m, n_excl = compute_gaps(m, THRESHOLD)
print(f"\n최종 채택 기준(내적 일관성 > {THRESHOLD}시간 제외): {n_excl}건 제외")

for label in ['bedtime', 'waketime']:
    valid = m.dropna(subset=[f'D_{label}_hr'])
    pmean = valid.groupby('id')[f'D_{label}_hr'].agg(['mean', 'count'])
    pmean = pmean[pmean['count'] >= MIN_WEEKS]
    sd = pmean['mean'].std()
    m[f'absGap_{label}'] = (m[f'D_{label}_hr'] / sd).abs()
    print(f"[{label}] canonical SD(D) = {sd:.4f}시간 (n_person={len(pmean)})")

m.to_csv(os.path.join(RESULT, 'd2_bedtime_gap_pw.csv'), index=False)

def icc_2_1(x, y):
    """ICC(2,1): 이원 임의효과, 절대일치도, 단일측정 (Shrout & Fleiss 1979 ANOVA 공식).
    주의: person-week를 독립 단위로 가정 - 동일 참여자의 반복측정 군집구조는 반영하지 않음(한계로 명시)."""
    data = np.column_stack([x, y])
    n, k = data.shape
    grand_mean = data.mean()
    row_means = data.mean(axis=1)
    col_means = data.mean(axis=0)
    SST = ((data - grand_mean) ** 2).sum()
    SSR = k * ((row_means - grand_mean) ** 2).sum()
    SSC = n * ((col_means - grand_mean) ** 2).sum()
    SSE = SST - SSR - SSC
    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))
    return (MSR - MSE) / (MSR + (k - 1) * MSE + k * (MSC - MSE) / n)

def cluster_bootstrap_bedtime(dd, label, n_boot=1000):
    """참여자 단위 군집 부트스트랩으로 bias·ICC의 95% CI 산출 (반복측정 구조 반영)."""
    ids = dd['id'].unique()
    biases, iccs = [], []
    for i in range(n_boot):
        sample_ids = RNG.choice(ids, size=len(ids), replace=True)
        boot = pd.concat([dd[dd['id'] == sid] for sid in sample_ids], ignore_index=True)
        if boot['id'].nunique() < 3:
            continue
        biases.append(boot[f'D_{label}_hr'].mean())
        try:
            iccs.append(icc_2_1(boot['s_hr'].values, boot['w_hr'].values))
        except Exception:
            pass
    return (np.percentile(biases, 2.5), np.percentile(biases, 97.5)), \
           (np.percentile(iccs, 2.5), np.percentile(iccs, 97.5))

results = []
for label, name in [('bedtime', '취침시각'), ('waketime', '기상시각')]:
    d = m.dropna(subset=[f'D_{label}_hr']).copy()
    d['s_hr'] = centered_hours(d[f's_{label}_raw'], REF[label])
    d['w_hr'] = centered_hours(d[f'w_{label}_raw'], REF[label])
    print(f"\n=== [{name}] 유효 n={len(d)} ({d['id'].nunique()}명) ===")

    diff = d[f'D_{label}_hr']
    bias, sd_diff = diff.mean(), diff.std()
    loa_low, loa_high = bias - 1.96*sd_diff, bias + 1.96*sd_diff
    print(f"[일치도] bias(원형차이)={bias:.3f}시간, 95% LOA=[{loa_low:.3f}, {loa_high:.3f}]")
    for thr_min in [30, 60, 120]:
        pct = (diff.abs() <= thr_min/60).mean() * 100
        print(f"  ±{thr_min}분 이내 일치율={pct:.1f}%")

    rho, p_rho = stats.spearmanr(d['s_hr'], d['w_hr'])
    print(f"[스피어만(보조지표, 반복측정 구조 미반영, 기준시각 대비 편차값 사용)] rho={rho:.3f}, p={p_rho:.4f}")

    icc = icc_2_1(d['s_hr'].values, d['w_hr'].values)
    icc_grade = 'poor(<0.5)' if icc < 0.5 else ('moderate(0.5~0.75)' if icc < 0.75 else ('good(0.75~0.9)' if icc < 0.9 else 'excellent(>=0.9)'))
    bias_ci, icc_ci = cluster_bootstrap_bedtime(d, label)
    print(f"[ICC(2,1) 절대일치도] {icc:.3f} -> {icc_grade} [군집부트스트랩95%CI: {icc_ci[0]:.3f}, {icc_ci[1]:.3f}]")
    print(f"[bias 군집부트스트랩95%CI] [{bias_ci[0]:.3f}, {bias_ci[1]:.3f}]")

    fit = smf.mixedlm('w_hr ~ s_hr', d, groups=d['id']).fit(reml=True)
    coef, p = fit.params['s_hr'], fit.pvalues['s_hr']
    print(f"[반복측정] wearable ~ survey: coef={coef:.3f}, p={p:.4f} (n={len(d)}, n_id={d['id'].nunique()})")
    print(f"  -> 해석: 설문 {name}이 1시간 늦어질 때 웨어러블은 평균 {coef*60:.1f}분 늦어지는 관계(완전일치=1시간과 비교, 유의성≠대체가능성)")
    results.append({'지표': name, 'n': len(d), 'n_id': d['id'].nunique(), 'coef': coef, 'p': p, 'icc': icc,
                     'icc_ci_low': icc_ci[0], 'icc_ci_high': icc_ci[1],
                     'bias_ci_low': bias_ci[0], 'bias_ci_high': bias_ci[1]})

res_df = pd.DataFrame(results)
rej, p_fdr, _, _ = multipletests(res_df['p'], method='fdr_bh')
res_df['p_fdr'] = p_fdr
res_df['reject_fdr'] = rej
print("\n=== 반복측정 관계분석 FDR 보정 결과 (family: 취침시각+기상시각, 2개 검정) ===")
print(res_df.to_string(index=False))

res_df.to_csv(os.path.join(RESULT, 'd2_bedtime_agreement_results.csv'), index=False)
print("\n저장 완료: d2_bedtime_gap_pw.csv, d2_bedtime_agreement_results.csv, d2_bedtime_sensitivity.csv")


[bedtime] 모집단 기준시각(원형평균) = 1.91시(raw)
[waketime] 모집단 기준시각(원형평균) = 9.28시(raw)

=== 내적 일관성 필터 민감도 분석 ===
threshold       지표  n_제외   n  n_id      bias     coef        p
     None  bedtime     0 242    34  0.094867 0.067168 0.012886
     None waketime     0 242    34 -0.460401 0.020850 0.322004
     2.0h  bedtime    88 195    33 -0.341202 0.190718 0.000017
     2.0h waketime    88 195    33 -0.596359 0.125914 0.012685
     3.0h  bedtime    76 204    33 -0.301454 0.155314 0.000249
     3.0h waketime    76 204    33 -0.526072 0.131150 0.005296
     4.0h  bedtime    76 204    33 -0.301454 0.155314 0.000249
     4.0h waketime    76 204    33 -0.526072 0.131150 0.005296

=== 3시간 기준 제외 행(76건)의 참여자 분포 ===
id
S605    14
S004     6
S505     6
S019     6
S049     6
S008     6
S026     4
S030     4
S985     3
S646     3
Name: count, dtype: int64

제외행의 평균 mq1(스트레스 관련) vs 전체 평균 비교:
  mq1: 제외행 평균=1.45, 전체 평균=1.49
  mq2: 제외행 평균=2.96, 전체 평균=2.97
  mq3: 제외행 평균=2.18, 전체 평균=2.39
  mq4: 제외행 평균=2.01, 전체 평균=2.0

## 6단계: 걷기 지표 일치도분석 (삼성헬스의 D-2 대응, 걷기빈도/시간)
- **Fitbit의 q7(산책)은 삼성헬스의 "10분 이상 연속 걷기"와 개념이 다름**(레저성 걷기 vs 일반 보행) — 주분석은 Fitbit이 운동 세션으로 감지한 걷기 전체(10분 기준 미적용, Fitbit 설문정의 정렬), 민감도분석은 10분 이상만(삼성헬스 정의 정렬)으로 분리
- **참고**: `activityName`이 대부분 '걷기'(한글)이나 S504만 'Walk'(영문)로 기록되어 있어, 두 라벨을 모두 인식해야 S504가 "걷기 0회"로 잘못 집계되지 않는다
- `q7-1`의 `-1`은 이상치가 아니라 "q7=아니오" 게이팅 코드이므로 0으로 치환한다(이상치로 결측 처리하면 표본이 29건까지 줄어든다)
- coverage=0인 주는 "걷기 0"이 아니라 결측으로 구분(steps.csv 기준)
- Bland-Altman(bias=웨어러블−자기보고 명시)+ICC+참여자 군집부트스트랩 95%CI+스피어만(보조)+반복측정(공식추론)+FDR
- 보조분석: 산책 여부(이분형) 일치율/Cohen's kappa/민감도·특이도/군집강건 로지스틱

In [5]:
import statsmodels.api as sm
RNG = np.random.default_rng(42)

mw = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
mw['survey_date'] = pd.to_datetime(mw['survey_date'])
both_ids = sorted(mw['id'].unique())

# ============================================================
# 1) 피험자별 걷기 세션 지표 로드
#    (Fitbit이 "운동"으로 감지한 걷기 세션 기반 person-week 집계는
#     data/fitbit_wearable_person_week.csv 생성 시 이미 계산됨 - coverage=0인 주의
#     결측처리(측정 자체가 없었던 것과 "걷기 0회"를 구분)까지 반영된 값을 그대로 사용)
# ============================================================
wearable_pw_wk = pd.read_csv(os.path.join(DATA, 'fitbit_wearable_person_week.csv'))
wearable_pw_wk['survey_date'] = pd.to_datetime(wearable_pw_wk['survey_date'])
wk_cols = ['id', 'survey_date', 'w_walk_n_all', 'w_walk_mean_all', 'w_walk_total_all',
           'w_walk_n_10', 'w_walk_mean_10', 'w_walk_total_10']
wk = wearable_pw_wk[wk_cols]

m = mw.merge(wk, on=['id', 'survey_date'], how='left')

# ============================================================
# 2) 설문 q7 계열 준비
# ============================================================
m['s_walk_yn'] = pd.to_numeric(m['q7'], errors='coerce').map({1: 0, 2: 1})  # 1=아니오, 2=예
m['s_walk_freq'] = pd.to_numeric(m['q7-1'], errors='coerce').replace(-1, 0)
n_oob = ((m['s_walk_freq'] < 0) | (m['s_walk_freq'] > 7)).sum()
print(f"q7-1(-1 게이팅 처리 후) 범위[0,7] 밖 진짜 이상치: {n_oob}건 -> 결측 처리")
m.loc[(m['s_walk_freq'] < 0) | (m['s_walk_freq'] > 7), 's_walk_freq'] = np.nan

m['s_walk_dur'] = pd.to_numeric(m['q7-2'], errors='coerce')
m.loc[m['s_walk_dur'] == -1, 's_walk_dur'] = np.nan

m['s_walk_total'] = m['s_walk_freq'] * m['s_walk_dur']
m.loc[m['s_walk_freq'] == 0, 's_walk_total'] = 0

m['w_walk_yn'] = (m['w_walk_n_all'] > 0).astype(float)
m.loc[m['w_walk_n_all'].isna(), 'w_walk_yn'] = np.nan

m.to_csv(os.path.join(RESULT, 'd2_walk_gap_pw.csv'), index=False)
print(f"저장 완료: d2_walk_gap_pw.csv (n={len(m)})")

# ============================================================
# 3) 유틸: ICC, 참여자단위 군집 부트스트랩(bias, ICC 95% CI)
# ============================================================
def icc_2_1(x, y):
    data = np.column_stack([x, y])
    n, k = data.shape
    grand_mean = data.mean()
    row_means, col_means = data.mean(axis=1), data.mean(axis=0)
    SST = ((data - grand_mean) ** 2).sum()
    SSR = k * ((row_means - grand_mean) ** 2).sum()
    SSC = n * ((col_means - grand_mean) ** 2).sum()
    SSE = SST - SSR - SSC
    MSR, MSC, MSE = SSR/(n-1), SSC/(k-1), SSE/((n-1)*(k-1))
    return (MSR - MSE) / (MSR + (k-1)*MSE + k*(MSC-MSE)/n)

def cluster_bootstrap(dd, scol, wcol, n_boot=1000):
    """참여자 단위 군집 부트스트랩으로 bias, ICC의 95% CI 산출 (반복측정 구조 반영)."""
    ids = dd['id'].unique()
    biases, iccs = [], []
    for i in range(n_boot):
        sample_ids = RNG.choice(ids, size=len(ids), replace=True)
        boot = pd.concat([dd[dd['id'] == sid] for sid in sample_ids], ignore_index=True)
        if boot['id'].nunique() < 3:
            continue
        diff = boot[wcol] - boot[scol]
        biases.append(diff.mean())
        try:
            iccs.append(icc_2_1(boot[scol].values, boot[wcol].values))
        except Exception:
            pass
    return (np.percentile(biases, 2.5), np.percentile(biases, 97.5)), \
           (np.percentile(iccs, 2.5), np.percentile(iccs, 97.5))

def run_agreement(d, scol, wcol, name, unit=''):
    dd = d.dropna(subset=[scol, wcol]).copy()
    if len(dd) < 10:
        print(f"  [{name}] 표본부족(n={len(dd)}) - 스킵")
        return None
    diff = dd[wcol] - dd[scol]  # bias 정의: 웨어러블 - 자기보고 (양수면 웨어러블이 더 큼)
    bias, sd_diff = diff.mean(), diff.std()
    loa = (bias - 1.96*sd_diff, bias + 1.96*sd_diff)
    icc = icc_2_1(dd[scol].values, dd[wcol].values)
    rho, p_rho = stats.spearmanr(dd[scol], dd[wcol])
    fit = smf.mixedlm(f'{wcol} ~ {scol}', dd, groups=dd['id']).fit(reml=True)
    coef, p = fit.params[scol], fit.pvalues[scol]
    bias_ci, icc_ci = cluster_bootstrap(dd, scol, wcol)
    print(f"\n=== [{name}] n={len(dd)} ({dd['id'].nunique()}명) ===")
    print(f"  bias(웨어러블-자기보고)={bias:.3f}{unit} [군집부트스트랩95%CI: {bias_ci[0]:.3f},{bias_ci[1]:.3f}]")
    print(f"  95%LOA=[{loa[0]:.3f},{loa[1]:.3f}]{unit}")
    print(f"  ICC={icc:.3f} [군집부트스트랩95%CI: {icc_ci[0]:.3f},{icc_ci[1]:.3f}]")
    print(f"  rho={rho:.3f}(p={p_rho:.4f}, 보조지표) | coef={coef:.3f}(p={p:.4f}, 공식추론)")
    return {'항목': name, 'n': len(dd), 'n_id': dd['id'].nunique(), 'bias': bias,
            'bias_ci_low': bias_ci[0], 'bias_ci_high': bias_ci[1],
            'loa_low': loa[0], 'loa_high': loa[1], 'icc': icc,
            'icc_ci_low': icc_ci[0], 'icc_ci_high': icc_ci[1],
            'rho': rho, 'p_rho': p_rho, 'coef': coef, 'p': p}

print("\n" + "="*70)
print("주분석: q7(산책) vs Fitbit 감지 걷기세션 (10분 기준 미적용, Fitbit 설문정의 정렬)")
print("(주의: 'Fitbit이 운동 세션으로 감지한 걷기'이며 '전체 걷기'가 아님)")
print("="*70)
main_results = []
main_results.append(run_agreement(m, 's_walk_freq', 'w_walk_n_all', '걷기빈도(주당횟수)', unit='회/주'))
main_results.append(run_agreement(m, 's_walk_dur', 'w_walk_mean_all', '걷기시간(1회당)', unit='분'))
main_results.append(run_agreement(m, 's_walk_total', 'w_walk_total_all', '주간총 걷기시간(파생)', unit='분/주'))

print("\n" + "="*70)
print("민감도분석: 10분 이상 세션만 (삼성헬스 데이터와의 정의 정렬용)")
print("="*70)
sens_results = []
sens_results.append(run_agreement(m, 's_walk_freq', 'w_walk_n_10', '걷기빈도(10분+)', unit='회/주'))
sens_results.append(run_agreement(m, 's_walk_dur', 'w_walk_mean_10', '걷기시간(10분+,1회당)', unit='분'))
sens_results.append(run_agreement(m, 's_walk_total', 'w_walk_total_10', '주간총 걷기시간(10분+,파생)', unit='분/주'))

main_df = pd.DataFrame([r for r in main_results if r])
rej, p_fdr, _, _ = multipletests(main_df['p'], method='fdr_bh')
main_df['p_fdr'] = p_fdr
main_df['reject_fdr'] = rej
print("\n=== 주분석 FDR 보정 결과 ===")
print(main_df.to_string(index=False))

# Spearman rho도 삼성헬스와 동일하게 자체 family로 별도 FDR 보정을 적용한다.
rej_rho, p_fdr_rho, _, _ = multipletests(main_df['p_rho'], method='fdr_bh')
main_df['rho_p_fdr'] = p_fdr_rho
main_df['rho_reject_fdr'] = rej_rho
print("\n=== 주분석 Spearman rho FDR 보정 결과 ===")
print(main_df[['항목', 'rho', 'p_rho', 'rho_p_fdr', 'rho_reject_fdr']].to_string(index=False))

main_df.to_csv(os.path.join(RESULT, 'd2_walk_agreement_main.csv'), index=False)

sens_df = pd.DataFrame([r for r in sens_results if r])
if len(sens_df) > 0:
    rej2, p_fdr2, _, _ = multipletests(sens_df['p'], method='fdr_bh')
    sens_df['p_fdr'] = p_fdr2
    sens_df['reject_fdr'] = rej2
    print("\n=== 민감도분석(10분+) FDR 보정 결과 ===")
    print(sens_df.to_string(index=False))
    rej_rho2, p_fdr_rho2, _, _ = multipletests(sens_df['p_rho'], method='fdr_bh')
    sens_df['rho_p_fdr'] = p_fdr_rho2
    sens_df['rho_reject_fdr'] = rej_rho2
    print("\n=== 민감도분석(10분+) Spearman rho FDR 보정 결과 ===")
    print(sens_df[['항목', 'rho', 'p_rho', 'rho_p_fdr', 'rho_reject_fdr']].to_string(index=False))
    sens_df.to_csv(os.path.join(RESULT, 'd2_walk_agreement_sensitivity.csv'), index=False)

# ============================================================
# 4) 보조 분석 1: 산책 여부(이분형) 일치도 - q7(O/X) vs 그 주 걷기세션 유무
# ============================================================
print("\n" + "="*70)
print("보조분석 1: 산책 여부(이분형) 일치도")
print("="*70)
db = m.dropna(subset=['s_walk_yn', 'w_walk_yn']).copy()
print(f"유효 n={len(db)} ({db['id'].nunique()}명)")

ct = pd.crosstab(db['s_walk_yn'], db['w_walk_yn'], rownames=['설문(산책)'], colnames=['웨어러블(걷기감지)'])
print("\n교차표:")
print(ct)

acc = (db['s_walk_yn'] == db['w_walk_yn']).mean()
tp = ((db['s_walk_yn']==1)&(db['w_walk_yn']==1)).sum()
fn = ((db['s_walk_yn']==1)&(db['w_walk_yn']==0)).sum()
fp = ((db['s_walk_yn']==0)&(db['w_walk_yn']==1)).sum()
tn = ((db['s_walk_yn']==0)&(db['w_walk_yn']==0)).sum()
sensitivity = tp/(tp+fn) if (tp+fn)>0 else np.nan
specificity = tn/(tn+fp) if (tn+fp)>0 else np.nan

po = acc
pe = ((db['s_walk_yn'].mean()*db['w_walk_yn'].mean()) + ((1-db['s_walk_yn'].mean())*(1-db['w_walk_yn'].mean())))
kappa = (po - pe) / (1 - pe) if (1-pe) != 0 else np.nan

print(f"\n일치율(accuracy)={acc*100:.1f}%")
print(f"민감도(설문=산책일 때 웨어러블도 감지)={sensitivity*100:.1f}%, 특이도(설문=안함일 때 웨어러블도 없음)={specificity*100:.1f}%")
print(f"Cohen's kappa={kappa:.3f}")

# 반복측정 로지스틱(참여자 군집강건 표준오차)
try:
    glm = sm.GLM(db['w_walk_yn'], sm.add_constant(db['s_walk_yn']), family=sm.families.Binomial())
    fit_logit = glm.fit(cov_type='cluster', cov_kwds={'groups': db['id']})
    print(f"\n[군집강건 로지스틱] wearable_walk ~ survey_walk: coef={fit_logit.params['s_walk_yn']:.3f}, p={fit_logit.pvalues['s_walk_yn']:.4f}")
except Exception as e:
    print(f"로지스틱 모형 실패: {e}")

# ============================================================
# 5) 보조 분석 2: 양쪽 모두 걷기 있는 주만 대상 (탐색적, 표본 작음 명시)
# ============================================================
print("\n" + "="*70)
print("보조분석 2 (탐색적, 표본 작음 주의): 양쪽 다 걷기 있는 주만 대상 빈도/시간 비교")
print("="*70)
both_walked = m[(m['s_walk_yn']==1) & (m['w_walk_yn']==1)].dropna(subset=['s_walk_dur','w_walk_mean_all'])
print(f"n={len(both_walked)} ({both_walked['id'].nunique()}명) - 이미 위 '걷기시간(1회당)' 분석과 동일 모집단")

print("\n저장 완료: d2_walk_gap_pw.csv, d2_walk_agreement_main.csv, d2_walk_agreement_sensitivity.csv")

q7-1(-1 게이팅 처리 후) 범위[0,7] 밖 진짜 이상치: 0건 -> 결측 처리
저장 완료: d2_walk_gap_pw.csv (n=386)

주분석: q7(산책) vs Fitbit 감지 걷기세션 (10분 기준 미적용, Fitbit 설문정의 정렬)
(주의: 'Fitbit이 운동 세션으로 감지한 걷기'이며 '전체 걷기'가 아님)

=== [걷기빈도(주당횟수)] n=292 (39명) ===
  bias(웨어러블-자기보고)=2.120회/주 [군집부트스트랩95%CI: 1.395,2.783]
  95%LOA=[-3.871,8.111]회/주
  ICC=-0.030 [군집부트스트랩95%CI: -0.139,0.064]
  rho=-0.025(p=0.6730, 보조지표) | coef=-0.211(p=0.1698, 공식추론)

=== [걷기시간(1회당)] n=41 (17명) ===
  bias(웨어러블-자기보고)=-6.021분 [군집부트스트랩95%CI: -13.097,0.852]
  95%LOA=[-42.182,30.141]분
  ICC=0.294 [군집부트스트랩95%CI: -0.063,0.599]
  rho=0.452(p=0.0030, 보조지표) | coef=0.251(p=0.0698, 공식추론)

=== [주간총 걷기시간(파생)] n=292 (39명) ===
  bias(웨어러블-자기보고)=54.766분/주 [군집부트스트랩95%CI: 30.810,82.707]
  95%LOA=[-178.492,288.024]분/주
  ICC=-0.046 [군집부트스트랩95%CI: -0.139,0.042]
  rho=-0.010(p=0.8678, 보조지표) | coef=-0.230(p=0.1214, 공식추론)

민감도분석: 10분 이상 세션만 (삼성헬스 데이터와의 정의 정렬용)

=== [걷기빈도(10분+)] n=292 (39명) ===
  bias(웨어러블-자기보고)=2.079회/주 [군집부트스트랩95%CI: 1.421,2.722]
  95%LOA=[-3.848,8.006]회/주
  

### Step 4~6 종합 해석 (논문 Table 15)

수면시간은 삼성헬스와 유사하게 순위상관·반복측정 관계가 유의했으나(ρ=.474, β=.308, 둘 다 FDR P<.001) ICC=0.394로 낮은 절대일치도를 보였다. 반면 산책빈도는 유의한 관련성이 없었고 편향의 방향도 삼성헬스의 걷기빈도와 반대(웨어러블이 자기보고보다 높음)였다 — Fitbit의 "산책" 문항이 삼성헬스의 "10분 이상 연속 걷기"보다 좁은 여가 목적 보행만 포착하는 문항범위 차이 때문일 수 있다. 취침·기상시각도 ICC가 각각 0.424, 0.382로 낮았다. **즉 수면 관련 지표에서는 주 분석과 유사하게 관련성과 절대일치도가 분리되는 양상이 두 표본에서 일관되게 관찰되었으나, 걷기(산책) 지표는 문항 정의 차이로 직접비교가 제한적이다.**

## 7단계: 스트레스 대리변수 구축 + Fitbit 전용 within/between 확장분석

- weekly mq1/mq2/mq4(GHQ류 심리적 디스트레스 3문항, mq2는 긍정문항이라 역채점)를 문항별 Z-score 평균으로 `stress_score` 구축 — mq3("자고 난 후에도 개운한 감이 없다")는 수면 상태를 직접 묻는 문항이라 수면질 근사 지표와 내용이 중첩되어 제외 (논문 Table 8과 동일 정의)
- 참여자 평균(`stress_between`)과 그 주의 편차(`stress_within`)로 분해 — 삼성헬스의 between-only 주분석과 달리 반복측정 구조를 활용한 확장분해(삼성헬스 D-3과 같은 틀이나 사양은 다름)
- 삼성헬스 D-3과 달리 달력주차(week 1~4)가 없고 응답 주차수가 불규칙하며 주차 고정효과를 쓰지 않는 등 모형 사양이 다르므로, 이 단계의 결과는 계수의 직접 재현이 아니라 "Fitbit 전용 탐색"으로 분류한다

In [6]:
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


# ============================================================
# 1) 스트레스 대리변수 구축: mq1/mq2/mq4(GHQ류 심리적 디스트레스 3문항) 문항별 Z-score 평균
#    (논문 Table 8과 동일 정의) - mq3는 수면 상태를 직접 묻는 문항이라 수면질 근사 지표와
#    내용이 중첩되어 제외한다.
#    - 문항별로 척도(응답범위)가 달라 원점수 그대로 합산하면 척도불일치 문제가 생길 수 있음
#      -> 문항별 Z-score 후 평균
#    - mq2는 긍정문항("편안하며 건강하다") -> 역채점 필요
#      데이터로 검증: mq1/mq4는 서로 양의 상관인데 mq2는 음의 상관 -> 역채점 방향 확인됨
# ============================================================
m = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))

# 각 Gap 지표는 서로 다른 단계 산출물에 있음 -> id+survey_date 기준으로 병합
sleep_gap = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))[['id', 'survey_date', 'absGap_sleeptime', 'absGap_sleepquality']]
bedtime_gap = pd.read_csv(os.path.join(RESULT, 'd2_bedtime_gap_pw.csv'))[['id', 'survey_date', 'absGap_bedtime', 'absGap_waketime']]
m = m.merge(sleep_gap, on=['id', 'survey_date'], how='left').merge(bedtime_gap, on=['id', 'survey_date'], how='left')

for c in ['mq1', 'mq2', 'mq4']:
    m[c] = pd.to_numeric(m[c], errors='coerce')
m['mq2_rev'] = 5 - m['mq2']  # 1~4 척도 역채점

def zscore(s):
    return (s - s.mean()) / s.std()

items_z = pd.DataFrame({
    'mq1_z': zscore(m['mq1']),
    'mq2_z': zscore(m['mq2_rev']),
    'mq4_z': zscore(m['mq4']),
})
m['stress_score'] = items_z.mean(axis=1, skipna=False)  # 3문항 중 하나라도 결측이면 그 주는 결측(전부 응답 필수)

print(f"stress_score 결측: {m['stress_score'].isna().sum()}/{len(m)}")

# ============================================================
# 2) 개인간(between)/개인내(within) 분해
# ============================================================
person_mean = m.groupby('id')['stress_score'].transform('mean')
m['stress_between'] = person_mean
m['stress_within'] = m['stress_score'] - person_mean

m.to_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'), index=False)
print(f"저장 완료: d2_stress_gap_pw.csv (n={len(m)})")

# ============================================================
# 3) 분석 함수: between/within 분해 모형 + 삼성헬스 직접비교용 비분해 모형
# ============================================================
def run_stress_test(d, outcome, label, extra_covariate=None):
    covs = ['stress_between', 'stress_within'] + ([extra_covariate] if extra_covariate else [])
    dd = d.dropna(subset=[outcome] + covs).copy()
    if len(dd) < 15 or dd['id'].nunique() < 5:
        print(f"  [{label}] 표본부족(n={len(dd)}) - 스킵")
        return None, None

    # 3-1. between/within 분해 모형 (권장구조)
    formula = f"{outcome} ~ stress_between + stress_within" + (f" + {extra_covariate}" if extra_covariate else "")
    fit = smf.mixedlm(formula, dd, groups=dd['id']).fit(reml=True)
    out_decomp = {
        '결과변수': label, 'n': len(dd), 'n_id': dd['id'].nunique(),
        'coef_between': fit.params['stress_between'], 'p_between': fit.pvalues['stress_between'],
        'ci_between': fit.conf_int().loc['stress_between'].tolist(),
        'coef_within': fit.params['stress_within'], 'p_within': fit.pvalues['stress_within'],
        'ci_within': fit.conf_int().loc['stress_within'].tolist(),
    }

    # 3-2. 비분해 모형 (삼성헬스의 concurrent 단일계수, 원점수 β=0.259 (혼합모형 P=.005, 군집강건 P=.026)와 방향 비교용)
    formula2 = f"{outcome} ~ stress_score" + (f" + {extra_covariate}" if extra_covariate else "")
    fit2 = smf.mixedlm(formula2, dd, groups=dd['id']).fit(reml=True)
    out_pooled = {
        '결과변수': label, 'n': len(dd), 'n_id': dd['id'].nunique(),
        'coef': fit2.params['stress_score'], 'p': fit2.pvalues['stress_score'],
        'ci': fit2.conf_int().loc['stress_score'].tolist(),
    }
    return out_decomp, out_pooled

print("\n" + "="*70)
print("[주분석] 수면시간 |D*| ~ 스트레스 (삼성헬스의 핵심결과 재현 대상)")
print("="*70)
main_decomp, main_pooled = run_stress_test(m, 'absGap_sleeptime', '수면시간간극', extra_covariate='wearable_coverage')
if main_decomp:
    print(f"[between/within] between: coef={main_decomp['coef_between']:.3f}, p={main_decomp['p_between']:.4f}, 95%CI={main_decomp['ci_between']}")
    print(f"                  within:  coef={main_decomp['coef_within']:.3f}, p={main_decomp['p_within']:.4f}, 95%CI={main_decomp['ci_within']}")
if main_pooled:
    print(f"[비분해, 삼성헬스 직접비교용] coef={main_pooled['coef']:.3f}, p={main_pooled['p']:.4f}, 95%CI={main_pooled['ci']}")
    print(f"  -> 삼성헬스 추정치(원점수 β=0.259; 혼합모형 P=.005, 군집강건 P=.026)와 비교: 방향 {'일치' if main_pooled['coef']>0 else '불일치'}, "
          f"삼성헬스 점추정이 Fitbit CI 안에 {'포함됨' if main_pooled['ci'][0]<=0.259<=main_pooled['ci'][1] else '포함 안됨'}")

print("\n" + "="*70)
print("[개념적 재현, 별도 family] 수면질(근사) 간극 ~ 스트레스")
print("="*70)
qual_decomp, qual_pooled = run_stress_test(m, 'absGap_sleepquality', '수면질간극(근사)', extra_covariate='wearable_coverage')
if qual_decomp:
    print(f"[between/within] between: coef={qual_decomp['coef_between']:.3f}, p={qual_decomp['p_between']:.4f}")
    print(f"                  within:  coef={qual_decomp['coef_within']:.3f}, p={qual_decomp['p_within']:.4f}")
if qual_pooled:
    print(f"[비분해] coef={qual_pooled['coef']:.3f}, p={qual_pooled['p']:.4f}")

print("\n" + "="*70)
print("[Fitbit 전용 탐색, 별도 family] 취침시각/기상시각 간극 ~ 스트레스")
print("="*70)
bed_decomp, bed_pooled = run_stress_test(m, 'absGap_bedtime', '취침시각간극', extra_covariate=None)
if bed_decomp:
    print(f"[취침시각 between/within] between: coef={bed_decomp['coef_between']:.3f}, p={bed_decomp['p_between']:.4f}")
    print(f"                          within:  coef={bed_decomp['coef_within']:.3f}, p={bed_decomp['p_within']:.4f}")
wake_decomp, wake_pooled = run_stress_test(m, 'absGap_waketime', '기상시각간극', extra_covariate=None)
if wake_decomp:
    print(f"[기상시각 between/within] between: coef={wake_decomp['coef_between']:.3f}, p={wake_decomp['p_between']:.4f}")
    print(f"                          within:  coef={wake_decomp['coef_within']:.3f}, p={wake_decomp['p_within']:.4f}")

# ============================================================
# 4) FDR 보정 - family별로 분리(측정 타당성이 다른 결과변수를 하나로 합치지 않음)
# ============================================================
def fdr_family(name, decomp_list):
    rows = []
    for label, d in decomp_list:
        if d is None:
            continue
        rows.append({'결과변수': label, '항': 'between', 'coef': d['coef_between'], 'p': d['p_between']})
        rows.append({'결과변수': label, '항': 'within', 'coef': d['coef_within'], 'p': d['p_within']})
    if not rows:
        return None
    df = pd.DataFrame(rows)
    rej, p_fdr, _, _ = multipletests(df['p'], method='fdr_bh')
    df['p_fdr'] = p_fdr
    df['reject_fdr'] = rej
    print(f"\n=== FDR 보정: {name} ===")
    print(df.to_string(index=False))
    return df

fam_main = fdr_family("주분석(수면시간)", [('수면시간간극', main_decomp)])
fam_qual = fdr_family("개념적재현(수면질근사)", [('수면질간극(근사)', qual_decomp)])
fam_explore = fdr_family("탐색(취침/기상시각)", [('취침시각간극', bed_decomp), ('기상시각간극', wake_decomp)])

for name, df in [('main', fam_main), ('qual', fam_qual), ('explore', fam_explore)]:
    if df is not None:
        df.to_csv(os.path.join(RESULT, f'd2_stress_gap_{name}.csv'), index=False)

print("\n저장 완료: d2_stress_gap_pw.csv, d2_stress_gap_main/qual/explore.csv")


stress_score 결측: 0/386
저장 완료: d2_stress_gap_pw.csv (n=386)

[주분석] 수면시간 |D*| ~ 스트레스 (삼성헬스의 핵심결과 재현 대상)
[between/within] between: coef=-0.195, p=0.3398, 95%CI=[-0.5963900393969702, 0.20570544159582047]
                  within:  coef=-0.004, p=0.9698, 95%CI=[-0.22378849039442109, 0.2153203096262539]
[비분해, 삼성헬스 직접비교용] coef=-0.048, p=0.6248, 95%CI=[-0.24162916215704608, 0.14511952637237954]
  -> 삼성헬스 추정치(원점수 β=0.259; 혼합모형 P=.005, 군집강건 P=.026)와 비교: 방향 불일치, 삼성헬스 점추정이 Fitbit CI 안에 포함 안됨

[개념적 재현, 별도 family] 수면질(근사) 간극 ~ 스트레스
[between/within] between: coef=0.518, p=0.0230
                  within:  coef=0.260, p=0.0310
[비분해] coef=0.315, p=0.0035

[Fitbit 전용 탐색, 별도 family] 취침시각/기상시각 간극 ~ 스트레스
[취침시각 between/within] between: coef=-0.525, p=0.2991
                          within:  coef=0.152, p=0.4686
[기상시각 between/within] between: coef=-0.884, p=0.1404
                          within:  coef=0.367, p=0.1167

=== FDR 보정: 주분석(수면시간) ===
  결과변수       항      coef        p    p_fdr  reject_fdr
수면시간간극 

### 7-보조: 수면질근사 within효과 단독 참여자내 순열검정

`absGap_sleepquality ~ stress_within` 단독 검정에 한해 참여자 내(그 사람 주차 순서만 셔플, between 불변) 순열검정. 이 숫자는 아래 7c의 8개 검정 전체 family-wise 순열검정으로 다시 평가된다(다중검정 미보정 단독 결과이므로 최종 판단 근거로 단독 사용하지 않음).

In [7]:
d_sq_perm = m.dropna(subset=['absGap_sleepquality', 'stress_between', 'stress_within', 'wearable_coverage']).copy()
obs_fit_sq = smf.mixedlm('absGap_sleepquality ~ stress_between + stress_within + wearable_coverage',
                          d_sq_perm, groups=d_sq_perm['id']).fit(reml=True)
obs_coef_sq = obs_fit_sq.params['stress_within']
print(f"순열검정 대상 표본: n={len(d_sq_perm)}, n_id={d_sq_perm['id'].nunique()}, 관측 within coef={obs_coef_sq:.4f}")

N_PERM_SQ = 3000
perm_coefs_sq = np.zeros(N_PERM_SQ)
rng_sq = np.random.default_rng(2024)
for i in range(N_PERM_SQ):
    dp = d_sq_perm.copy()
    dp['stress_score_perm'] = dp.groupby('id')['stress_score'].transform(lambda s: rng_sq.permutation(s.values))
    dp['stress_within_perm'] = dp['stress_score_perm'] - dp['stress_between']
    try:
        fit_p = smf.mixedlm('absGap_sleepquality ~ stress_between + stress_within_perm + wearable_coverage',
                             dp, groups=dp['id']).fit(reml=True)
        perm_coefs_sq[i] = fit_p.params['stress_within_perm']
    except Exception:
        perm_coefs_sq[i] = np.nan
valid_sq = ~np.isnan(perm_coefs_sq)
perm_p_sq = (np.abs(perm_coefs_sq[valid_sq]) >= np.abs(obs_coef_sq)).mean()
print(f"순열 성공: {valid_sq.sum()}/{N_PERM_SQ}, [단독검정] 참여자내 순열검정 p={perm_p_sq:.4f}")
print("(참고: 아래 7c의 family-wise 검정으로 다중검정 보정된 결과를 최종 판단에 사용)")

순열검정 대상 표본: n=242, n_id=34, 관측 within coef=0.2599
순열 성공: 3000/3000, [단독검정] 참여자내 순열검정 p=0.0387
(참고: 아래 7c의 family-wise 검정으로 다중검정 보정된 결과를 최종 판단에 사용)


## 7c단계: mq3 문항중첩 민감도분석 + family-wise 순열검정 + 공통방법편향 확인

- mq3("수면 후 개운치 않음")는 수면 상태를 직접 묻는 문항이라 수면질 결과변수와 내용이 겹칠 수 있음 → 전체합성/mq3제외합성/개별문항으로 민감도 확인
- max-|t| 순열검정은 within은 참여자 내 주차순서 셔플, between(`stress_between`)은 참여자 간 라벨 재배정으로 각각 순열한다
- 공통방법편향 점검은 person-week 쌍 구조를 반영하지 못하는 long-format method×interaction 모형 대신, 이미 그 차이인 `R_qual(=Zs-Zw)~stress`로 대체(수학적으로 동등하며 쌍 구조를 차분으로 자동 반영)

In [8]:
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests


m = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))
qual_components = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))[['id', 'survey_date', 'Zs_qual', 'Zw_qual', 'R_qual']]
m = m.merge(qual_components, on=['id', 'survey_date'], how='left')

def zscore(s):
    return (s - s.mean()) / s.std()

# ============================================================
# 1) mq3(수면 후 개운함 없음) 문항 내용중첩 민감도분석
#    - mq3는 심리적 디스트레스가 아니라 수면 상태를 직접 묻는 문항이라
#      결과변수(q4+q5 불면 기반 수면질근사)와 내용이 겹칠 위험이 있음
#    - 전체합성 vs mq3제외 합성 vs 개별문항으로 absGap_sleepquality와의 관계 재검증
# ============================================================
print("="*70)
print("[민감도] mq3 포함/제외 및 개별문항별 absGap_sleepquality ~ 심리디스트레스(within)")
print("="*70)

m['distress_full'] = pd.DataFrame({
    'mq1_z': zscore(m['mq1']), 'mq2_z': zscore(m['mq2_rev']),
    'mq3_z': zscore(m['mq3']), 'mq4_z': zscore(m['mq4']),
}).mean(axis=1)
m['distress_excl_mq3'] = pd.DataFrame({
    'mq1_z': zscore(m['mq1']), 'mq2_z': zscore(m['mq2_rev']), 'mq4_z': zscore(m['mq4']),
}).mean(axis=1)

def add_between_within(df, col):
    pm = df.groupby('id')[col].transform('mean')
    df[f'{col}_between'] = pm
    df[f'{col}_within'] = df[col] - pm
    return df

for col in ['distress_full', 'distress_excl_mq3', 'mq1', 'mq2_rev', 'mq3', 'mq4']:
    m = add_between_within(m, col)

sens_rows = []
for label, col in [('전체합성(mq1-4)', 'distress_full'), ('mq3제외 합성(mq1,2,4)', 'distress_excl_mq3'),
                    ('mq1 단독', 'mq1'), ('mq2_rev 단독', 'mq2_rev'), ('mq3 단독', 'mq3'), ('mq4 단독', 'mq4')]:
    dd = m.dropna(subset=['absGap_sleepquality', f'{col}_between', f'{col}_within', 'wearable_coverage']).copy()
    if len(dd) < 15:
        continue
    fit = smf.mixedlm(f'absGap_sleepquality ~ {col}_between + {col}_within + wearable_coverage',
                       dd, groups=dd['id']).fit(reml=True)
    sens_rows.append({'변수': label, 'n': len(dd), 'n_id': dd['id'].nunique(),
                       'coef_within': fit.params[f'{col}_within'], 'p_within': fit.pvalues[f'{col}_within'],
                       'coef_between': fit.params[f'{col}_between'], 'p_between': fit.pvalues[f'{col}_between']})
    print(f"[{label}] n={len(dd)}({dd['id'].nunique()}명) within: coef={fit.params[f'{col}_within']:.3f}, p={fit.pvalues[f'{col}_within']:.4f} "
          f"| between: coef={fit.params[f'{col}_between']:.3f}, p={fit.pvalues[f'{col}_between']:.4f}")

sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(os.path.join(RESULT, 'd2_stress_mq3_sensitivity.csv'), index=False)

# ============================================================
# 2) max|t| 순열검정: within-effect와 between-effect를 별도 순열방식으로
#    - within 4개: 참여자 내 주차 순서 셔플(stress_between 불변)
#    - between 4개: 참여자 평균(stress_between)을 참여자 "간" 재배정(스트레스 라벨을 다른 사람에게)
#    - 두 순열을 각 반복에서 함께 적용해 8개 검정의 결합 max|t| 널분포 구성
# ============================================================
print("\n" + "="*70)
print("[max|t| 순열검정] within/between 각각 별도 방식으로 순열")
print("="*70)

outcomes_covs = [
    ('absGap_sleeptime', 'wearable_coverage'),
    ('absGap_sleepquality', 'wearable_coverage'),
    ('absGap_bedtime', None),
    ('absGap_waketime', None),
]

def fit_all_correct(dframe, between_col, within_col):
    ts = []
    for outcome, cov in outcomes_covs:
        cols = [between_col, within_col] + ([cov] if cov else [])
        dd = dframe.dropna(subset=[outcome] + cols).copy()
        if len(dd) < 15 or dd['id'].nunique() < 5:
            continue
        formula = f"{outcome} ~ {between_col} + {within_col}" + (f" + {cov}" if cov else "")
        try:
            fit = smf.mixedlm(formula, dd, groups=dd['id']).fit(reml=True)
            for term in [between_col, within_col]:
                ts.append(abs(fit.params[term] / fit.bse[term]))
        except Exception:
            pass
    return max(ts) if ts else np.nan

obs_max_t = fit_all_correct(m, 'stress_between', 'stress_within')
print(f"관측 max|t| = {obs_max_t:.3f}")

# 참여자별 대표 stress_between 매핑(참여자 1명당 1개 값)
person_between = m.drop_duplicates('id')[['id', 'stress_between']].set_index('id')['stress_between']
id_list = person_between.index.tolist()

N_PERM = 5000
perm_max_ts = np.zeros(N_PERM)
rng1 = np.random.default_rng(111)
rng2 = np.random.default_rng(222)
for i in range(N_PERM):
    dp = m.copy()
    # within: 참여자 내 주차 순서 셔플 (between 불변)
    dp['stress_score_perm'] = dp.groupby('id')['stress_score'].transform(lambda s: rng1.permutation(s.values))
    # between: 참여자 간 라벨 재배정 (누가 어떤 평균 스트레스를 가졌는지 셔플)
    shuffled_between_map = pd.Series(rng2.permutation(person_between.values), index=id_list)
    dp['stress_between_perm'] = dp['id'].map(shuffled_between_map)
    dp['stress_within_perm'] = dp['stress_score_perm'] - dp['stress_between']  # within은 원래 between 기준으로 재계산(구조 유지)
    perm_max_ts[i] = fit_all_correct(dp, 'stress_between_perm', 'stress_within_perm')

valid = ~np.isnan(perm_max_ts)
perm_p_max_correct = (perm_max_ts[valid] >= obs_max_t).mean()
print(f"순열 성공: {valid.sum()}/{N_PERM}")
print(f"[max|t| 순열검정] p={perm_p_max_correct:.4f}")
print(f"(참고: 단순 대안으로 8개 raw p에 BH-FDR 적용한 결과도 병기)")

# 단순 대안: 8개 raw p에 BH-FDR
raw_ps = []
for outcome, cov in outcomes_covs:
    cols = ['stress_between', 'stress_within'] + ([cov] if cov else [])
    dd = m.dropna(subset=[outcome] + cols).copy()
    if len(dd) < 15 or dd['id'].nunique() < 5:
        continue
    formula = f"{outcome} ~ stress_between + stress_within" + (f" + {cov}" if cov else "")
    fit = smf.mixedlm(formula, dd, groups=dd['id']).fit(reml=True)
    for term in ['stress_between', 'stress_within']:
        raw_ps.append({'outcome': outcome, 'term': term, 'p': fit.pvalues[term]})
raw_df = pd.DataFrame(raw_ps)
rej, p_fdr, _, _ = multipletests(raw_df['p'], method='fdr_bh')
raw_df['p_fdr'] = p_fdr
raw_df['reject_fdr'] = rej
print(raw_df.to_string(index=False))
raw_df.to_csv(os.path.join(RESULT, 'd2_stress_8test_BHFDR.csv'), index=False)

# ============================================================
# 3) 공통방법편향 확인: R_qual(signed, 쌍 구조가 이미 차분으로 반영됨)로 검정
#    - long-format method×interaction 모형은 person-week 쌍 구조를 반영하지 못해 사용하지 않음
#    - R_qual = Zs_qual - Zw_qual 자체가 이미 그 차이이므로, R_qual~stress가 수학적으로 동등한 검정
# ============================================================
print("\n" + "="*70)
print("[공통방법편향] R_qual(=Zs-Zw, 쌍 구조 자동반영) ~ stress_within 검정 + 순열검정")
print("="*70)

d3 = m.dropna(subset=['R_qual', 'stress_between', 'stress_within', 'wearable_coverage']).copy()
fit_r = smf.mixedlm('R_qual ~ stress_between + stress_within + wearable_coverage', d3, groups=d3['id']).fit(reml=True)
obs_r_within = fit_r.params['stress_within']
obs_r_t = obs_r_within / fit_r.bse['stress_within']
print(f"[R_qual(signed) ~ stress_within] coef={obs_r_within:.4f}, p={fit_r.pvalues['stress_within']:.4f}, n={len(d3)}")
print("이 검정이 'method x stress_within 상호작용'과 수학적으로 동등하며, person-week 쌍 구조를 자동으로 반영함(차분이므로)")

N_PERM_R = 5000
perm_r = np.zeros(N_PERM_R)
rng3 = np.random.default_rng(333)
for i in range(N_PERM_R):
    dp = d3.copy()
    dp['stress_score_perm'] = dp.groupby('id')['stress_score'].transform(lambda s: rng3.permutation(s.values))
    dp['stress_within_perm'] = dp['stress_score_perm'] - dp['stress_between']
    try:
        fit_p = smf.mixedlm('R_qual ~ stress_between + stress_within_perm + wearable_coverage', dp, groups=dp['id']).fit(reml=True)
        perm_r[i] = fit_p.params['stress_within_perm']
    except Exception:
        perm_r[i] = np.nan
valid_r = ~np.isnan(perm_r)
perm_p_r = (np.abs(perm_r[valid_r]) >= np.abs(obs_r_within)).mean()
print(f"순열 성공: {valid_r.sum()}/{N_PERM_R}, [참여자내 순열검정] p={perm_p_r:.4f}")

print("\n저장 완료: d2_stress_mq3_sensitivity.csv, d2_stress_8test_BHFDR.csv")


[민감도] mq3 포함/제외 및 개별문항별 absGap_sleepquality ~ 심리디스트레스(within)
[전체합성(mq1-4)] n=242(34명) within: coef=0.316, p=0.0159 | between: coef=0.523, p=0.0290
[mq3제외 합성(mq1,2,4)] n=242(34명) within: coef=0.260, p=0.0310 | between: coef=0.518, p=0.0230
[mq1 단독] n=242(34명) within: coef=0.170, p=0.1925 | between: coef=0.903, p=0.0008
[mq2_rev 단독] n=242(34명) within: coef=0.186, p=0.0473 | between: coef=0.443, p=0.0469
[mq3 단독] n=242(34명) within: coef=0.137, p=0.1113 | between: coef=0.278, p=0.2040
[mq4 단독] n=242(34명) within: coef=0.109, p=0.2003 | between: coef=0.195, p=0.4196

[max|t| 순열검정] within/between 각각 별도 방식으로 순열
관측 max|t| = 2.273
순열 성공: 5000/5000
[max|t| 순열검정] p=0.1422
(참고: 단순 대안으로 8개 raw p에 BH-FDR 적용한 결과도 병기)
            outcome           term        p    p_fdr  reject_fdr
   absGap_sleeptime stress_between 0.339750 0.453000       False
   absGap_sleeptime  stress_within 0.969849 0.969849       False
absGap_sleepquality stress_between 0.023007 0.123894       False
absGap_sleepquality  stress_

## 8단계: 삼성헬스 방법론 재현모형 — timeInBed 재구축

삼성헬스 D-1A와 동일한 정의·집계로 Fitbit 재현모형을 구축한다:
1. 웨어러블 수면시간: 삼성헬스와 동일하게 wake-inclusive(`timeInBed`)를 주 정의로 하고, wake-exclusive(`minutesAsleep`)는 민감도로 병행한다
2. 집계 단위: person-week가 아니라 **person-level**(그 사람 여러 주 D를 먼저 평균 후 표준화), `mean_stress`도 between-only

삼성헬스의 M1-S 사양(`absD_star ~ C(gender) + coverage_ratio + mean_stress`, Fitbit은 지표 1개뿐이라 `C(indicator)` 불필요)을 그대로 적용해 재현모형(timeInBed)과 정의 민감도(minutesAsleep)를 나란히 구축.

추정방식(OLS+HC3, 참여자단위 부트스트랩→cluster-robust 폴백)도 삼성헬스와 동일하게 적용한다(8~12단계 공통).

In [9]:
# 삼성헬스 D-1A 설계에 맞춘 Fitbit 직접재현분석
# 핵심 사양 3가지:
#  (1) 웨어러블 수면시간 정의: timeInBed(각성포함, 삼성헬스 정의와 동일; minutesAsleep(각성제외)은 민감도)
#  (2) 집계 단위: person-level(삼성헬스처럼 그 사람 주차 평균 D를 먼저 낸 뒤 표준화)
#  (3) 모형: within/within-between 분해 없이, 삼성헬스의 M1-S와 동일한 between-only 스펙
#            absD_star ~ C(gender) + coverage_ratio + mean_stress  (Fitbit은 지표 1개뿐이라 C(indicator) 불필요)
import glob
import statsmodels.formula.api as smf

merged = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
merged['survey_date'] = pd.to_datetime(merged['survey_date']).dt.date

# ============================================================
# 1) timeInBed 기반 웨어러블 수면시간 재구축 (person-week)
#    - d2_merged_person_week.csv는 minutesAsleep(각성제외)만 포함하므로
#      timeInBed(각성포함)을 별도로 person-week 집계
# ============================================================
final_ids = sorted(merged['id'].unique())

# timeInBed(각성포함) person-week 값은 data/fitbit_wearable_person_week.csv 생성 시
# 이미 계산됨(coverage=0 결측처리 포함)
wearable_pw_tib = pd.read_csv(os.path.join(DATA, 'fitbit_wearable_person_week.csv'))
wearable_pw_tib['survey_date'] = pd.to_datetime(wearable_pw_tib['survey_date']).dt.date
tib_df = wearable_pw_tib[['id', 'survey_date', 'w_mean_sleep_minutes_tib']]

m = merged.merge(tib_df, on=['id', 'survey_date'], how='left')
print(f"timeInBed 병합 완료: {m['w_mean_sleep_minutes_tib'].notna().sum()}/{len(m)} 주 유효")

# 검산: timeInBed >= minutesAsleep 이어야 함(각성포함이 더 크거나 같음)
chk = m.dropna(subset=['w_mean_sleep_minutes_tib', 'w_mean_sleep_minutes'])
print(f"검산(timeInBed >= minutesAsleep): {(chk['w_mean_sleep_minutes_tib'] >= chk['w_mean_sleep_minutes'] - 1e-6).all()}")
print(f"평균 차이(timeInBed - minutesAsleep) = {(chk['w_mean_sleep_minutes_tib'] - chk['w_mean_sleep_minutes']).mean():.1f}분")

m['survey_sleep_hr'] = pd.to_numeric(m['q1-1'], errors='coerce') + pd.to_numeric(m['q1-2'], errors='coerce') / 60
m['wearable_sleep_hr_tib'] = m['w_mean_sleep_minutes_tib'] / 60
m['D_sleep_hr_tib'] = m['survey_sleep_hr'] - m['wearable_sleep_hr_tib']

# 민감도용(minutesAsleep, 각성제외) 병행
m['wearable_sleep_hr_asleep'] = m['w_mean_sleep_minutes'] / 60
m['D_sleep_hr_asleep'] = m['survey_sleep_hr'] - m['wearable_sleep_hr_asleep']

m.to_csv(os.path.join(RESULT, 'd2_sleep_gap_tib_pw.csv'), index=False)

# ============================================================
# 2) person-level 집계: 삼성헬스처럼 "그 사람의 여러 주 D를 먼저 평균"한 뒤
#    그 person-level D를 전참여자 분포로 표준화 -> |D_star_person|
#    (주의: 이는 주별 |D*|의 평균과 다름. person-level 평균D의 절대값이므로
#     "그 사람의 체계적 편향 크기"를 재는 삼성헬스의 정의와 정확히 일치시키기 위함)
# ============================================================
def build_person_level(dframe, dcol, label):
    valid = dframe.dropna(subset=[dcol, 'wearable_coverage'])
    g = valid.groupby('id').agg(
        D=(dcol, 'mean'),
        n_weeks=(dcol, 'size'),
        coverage_ratio=('wearable_coverage', 'mean'),
    )
    g = g[g['n_weeks'] >= MIN_WEEKS].reset_index()
    sd_D = g['D'].std()
    g['D_star'] = g['D'] / sd_D
    g['absD_star'] = g['D_star'].abs()
    print(f"\n[{label}] person-level n={len(g)}, SD(D)={sd_D:.4f} (across-person, MIN_WEEKS>={MIN_WEEKS})")
    return g

person_tib = build_person_level(m, 'D_sleep_hr_tib', 'timeInBed(각성포함, 주 재현대상)')
person_asleep = build_person_level(m, 'D_sleep_hr_asleep', 'minutesAsleep(각성제외, 민감도)')

# ============================================================
# 3) gender(q5) + mean_stress(=stress_between, 참여자당 1값) 병합
# ============================================================
survey_pw_g = pd.read_csv(os.path.join(DATA, 'fitbit_survey_person_week.csv'))
gender_df = survey_pw_g[['id', 'gender']].dropna().drop_duplicates(subset=['id'])

stress_pw = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))[['id', 'stress_between']].drop_duplicates('id')
stress_pw = stress_pw.rename(columns={'stress_between': 'psychological_distress_between'})

for name, person_df in [('timeInBed', person_tib), ('minutesAsleep', person_asleep)]:
    person_df = person_df.merge(gender_df, on='id', how='left').merge(stress_pw, on='id', how='left')
    if name == 'timeInBed':
        person_tib = person_df
    else:
        person_asleep = person_df

print(f"\n[timeInBed] gender/stress 병합 후 결측: gender={person_tib['gender'].isna().sum()}, "
      f"stress={person_tib['psychological_distress_between'].isna().sum()} (총 {len(person_tib)}명)")

person_tib.to_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib.csv'), index=False)
person_asleep.to_csv(os.path.join(RESULT, 'd2_person_sleeptime_asleep.csv'), index=False)

# ============================================================
# 4) 삼성헬스의 M1-S와 동일 스펙(단, C(indicator) 없음: Fitbit은 지표가 1개뿐)
#    absD_star ~ C(gender) + coverage_ratio + psychological_distress_between
# ============================================================
def fit_reproduction(person_df, label):
    d = person_df.dropna(subset=['absD_star', 'gender', 'coverage_ratio', 'psychological_distress_between']).copy()
    d['gender'] = d['gender'].astype(int).astype('category')
    if len(d) < 8:
        print(f"[{label}] 표본부족(n={len(d)}) - 스킵")
        return None
    fit = smf.ols('absD_star ~ C(gender) + coverage_ratio + psychological_distress_between', d).fit(cov_type='HC3')
    coef = fit.params['psychological_distress_between']
    ci = fit.conf_int().loc['psychological_distress_between'].tolist()
    p = fit.pvalues['psychological_distress_between']
    print(f"\n=== [{label}] n={len(d)} ===")
    print(fit.summary().tables[1])
    print(f"-> psychological_distress_between: coef={coef:.4f}, 95%CI=[{ci[0]:.4f}, {ci[1]:.4f}], p={p:.4f}")
    return {'label': label, 'n': len(d), 'coef': coef, 'ci_lo': ci[0], 'ci_hi': ci[1], 'p': p,
            'stress_sd': d['psychological_distress_between'].std(), 'fit': fit, 'data': d}

print("\n" + "=" * 70)
print("[주분석] Fitbit direct reproduction: timeInBed 기반 |D*| ~ gender+coverage+distress_between")
print("(삼성헬스의 M1-S와 동일 모형 스펙; 삼성헬스 계수: coef=0.2587 (혼합모형 p=0.0048; 군집강건 p=0.026), n_obs=193,n_id=55)")
print("=" * 70)
res_tib = fit_reproduction(person_tib, 'timeInBed(주분석, 삼성헬스 정의와 동일)')

print("\n" + "=" * 70)
print("[민감도] minutesAsleep 기반(각성제외) 동일 모형")
print("=" * 70)
res_asleep = fit_reproduction(person_asleep, 'minutesAsleep(민감도, 각성제외)')

print("\n저장 완료: d2_sleep_gap_tib_pw.csv, d2_person_sleeptime_tib.csv, d2_person_sleeptime_asleep.csv")


timeInBed 병합 완료: 242/386 주 유효
검산(timeInBed >= minutesAsleep): True
평균 차이(timeInBed - minutesAsleep) = 57.9분

[timeInBed(각성포함, 주 재현대상)] person-level n=29, SD(D)=0.7673 (across-person, MIN_WEEKS>=3)

[minutesAsleep(각성제외, 민감도)] person-level n=29, SD(D)=0.8377 (across-person, MIN_WEEKS>=3)

[timeInBed] gender/stress 병합 후 결측: gender=0, stress=0 (총 29명)

[주분석] Fitbit direct reproduction: timeInBed 기반 |D*| ~ gender+coverage+distress_between
(삼성헬스의 M1-S와 동일 모형 스펙; 삼성헬스 계수: coef=0.2587 (혼합모형 p=0.0048; 군집강건 p=0.026), n_obs=193,n_id=55)

=== [timeInBed(주분석, 삼성헬스 정의와 동일)] n=29 ===
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -1.0784      1.453     -0.742      0.458      -3.926       1.769
C(gender)[T.2]                     0.2844      0.259      1.096      0.273      -0.224       0.793
coverage_ratio              

## 9단계: 삼성헬스 수면시간 지표만 추출해 동일 사양 재추정

삼성헬스 D-1A 사양에서 걷기 지표를 빼고 수면시간(주중/주말) 지표만 남겨 재추정한다. 입력은 `samsunghealth/data`의 person-level 수면 간극 D(`wearable_sleep_person.csv`)와 설문 스트레스·성별(`survey_final.csv`)이다. Fitbit과 완전히 동일한 구조(단일지표, `C(indicator)` 없음)로 축소한 버전도 함께 산출해 Fitbit과 직접 비교 가능하게 한다.

In [10]:
# 삼성헬스 D-1A에서 걷기 지표를 빼고 수면시간(주중+주말) 지표만 남겨 재추정 ->
# Fitbit의 timeInBed 기반 재현모형과 "수면시간 한정" 기준으로 직접 비교 가능한 삼성헬스측 계수를 얻는다.
import statsmodels.formula.api as smf

SH_DATA = os.path.join("..", "samsunghealth", "data")
sleep_person = pd.read_csv(os.path.join(SH_DATA, 'wearable_sleep_person.csv'))
survey_sh = pd.read_csv(os.path.join(SH_DATA, 'survey_final.csv'))

gender_by_id = survey_sh[['id', 'gender']].dropna().drop_duplicates(subset=['id']).set_index('id')['gender']
mean_stress = survey_sh.groupby('id')['stress'].mean()

d_wd_sleep = sleep_person[sleep_person['main_weekday_flag']][['id', 'D_weekday', 'n_weekday_nights_total']].copy()
d_wd_sleep.columns = ['id', 'D', 'cov_raw']
d_wd_sleep['coverage_ratio'] = d_wd_sleep['cov_raw'] / 20

d_we_sleep = sleep_person[sleep_person['main_weekend_flag']][['id', 'D_weekend', 'n_weekend_nights_total']].copy()
d_we_sleep.columns = ['id', 'D', 'cov_raw']
d_we_sleep['coverage_ratio'] = d_we_sleep['cov_raw'] / 8

# ============================================================
# (A) 수면시간 2개 지표(주중/주말)만 남기고 재표준화
#     -> D_star의 SD는 "수면시간만의" 분포로 다시 계산됨(삼성헬스 4지표 SD와 다름)
#     -> C(indicator)로 주중/주말 차이는 통제(삼성헬스 원래 구조 보존)
# ============================================================
frames_sleep_only = {'주중_수면시간': d_wd_sleep, '주말_수면시간': d_we_sleep}
rows = []
for name, frame in frames_sleep_only.items():
    for _, r in frame.iterrows():
        rows.append({'id': r['id'], 'indicator': name, 'D': r['D'], 'coverage_ratio': r['coverage_ratio']})
d1a_sleep = pd.DataFrame(rows)
sd_by_ind = d1a_sleep.groupby('indicator')['D'].std()
d1a_sleep['D_star'] = d1a_sleep.apply(lambda r: r['D'] / sd_by_ind[r['indicator']], axis=1)
d1a_sleep['absD_star'] = d1a_sleep['D_star'].abs()
d1a_sleep['gender'] = d1a_sleep['id'].map(gender_by_id)
d1a_sleep['mean_stress'] = d1a_sleep['id'].map(mean_stress)
d1a_sleep['mean_stress_z'] = (d1a_sleep['mean_stress'] - d1a_sleep['mean_stress'].mean()) / d1a_sleep['mean_stress'].std()

d = d1a_sleep.dropna(subset=['absD_star', 'gender', 'coverage_ratio', 'mean_stress']).copy()
d['gender'] = d['gender'].astype(int).astype('category')

print("=" * 70)
print("[삼성헬스, 수면시간 지표만] absD_star ~ C(indicator) + C(gender) + coverage_ratio + mean_stress")
print("(삼성헬스 4지표 결과: coef=0.2587 (혼합모형 p=0.0048; 군집강건 p=0.026), n_obs=193, n_id=55)")
print("=" * 70)
fit = smf.mixedlm("absD_star ~ C(indicator) + C(gender) + coverage_ratio + mean_stress", d, groups=d['id']).fit(reml=True)
print(f"n_obs={len(d)}, n_id={d['id'].nunique()}")
print(fit.summary().tables[1])
coef_raw = fit.params['mean_stress']
ci_raw = fit.conf_int().loc['mean_stress'].tolist()
p_raw = fit.pvalues['mean_stress']
print(f"-> mean_stress(raw): coef={coef_raw:.4f}, 95%CI=[{ci_raw[0]:.4f},{ci_raw[1]:.4f}], p={p_raw:.4f}")

fit_z = smf.mixedlm("absD_star ~ C(indicator) + C(gender) + coverage_ratio + mean_stress_z", d, groups=d['id']).fit(reml=True)
coef_z = fit_z.params['mean_stress_z']
ci_z = fit_z.conf_int().loc['mean_stress_z'].tolist()
p_z = fit_z.pvalues['mean_stress_z']
print(f"-> mean_stress(표준화, 1SD단위): coef={coef_z:.4f}, 95%CI=[{ci_z[0]:.4f},{ci_z[1]:.4f}], p={p_z:.4f}")

# ============================================================
# (B) Fitbit과 구조를 완전히 맞춘 버전: 주중/주말을 사람별로 다시 평균해 "지표 1개"로 축소
#     (C(indicator) 없이 Fitbit과 동일한 단일지표 person-level 모형)
# ============================================================
print("\n" + "=" * 70)
print("[삼성헬스, 수면시간 단일값(주중+주말 평균)] absD_star ~ C(gender) + coverage_ratio + mean_stress")
print("(Fitbit과 완전히 동일한 모형구조: 지표 1개, C(indicator) 없음)")
print("=" * 70)
combined = d1a_sleep.groupby('id').agg(
    D_star_mean=('D_star', 'mean'),
    coverage_ratio=('coverage_ratio', 'mean'),
).reset_index()
combined['absD_star_single'] = combined['D_star_mean'].abs()
combined['gender'] = combined['id'].map(gender_by_id)
combined['mean_stress'] = combined['id'].map(mean_stress)
combined['mean_stress_z'] = (combined['mean_stress'] - combined['mean_stress'].mean()) / combined['mean_stress'].std()

dc = combined.dropna(subset=['absD_star_single', 'gender', 'coverage_ratio', 'mean_stress']).copy()
dc['gender'] = dc['gender'].astype(int).astype('category')
fit_single = smf.ols("absD_star_single ~ C(gender) + coverage_ratio + mean_stress", dc).fit(cov_type='HC3')
print(f"n={len(dc)}")
print(fit_single.summary().tables[1])
coef_single = fit_single.params['mean_stress']
ci_single = fit_single.conf_int().loc['mean_stress'].tolist()
p_single = fit_single.pvalues['mean_stress']
print(f"-> mean_stress(raw): coef={coef_single:.4f}, 95%CI=[{ci_single[0]:.4f},{ci_single[1]:.4f}], p={p_single:.4f}")

fit_single_z = smf.ols("absD_star_single ~ C(gender) + coverage_ratio + mean_stress_z", dc).fit(cov_type='HC3')
coef_single_z = fit_single_z.params['mean_stress_z']
ci_single_z = fit_single_z.conf_int().loc['mean_stress_z'].tolist()
p_single_z = fit_single_z.pvalues['mean_stress_z']
print(f"-> mean_stress(표준화, 1SD단위): coef={coef_single_z:.4f}, 95%CI=[{ci_single_z[0]:.4f},{ci_single_z[1]:.4f}], p={p_single_z:.4f}")

# Fitbit과의 비교표 작성용
sh_reest = pd.DataFrame([
    {'모형': '삼성헬스 수면시간만(C(indicator) 유지, mixedlm)', 'n_obs': len(d), 'n_id': d['id'].nunique(),
     'coef_raw': coef_raw, 'p_raw': p_raw, 'coef_z': coef_z, 'ci_z_lo': ci_z[0], 'ci_z_hi': ci_z[1], 'p_z': p_z,
     'stress_sd': d1a_sleep['mean_stress'].std()},
    {'모형': '삼성헬스 수면시간 단일값(Fitbit과 동일구조, OLS)', 'n_obs': len(dc), 'n_id': dc['id'].nunique(),
     'coef_raw': coef_single, 'p_raw': p_single, 'coef_z': coef_single_z, 'ci_z_lo': ci_single_z[0], 'ci_z_hi': ci_single_z[1], 'p_z': p_single_z,
     'stress_sd': combined['mean_stress'].std()},
])


[삼성헬스, 수면시간 지표만] absD_star ~ C(indicator) + C(gender) + coverage_ratio + mean_stress
(삼성헬스 4지표 결과: coef=0.2587 (혼합모형 p=0.0048; 군집강건 p=0.026), n_obs=193, n_id=55)
n_obs=89, n_id=47
                          Coef. Std.Err.       z  P>|z|  [0.025  0.975]
Intercept                 1.249    0.507   2.463  0.014   0.255   2.243
C(indicator)[T.주중_수면시간]  -0.090    0.119  -0.757  0.449  -0.324   0.143
C(gender)[T.2]            0.252    0.152   1.663  0.096  -0.045   0.549
coverage_ratio           -1.236    0.489  -2.527  0.012  -2.195  -0.277
mean_stress               0.223    0.117   1.905  0.057  -0.006   0.453
Group Var                 0.098    0.162                               
-> mean_stress(raw): coef=0.2232, 95%CI=[-0.0064,0.4529], p=0.0568
-> mean_stress(표준화, 1SD단위): coef=0.1418, 95%CI=[-0.0041,0.2877], p=0.0568

[삼성헬스, 수면시간 단일값(주중+주말 평균)] absD_star ~ C(gender) + coverage_ratio + mean_stress
(Fitbit과 완전히 동일한 모형구조: 지표 1개, C(indicator) 없음)
n=47
                     coef    std err      

## 10단계: 삼성헬스 vs Fitbit 표준화 비교 + dataset×distress 정식 상호작용 검정

삼성헬스·Fitbit은 스트레스 척도 자체가 다르므로 원계수 직접비교 금지 — 두 계수를 각자 1SD 표준화한 뒤 비교하고, 두 표본을 풀링해 `dataset × distress_z` 상호작용을 정식 검정한다(비유의하면 "두 데이터셋이 다르다"는 근거 부족이라는 뜻이지, "같다/재현됐다"는 뜻은 아님).

**참고**: 성별 코드(1/2)의 실제 남/여 대응은 삼성헬스·Fitbit 모두 원자료에서 검증된 적이 없다. 풀링모형에서 성별을 (a)공통코드로 취급/(b)제외/(c)데이터셋별 gender group으로 분리 통제 — 3가지로 비교한 결과 결론(상호작용 비유의)은 동일했고, 1/2 대응을 몰라도 되는 (c)를 최종 채택했다. coverage_ratio는 캐노니컬(union-day 기반) 정의를 사용한다.

In [11]:
# 삼성헬스(수면시간만, Fitbit과 동일 단일지표 구조로 축소) vs Fitbit(timeInBed 기반) 직접비교
# - 두 데이터셋은 스트레스 척도가 근본적으로 다르므로 원계수 직접비교 금지 -> 1SD 표준화 단위로만 비교
# - 두 모형을 하나로 합쳐 dataset x distress(표준화) 상호작용 정식 검정
# - 성별 코드(1/2)의 실제 남/여 대응은 삼성헬스·Fitbit 모두 원자료에서 확인된 바 없음
#   -> 풀링모형에서 성별 공통코드/제외/데이터셋별 그룹통제 3가지를 민감도로 비교
import statsmodels.formula.api as smf

# ============================================================
# 1) Fitbit: timeInBed 기반(11단계 union-pooling, 캐노니컬 coverage 포함)
# ============================================================
p02 = pd.read_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib_unionpooled.csv'))
p02['gender'] = p02['gender'].astype(int).astype('category')
p02['stress_z'] = (p02['psychological_distress_between'] - p02['psychological_distress_between'].mean()) / p02['psychological_distress_between'].std()
p02['coverage_z'] = (p02['coverage_ratio'] - p02['coverage_ratio'].mean()) / p02['coverage_ratio'].std()

fit02 = smf.ols('absD_star ~ C(gender) + coverage_ratio + stress_z', p02).fit(cov_type='HC3')
coef02 = fit02.params['stress_z']
ci02 = fit02.conf_int().loc['stress_z'].tolist()
p02_p = fit02.pvalues['stress_z']
print(f"[Fitbit, timeInBed(union-pooled), n={len(p02)}] stress_z: coef={coef02:.4f}, 95%CI=[{ci02[0]:.4f},{ci02[1]:.4f}], p={p02_p:.4f}")

# 단일 참여자 영향력 확인(S001, 최대영향점)
p02_ex = p02[p02['id'] != 'S001']
fit02_ex = smf.ols('absD_star ~ C(gender) + coverage_ratio + stress_z', p02_ex).fit(cov_type='HC3')
sign_flip = (fit02_ex.params['stress_z'] > 0) != (coef02 > 0)
print(f"  (참고, 최대영향점 S001 제외 n={len(p02_ex)}): coef={fit02_ex.params['stress_z']:.4f}, p={fit02_ex.pvalues['stress_z']:.4f}"
      f"  <- {'부호가 바뀜: 점추정치가 소수 영향점에 민감함' if sign_flip else '부호 유지'}")

# 해석용 원단위(시간) 효과: D_star의 표준편차가 몇 시간에 해당하는지 역산
raw02 = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_tib_pw.csv'))
valid02 = raw02.dropna(subset=['D_sleep_hr_tib'])
pm02 = valid02.groupby('id')['D_sleep_hr_tib'].mean()
sd_D02_hr = pm02.std()
print(f"  (참고) Fitbit SD(D)={sd_D02_hr:.3f}시간 -> D_star 1단위 = {sd_D02_hr*60:.1f}분")

# ============================================================
# 2) 삼성헬스: 수면시간 단일값(주중+주말 평균, C(indicator) 없는 버전) - 9단계 결과 재사용
# ============================================================
print("\n[삼성헬스 재추정 요약 (9단계)]")
print(sh_reest.to_string(index=False))

row_single = sh_reest[sh_reest['모형'].str.contains('단일값')].iloc[0]
print(f"\n[삼성헬스, 수면시간 단일값, n={row_single['n_obs']}] stress_z: coef={row_single['coef_z']:.4f}, "
      f"95%CI=[{row_single['ci_z_lo']:.4f},{row_single['ci_z_hi']:.4f}], p={row_single['p_z']:.4f}")

# ============================================================
# 3) 표준화 계수 직접비교 (1SD 스트레스 단위 기준)
# ============================================================
print("\n" + "=" * 70)
print("표준화(1SD 디스트레스) 계수 직접비교: 삼성헬스(수면시간단일) vs Fitbit(timeInBed)")
print("=" * 70)
print(f"  삼성헬스: coef_z={row_single['coef_z']:.4f} [95%CI {row_single['ci_z_lo']:.4f},{row_single['ci_z_hi']:.4f}], p={row_single['p_z']:.4f}, n={int(row_single['n_obs'])}")
print(f"  Fitbit: coef_z={coef02:.4f} [95%CI {ci02[0]:.4f},{ci02[1]:.4f}], p={p02_p:.4f}, n={len(p02)}")
overlap = not (ci02[1] < row_single['ci_z_lo'] or ci02[0] > row_single['ci_z_hi'])
print(f"  -> 두 95%CI 중첩 여부: {'중첩됨(이질성 근거 약함)' if overlap else '중첩 안됨(방향/크기가 다를 가능성)'}")
print(f"  -> 부호: 삼성헬스={'양(+)' if row_single['coef_z']>0 else '음(-)'}, Fitbit={'양(+)' if coef02>0 else '음(-)'}"
      f" {'(전체표본 점추정치는 삼성헬스와 같은 부호)' if (row_single['coef_z']>0) == (coef02>0) else '(부호 불일치)'}")

# ============================================================
# 4) 정식 dataset x distress(표준화) 상호작용 검정 (풀링)
#    - coverage_ratio는 데이터셋마다 척도(정의)가 달라 각자 z표준화 후 풀링
#    - 반복측정 구조 없음(참여자당 1행)이므로 OLS로 충분, dataset을 고정효과로 포함
#    - 기준(reference) 데이터셋은 삼성헬스이므로 상호작용 계수는 "Fitbit − 삼성헬스" 기울기 차이
# ============================================================
print("\n" + "=" * 70)
print("[정식검정] dataset x distress(z) 상호작용 (삼성헬스 vs Fitbit, 풀링)")
print("=" * 70)

d01 = dc.rename(columns={'absD_star_single': 'absD_star'}).copy()
d01['stress_z'] = (d01['mean_stress'] - d01['mean_stress'].mean()) / d01['mean_stress'].std()
d01['coverage_z'] = (d01['coverage_ratio'] - d01['coverage_ratio'].mean()) / d01['coverage_ratio'].std()
d01['dataset'] = '삼성헬스'
d01['gender'] = d01['gender'].astype(int)

p02_pool = p02.copy()
p02_pool['dataset'] = 'Fitbit'
p02_pool['gender'] = p02_pool['gender'].astype(int)

cols = ['id', 'dataset', 'absD_star', 'gender', 'coverage_z', 'stress_z']
pooled = pd.concat([d01[cols], p02_pool[cols]], ignore_index=True)
pooled['gender_code'] = pooled['gender'].astype('category')
pooled['dataset'] = pd.Categorical(pooled['dataset'], categories=['삼성헬스', 'Fitbit'])
# 데이터셋별 성별코드 그룹 (삼성헬스의 1/2, Fitbit의 1/2이 같은 성별이라는 가정 없이 4개 범주로 분리)
pooled['dataset_gender_group'] = (pooled['dataset'].astype(str) + '_g' + pooled['gender'].astype(str)).astype('category')

print(f"풀링 표본: n={len(pooled)} (삼성헬스: n={len(d01)}, Fitbit: n={len(p02_pool)})")

# ============================================================
# 4-보완: 성별 코드 처리방식 3가지 민감도 비교
#    (a) 공통코드로 취급(삼성헬스의 1과 Fitbit의 1이 같다고 가정)
#    (b) 성별 제외
#    (c) 데이터셋별 gender group으로 분리 통제(1/2 대응을 몰라도 됨, 가장 안전)
# ============================================================
print("\n[성별 코드 처리방식 3가지 민감도 비교 - dataset x stress_z 상호작용 계수]")
specs = {
    '(a) 공통코드(가정 필요)': 'absD_star ~ C(dataset)*stress_z + C(gender_code) + coverage_z',
    '(b) 성별 제외': 'absD_star ~ C(dataset)*stress_z + coverage_z',
    # (c): C(dataset_gender_group)가 이미 dataset을 포함하므로 C(dataset) 주효과를 별도로 넣으면 완전공선이 되어
    #      상호작용항만 C(dataset):stress_z로 명시하고 주효과는 dataset_gender_group에만 맡김
    '(c) 데이터셋별 gender group(1/2 대응 불필요, 권장)': 'absD_star ~ C(dataset_gender_group) + stress_z + C(dataset):stress_z + coverage_z',
}
for label, formula in specs.items():
    fit = smf.ols(formula, pooled).fit(cov_type='HC3')
    inter_term = [t for t in fit.params.index if 'dataset' in t and 'stress_z' in t and 'gender' not in t][0]
    ci_i = fit.conf_int().loc[inter_term].tolist()
    print(f"  {label}: coef={fit.params[inter_term]:.4f}, 95%CI=[{ci_i[0]:.4f},{ci_i[1]:.4f}], p={fit.pvalues[inter_term]:.4f}")
print("  -> (c)를 최종 보고 기준으로 채택.")

fit_final = smf.ols(specs['(c) 데이터셋별 gender group(1/2 대응 불필요, 권장)'], pooled).fit(cov_type='HC3')
print("\n[최종 채택 모형 상세]")
print(fit_final.summary().tables[1])


[Fitbit, timeInBed(union-pooled), n=26] stress_z: coef=0.0671, 95%CI=[-0.3746,0.5089], p=0.7658
  (참고, 최대영향점 S001 제외 n=25): coef=-0.0577, p=0.6898  <- 부호가 바뀜: 점추정치가 소수 영향점에 민감함
  (참고) Fitbit SD(D)=1.280시간 -> D_star 1단위 = 76.8분

[삼성헬스 재추정 요약 (9단계)]
                                  모형  n_obs  n_id  coef_raw    p_raw   coef_z   ci_z_lo  ci_z_hi      p_z  stress_sd
삼성헬스 수면시간만(C(indicator) 유지, mixedlm)     89    47  0.223243 0.056779 0.141780 -0.004090 0.287651 0.056779   0.635095
    삼성헬스 수면시간 단일값(Fitbit과 동일구조, OLS)     47    47  0.167831 0.217850 0.107416 -0.063433 0.278265 0.217850   0.640025

[삼성헬스, 수면시간 단일값, n=47] stress_z: coef=0.1074, 95%CI=[-0.0634,0.2783], p=0.2179

표준화(1SD 디스트레스) 계수 직접비교: 삼성헬스(수면시간단일) vs Fitbit(timeInBed)
  삼성헬스: coef_z=0.1074 [95%CI -0.0634,0.2783], p=0.2179, n=47
  Fitbit: coef_z=0.0671 [95%CI -0.3746,0.5089], p=0.7658, n=26
  -> 두 95%CI 중첩 여부: 중첩됨(이질성 근거 약함)
  -> 부호: 삼성헬스=양(+), Fitbit=양(+) (전체표본 점추정치는 삼성헬스와 같은 부호)

[정식검정] dataset x distress(z) 상호작용 (삼성헬스 vs Fi

**결과**: 심리적 변수와 데이터셋의 상호작용은 유의하지 않았다(β=−0.021, 95% CI[−0.463, 0.421], P=.925). 수면시간 지표에 한정했을 때 삼성헬스와 Fitbit 간 스트레스-간극 기울기가 통계적으로 다르다는 근거가 없음을 의미한다(재현/동등성을 입증하는 것은 아님).

### 11-보완: 캐노니컬 person-level coverage_ratio 단일화

특정 결과변수의 결측 여부와 무관하게 "그 사람이 응답한 모든 주차"를 union으로 묶은 단일 캐노니컬 coverage_ratio를 정의한다. 10단계 풀링모형, 11단계 D-1A, 12단계 D-1B 모형이 모두 이 값을 사용한다.

In [12]:
# 캐노니컬 person-level coverage_ratio 단일화
# 특정 결과변수(수면시간/수면질) 결측 여부와 무관하게, 그 사람이 응답한 모든 주차의 윈도우를
# union으로 묶어 "이 사람에 대해 웨어러블 데이터가 실제로 얼마나 있는가"를 재는 하나의
# canonical coverage_ratio를 만들고, 모든 person-level 모형(10단계 풀링, 11단계 D-1A, 12단계 D-1B)이
# 동일하게 사용한다.

COVERAGE_FLOOR = 0.5

merged = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
merged['survey_date'] = pd.to_datetime(merged['survey_date']).dt.date
final_ids = sorted(merged['id'].unique())

# 캐노니컬 coverage_ratio(그 사람이 응답한 모든 주차 윈도우의 union 기준)는
# data/fitbit_wearable_person_week.csv 생성 시 이미 계산되어 각 person-week 행에
# 동일하게 병합돼 있음
wearable_pw_cov = pd.read_csv(os.path.join(DATA, 'fitbit_wearable_person_week.csv'))
cov = wearable_pw_cov[['id', 'n_weeks_responded', 'n_valid_days', 'n_total_days', 'coverage_ratio']].drop_duplicates(subset=['id'])
cov_floor = cov[cov['coverage_ratio'] >= COVERAGE_FLOOR].copy()
print(f"캐노니컬 coverage_ratio: n={len(cov)} (전체), 하한{COVERAGE_FLOOR} 이상: n={len(cov_floor)}")
print(cov.to_string(index=False))
cov.to_csv(os.path.join(RESULT, 'd2_canonical_coverage.csv'), index=False)
print("\n저장 완료: d2_canonical_coverage.csv (하한 미적용 전체값 - 각 모형에서 필요시 하한 적용)")


캐노니컬 coverage_ratio: n=42 (전체), 하한0.5 이상: n=28
  id  n_weeks_responded  n_valid_days  n_total_days  coverage_ratio
S001                 10            67            67        1.000000
S004                 10            60            66        0.909091
S006                 11             0            75        0.000000
S007                 10            17            70        0.242857
S008                 11            46            71        0.647887
S012                 10            42            69        0.608696
S013                 11            62            64        0.968750
S015                 10             0            68        0.000000
S016                 11            49            71        0.690141
S019                 11            68            75        0.906667
S021                 10             6            69        0.086957
S026                  7            32            48        0.666667
S030                  5            34            35        0.971429
S

## 11단계: person-level 집계 union-pooling 수정

삼성헬스와 동일하게 참여자의 여러 주차 윈도우를 union으로 묶어 유니크한 날짜 단위로 한 번만 집계한다(day-level pooling). 연속 설문 간격의 24.6%가 7일 미만이라 윈도우가 겹치므로, 주차별 평균의 평균으로는 겹치는 날짜가 중복 반영된다. 또한 삼성헬스처럼 최소 커버리지 하한(~50%)을 적용해 저커버리지 참여자(예: 유효일 5일뿐인 극단사례)가 결과를 왜곡하지 않도록 한다.

In [13]:
# 삼성헬스의 person-level 집계 방식(build_sleep_person_level)에 맞춘 union-day pooling.
# 연속 설문 간격의 24.6%(59/240)가 7일 미만이라 주차 윈도우가 겹치므로, 주차별 평균의 평균으로는
# 겹치는 날짜가 여러 주차에 중복 반영되어 가중치가 왜곡된다.
# 따라서 참여자별로 모든 주차 윈도우의 날짜를 union으로 묶어 유니크한 날짜 단위로 웨어러블 값을
# 한 번만 집계한다(day-level pooling). 설문(S)은 주 1회 응답이라 겹침이 없어 주차별 응답의 단순평균을 쓴다.
import statsmodels.formula.api as smf


merged = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
merged['survey_date'] = pd.to_datetime(merged['survey_date']).dt.date
merged['survey_sleep_hr'] = pd.to_numeric(merged['q1-1'], errors='coerce') + pd.to_numeric(merged['q1-2'], errors='coerce') / 60
final_ids = sorted(merged['id'].unique())

# ============================================================
# 1) 일 단위 웨어러블 시계열 준비 (timeInBed / minutesAsleep 둘 다)
# ============================================================
# 삼성헬스의 build_sleep_person_level과 동일한 논리(참여자별 설문응답 주차 윈도우를 union하여
# day-level로 한 번만 집계)로 만든 union-pooled 수면 person-level 값은
# data/fitbit_sleep_person_unionpooled.csv 생성 시 이미 계산됨
person = pd.read_csv(os.path.join(DATA, 'fitbit_sleep_person_unionpooled.csv'))
person = person.rename(columns={'coverage_ratio_sleeptime_only': 'coverage_ratio'})

# ============================================================
# 2-보완: coverage_ratio를 캐노니컬 버전(d2_canonical_coverage.csv, 11-보완)으로 통일
#    - 위 coverage_ratio는 survey_sleep_hr 유효 주차만으로 union을 잡아 특정 결과변수(수면시간)에 종속되므로,
#      10단계 풀링·12단계 D-1B와 정의를 통일하기 위해 "그 사람이 응답한 모든 주차" 기준 캐노니컬 값으로 교체
# ============================================================
person = person.rename(columns={'coverage_ratio': 'coverage_ratio_sleeptime_only'})
canon_cov = pd.read_csv(os.path.join(RESULT, 'd2_canonical_coverage.csv'))[['id', 'coverage_ratio']]
person = person.merge(canon_cov, on='id', how='left')
print(f"캐노니컬 coverage_ratio와 수면시간전용 coverage_ratio 상관: "
      f"{person['coverage_ratio'].corr(person['coverage_ratio_sleeptime_only']):.3f}")

# 삼성헬스와 동일한 포함기준: 응답 주차 >= MIN_WEEKS + 삼성헬스처럼 최소 커버리지 하한(~50%, 10/20·4/8과 동일 취지)
# (하한 없이는 union 유효일이 극소수인 사람의 평균이 튄값 하나에 좌우될 수 있음 - S605 사례로 확인)
COVERAGE_FLOOR = 0.5
person_main_nofloor = person[(person['n_weeks'] >= MIN_WEEKS) & person['W_tib'].notna()].copy()
person_main = person_main_nofloor[person_main_nofloor['coverage_ratio'] >= COVERAGE_FLOOR].copy()
print(f"union-pooling 재계산(커버리지 하한 {COVERAGE_FLOOR} 적용): person-level n={len(person_main)} "
      f"(하한 미적용 n={len(person_main_nofloor)})")
dropped = person_main_nofloor[person_main_nofloor['coverage_ratio'] < COVERAGE_FLOOR]
if len(dropped) > 0:
    print(f"  하한 미달로 제외: {dropped['id'].tolist()} (coverage_ratio={dropped['coverage_ratio'].round(3).tolist()})")
print(person_main[['id', 'n_weeks', 'n_valid_days', 'n_total_days', 'coverage_ratio', 'S_person', 'W_tib', 'D_tib']].to_string(index=False))

# ============================================================
# 3) 주차평균의 평균 방식(d2_person_sleeptime_tib.csv)과 나란히 비교
# ============================================================
old = pd.read_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib.csv'))[['id', 'D_star', 'coverage_ratio']]
old.columns = ['id', 'D_star_old', 'coverage_ratio_old']
cmp = person_main.merge(old, on='id', how='inner')
sd_new = person_main['D_tib'].std()
cmp['D_star_new'] = cmp['D_tib'] / sd_new
print(f"\nunion-pooling SD(D_tib)={sd_new:.4f} (주차평균 방식 SD와 비교)")
print("\n=== 주차평균의 평균 vs union day-level pooling D_star 비교 ===")
print(cmp[['id', 'D_star_old', 'D_star_new', 'coverage_ratio_old', 'coverage_ratio']].to_string(index=False))
print(f"\n두 D_star 간 상관: {cmp['D_star_old'].corr(cmp['D_star_new']):.3f}")
print(f"두 coverage_ratio 간 상관: {cmp['coverage_ratio_old'].corr(cmp['coverage_ratio']):.3f}")

# ============================================================
# 4) gender + psychological_distress_between 병합 후 동일 모형 재적합
# ============================================================
survey_pw_g24 = pd.read_csv(os.path.join(DATA, 'fitbit_survey_person_week.csv'))
gender_df = survey_pw_g24[['id', 'gender']].dropna().drop_duplicates(subset=['id'])

stress_pw = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))[['id', 'stress_between']].drop_duplicates('id')
stress_pw = stress_pw.rename(columns={'stress_between': 'psychological_distress_between'})

person_main['absD_star'] = (person_main['D_tib'] / sd_new).abs()
person_main = person_main.merge(gender_df, on='id', how='left').merge(stress_pw, on='id', how='left')
person_main.to_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib_unionpooled.csv'), index=False)

d = person_main.dropna(subset=['absD_star', 'gender', 'coverage_ratio', 'psychological_distress_between']).copy()
d['gender'] = d['gender'].astype(int).astype('category')
print("\n" + "=" * 70)
print("[재현모형, union-pooling] absD_star ~ C(gender) + coverage_ratio + psychological_distress_between")
print("(삼성헬스 base_code.ipynb의 소표본 OLS 추정관행에 맞춰 HC3 강건표준오차 적용)")
print("=" * 70)
fit = smf.ols('absD_star ~ C(gender) + coverage_ratio + psychological_distress_between', d).fit(cov_type='HC3')
print(f"n={len(d)}")
print(fit.summary().tables[1])
coef = fit.params['psychological_distress_between']
ci = fit.conf_int().loc['psychological_distress_between'].tolist()
p = fit.pvalues['psychological_distress_between']
print(f"-> psychological_distress_between(HC3): coef={coef:.4f}, 95%CI=[{ci[0]:.4f},{ci[1]:.4f}], p={p:.4f}")

# S001 민감도 재확인
d_ex = d[d['id'] != 'S001']
if len(d_ex) > 0:
    fit_ex = smf.ols('absD_star ~ C(gender) + coverage_ratio + psychological_distress_between', d_ex).fit(cov_type='HC3')
    print(f"   (S001 제외, n={len(d_ex)}): coef={fit_ex.params['psychological_distress_between']:.4f}, p={fit_ex.pvalues['psychological_distress_between']:.4f}")

print("\n저장 완료: d2_person_sleeptime_tib_unionpooled.csv")


캐노니컬 coverage_ratio와 수면시간전용 coverage_ratio 상관: 1.000
union-pooling 재계산(커버리지 하한 0.5 적용): person-level n=26 (하한 미적용 n=34)
  하한 미달로 제외: ['S021', 'S033', 'S605', 'S648', 'S984', 'S987', 'S988', 'S991'] (coverage_ratio=[0.087, 0.059, 0.053, 0.423, 0.488, 0.294, 0.485, 0.368])
  id  n_weeks  n_valid_days  n_total_days  coverage_ratio  S_person    W_tib     D_tib
S001       10            67            67        1.000000  5.050000 8.090643 -3.040643
S008       11            46            71        0.647887  7.757576 7.890385 -0.132809
S012       10            42            69        0.608696  6.800000 6.915476 -0.115476
S013       11            62            64        0.968750  8.090909 8.001852  0.089057
S016       11            49            71        0.690141  8.090909 8.700980 -0.610071
S019       11            68            75        0.906667  7.677273 7.703333 -0.026061
S030        5            34            35        0.971429  7.700000 7.759722 -0.059722
S040        9            62     

## 12단계: D-1B(수면질) person-level between-only 재구축 + 영향도 진단 + 검정력/MDE 시뮬레이션

- 삼성헬스의 `build_person_SW`와 동일 논리로 D-1B(수면질)를 person-level between-only로 재구축(person-week 단위 R_qual과 구분되는 person-level 지표)
- Cook's distance/leverage/DFBETA로 영향점 진단(제외 기준이 아니라 진단 기준으로만 사용)
- 관측된 X분포·잔차SD를 그대로 쓴 시뮬레이션으로 "D-1A(n=26)·D-1B(n=29) 표본에서 삼성헬스 크기의 효과를 검출할 검정력이 얼마였는가"를 정량화

In [14]:
# 1) Fitbit D-1B(수면질) between-only 재구축: 삼성헬스의 build_person_SW와 동일하게
#    "참여자별 S/W를 먼저 평균 -> 그 평균들을 참여자간 분포로 z표준화 -> R=Zs-Zw(참여자당 1개)"
#    (R_qual/absGap_sleepquality는 person-week 단위 z-diff이며, 여기서는 person-level로 다시 구성)
# 2) S001 영향도 진단(Cook's distance, leverage, DFBETA) - D-1A 재현모형 + 새 D-1B 모형 둘 다
# 3) 시뮬레이션 기반 검정력/MDE - D-1A(n=26)/D-1B(n=29) 표본에서 삼성헬스 효과크기를 검출할 힘이 있었는지
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

RNG = np.random.default_rng(2024)

# ============================================================
# 1) D-1B(수면질) between-only 재구축
# ============================================================
sleep_gap = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))
m = sleep_gap[['id', 'survey_date', 'Squal', 'w_mean_sleep_score', 'wearable_coverage']].copy()

valid = m.dropna(subset=['Squal', 'w_mean_sleep_score'])
pcount = valid.groupby('id').size()
valid_ids = pcount[pcount >= MIN_WEEKS].index
person_qual = valid[valid['id'].isin(valid_ids)].groupby('id').agg(
    S=('Squal', 'mean'), W=('w_mean_sleep_score', 'mean'), n_weeks=('Squal', 'size'),
).reset_index()
# coverage_ratio는 "주차별 wearable_coverage 평균"이 아니라 11-보완의 캐노니컬(union-day 기반) 값을 사용한다
# (D-1A/D-1B/풀링모형에 동일하게 적용)
canon_cov = pd.read_csv(os.path.join(RESULT, 'd2_canonical_coverage.csv'))[['id', 'coverage_ratio']]
person_qual = person_qual.merge(canon_cov, on='id', how='left')

zs = (person_qual['S'] - person_qual['S'].mean()) / person_qual['S'].std()
zw = (person_qual['W'] - person_qual['W'].mean()) / person_qual['W'].std()
person_qual['R_person'] = (zs - zw).values
person_qual['absR_person'] = person_qual['R_person'].abs()
print(f"[D-1B 수면질 person-level 재구축] n={len(person_qual)} (삼성헬스의 build_person_SW와 동일 논리)")

survey_pw_g28 = pd.read_csv(os.path.join(DATA, 'fitbit_survey_person_week.csv'))
gender_df = survey_pw_g28[['id', 'gender']].dropna().drop_duplicates(subset=['id'])

stress_pw = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))[['id', 'stress_between']].drop_duplicates('id')
stress_pw = stress_pw.rename(columns={'stress_between': 'psychological_distress_between'})

person_qual = person_qual.merge(gender_df, on='id', how='left').merge(stress_pw, on='id', how='left')
person_qual.to_csv(os.path.join(RESULT, 'd2_person_sleepquality_d1b.csv'), index=False)

d_b = person_qual.dropna(subset=['R_person', 'gender', 'coverage_ratio', 'psychological_distress_between']).copy()
d_b['gender'] = d_b['gender'].astype(int).astype('category')

print("\n" + "=" * 70)
print("[D-1B, 삼성헬스와 동일 사양] R_person(signed) ~ coverage_ratio + C(gender) + psychological_distress_between")
print("(삼성헬스 D-1B와 동일 추정방식: OLS+HC3 강건표준오차, 지표별 개별회귀)")
print("(삼성헬스 계수: coef=-0.735, 95%CI[-1.255,-0.215], p=0.0056, D-1B M1-S)")
print("=" * 70)
# 삼성헬스 노트북의 D-1B 추정방식은 mixedlm이 아니라 OLS+HC3 강건표준오차다(인당 지표당 관측치 1개라
# 혼합모형 대신 지표별 개별 OLS를 씀). 동일한 방식(OLS+HC3)을 적용한다.
fit_b = smf.ols('R_person ~ coverage_ratio + C(gender) + psychological_distress_between', d_b).fit(cov_type='HC3')
print(f"n={len(d_b)}")
print(fit_b.summary().tables[1])
coef_b = fit_b.params['psychological_distress_between']
ci_b = fit_b.conf_int().loc['psychological_distress_between'].tolist()
p_b = fit_b.pvalues['psychological_distress_between']
print(f"-> psychological_distress_between(HC3): coef={coef_b:.4f}, 95%CI=[{ci_b[0]:.4f},{ci_b[1]:.4f}], p={p_b:.4f}")

# ============================================================
# 2) S001 등 영향도 진단 - D-1A(timeInBed, union-pooled) 모형 + 새 D-1B 모형
# ============================================================
print("\n" + "=" * 70)
print("[영향도 진단] Cook's distance / leverage / DFBETA(distress_between)")
print("=" * 70)

def influence_report(fit, d, target_col, label):
    infl = fit.get_influence()
    cooks = infl.cooks_distance[0]
    leverage = infl.hat_matrix_diag
    dfbetas = infl.dfbetas
    target_idx = list(fit.params.index).index(target_col)
    dfb_target = dfbetas[:, target_idx]
    rep = pd.DataFrame({
        'id': d['id'].values, 'cooks_d': cooks, 'leverage': leverage, 'dfbeta_distress': dfb_target,
    }).sort_values('cooks_d', ascending=False)
    n = len(d)
    cooks_thr = 4 / n
    print(f"\n[{label}] n={n}, Cook's D 경험적 임계값(4/n)={cooks_thr:.3f}")
    print(rep.head(5).to_string(index=False))
    flagged = rep[rep['cooks_d'] > cooks_thr]
    print(f"임계값 초과(잠재적 영향점): {flagged['id'].tolist()}")
    return rep

d_tib = pd.read_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib_unionpooled.csv'))
d_tib_fit = d_tib.dropna(subset=['absD_star', 'gender', 'coverage_ratio', 'psychological_distress_between']).copy()
d_tib_fit['gender'] = d_tib_fit['gender'].astype(int).astype('category')
fit_tib = smf.ols('absD_star ~ C(gender) + coverage_ratio + psychological_distress_between', d_tib_fit).fit(cov_type='HC3')
influence_report(fit_tib, d_tib_fit, 'psychological_distress_between', 'D-1A(timeInBed, union-pooled) absD_star 모형')

influence_report(fit_b, d_b, 'psychological_distress_between', 'D-1B(수면질) R_person 모형')

# ============================================================
# 삼성헬스 노트북과 동일한 절차 - 참여자단위 부트스트랩(반복별 고유ID) 시도,
# 성공률<50%면 ID-cluster robust OLS로 대체
# (Fitbit은 참여자당 이미 1행뿐이라 "참여자단위"="행단위" 부트스트랩과 동일하게 축약됨)
# ============================================================
print("\n" + "=" * 70)
print("[삼성헬스와 동일 절차] 참여자단위 부트스트랩 -> 성공률<50%면 ID-cluster robust OLS로 대체")
print("=" * 70)

def boot_or_cluster_fallback(d, formula, target, label, n_target=1000, max_attempts=3000, seed=7):
    rng_local = np.random.default_rng(seed)
    ids = d['id'].values
    boot_coefs = []
    attempts = 0
    while len(boot_coefs) < n_target and attempts < max_attempts:
        attempts += 1
        sample_ids = rng_local.choice(ids, size=len(ids), replace=True)
        parts = []
        for rep_i, sid in enumerate(sample_ids):
            sub = d[d['id'] == sid].copy()
            sub['_boot_id'] = f"{sid}_{rep_i}"
            parts.append(sub)
        dboot = pd.concat(parts, ignore_index=True)
        try:
            res = smf.ols(formula, dboot).fit()
            boot_coefs.append(res.params[target])
        except Exception:
            continue
    boot_coefs = np.array(boot_coefs)
    success_rate = len(boot_coefs) / attempts if attempts > 0 else 0
    print(f"[{label}] 참여자단위 부트스트랩: 시도 {attempts}회 중 성공 {len(boot_coefs)}회 (성공률 {success_rate*100:.1f}%)")
    if success_rate < 0.5:
        print("  성공률<50% -> percentile CI는 선택편향 우려로 추론에 미사용. ID-cluster robust OLS로 대체")
        ols_res = smf.ols(formula, d).fit(cov_type='cluster', cov_kwds={'groups': d['id']})
        ci = ols_res.conf_int().loc[target]
        print(f"  ID-cluster robust OLS: {target} coef={ols_res.params[target]:.4f}, 95%CI=[{ci[0]:.4f},{ci[1]:.4f}], p={ols_res.pvalues[target]:.4f}")
        return ols_res, None
    else:
        ci_lo, ci_hi = np.percentile(boot_coefs, [2.5, 97.5])
        print(f"  참여자단위 부트스트랩 percentile 95%CI=[{ci_lo:.4f},{ci_hi:.4f}]")
        return None, (ci_lo, ci_hi)

boot_or_cluster_fallback(d_tib_fit, 'absD_star ~ C(gender) + coverage_ratio + psychological_distress_between',
                          'psychological_distress_between', 'D-1A(timeInBed)')
boot_or_cluster_fallback(d_b, 'R_person ~ coverage_ratio + C(gender) + psychological_distress_between',
                          'psychological_distress_between', 'D-1B(수면질)')

# ============================================================
# VIF 확인 (삼성헬스 노트북이 D-1A/D-1B 모두에서 확인하는 절차, Fitbit도 동일 적용)
# ============================================================
print("\n" + "=" * 70)
print("[VIF] 삼성헬스와 동일하게 공선성 확인 (coverage_ratio, psychological_distress_between)")
print("=" * 70)
for d_vif, label in [(d_tib_fit, 'D-1A(timeInBed)'), (d_b, 'D-1B(수면질)')]:
    Xv = sm.add_constant(d_vif[['coverage_ratio', 'psychological_distress_between']])
    print(f"[{label}]")
    for i, col in enumerate(Xv.columns):
        if col != 'const':
            print(f"  VIF({col}) = {variance_inflation_factor(Xv.values, i):.3f}")

# ============================================================
# 3) 시뮬레이션 기반 검정력/MDE (D-1A timeInBed 모형 기준, n=26)
#    - 관측된 X(gender, coverage_ratio, distress_between)와 잔차 SD를 그대로 사용
#    - "삼성헬스 정도의 표준화 효과(coef_z~0.107~0.161)가 있었다면 Fitbit이 몇 %로 검출했을지" 계산
#      (0.259는 삼성헬스 mean_stress 원점수(1~4점) 계수이며, 표준화하면 0.161. 아래 scenarios 참고)
#    - 추가로 80% 검정력에 필요한 최소검출효과(MDE)도 산출
# ============================================================
print("\n" + "=" * 70)
print("[시뮬레이션 기반 검정력/MDE] D-1A timeInBed 재현모형, n=26 기준")
print("=" * 70)

d_sim = d_tib_fit.copy()
n = len(d_sim)
resid_sd = np.sqrt(fit_tib.mse_resid)
X = pd.DataFrame({
    'const': 1.0,
    'gender2': (d_sim['gender'].astype(int) == 2).astype(float).values,
    'coverage_ratio': d_sim['coverage_ratio'].values,
    'distress_z': ((d_sim['psychological_distress_between'] - d_sim['psychological_distress_between'].mean())
                    / d_sim['psychological_distress_between'].std()).values,
})
base_params = np.array([fit_tib.params['Intercept'],
                         fit_tib.params['C(gender)[T.2]'],
                         fit_tib.params['coverage_ratio'],
                         0.0])  # true distress effect를 시나리오별로 대입

N_SIM = 3000
scenarios = {
    '삼성헬스 통합모형 크기(coef_z=0.161)': 0.161,  # 0.259는 mean_stress 원점수(1~4점) 계수이며, 표준화하면 0.161
    # 삼성헬스 노트북의 D-1A 4지표 통합모형(n=193/55)을 stress_z로 재적합해 직접 확인한 값
    # (coef=0.1606은 원점수 결과 0.2587을 스트레스 SD로 표준화한 값과 수치적으로 일치함).
    '삼성헬스 수면시간단독(coef_z=0.107)': 0.107,
    '삼성헬스 수면시간단독(C(indicator)유지, coef_z=0.142)': 0.142,
    'Fitbit 관측값(coef_z=0.025)': 0.025,
}

def simulate_power(true_beta_z):
    rejects = 0
    for _ in range(N_SIM):
        params = base_params.copy()
        params[3] = true_beta_z
        y_true = X.values @ params
        y_sim = y_true + RNG.normal(0, resid_sd, size=n)
        sim_df = d_sim.copy()
        sim_df['y_sim'] = y_sim
        sim_df['distress_z'] = X['distress_z'].values
        try:
            f = smf.ols('y_sim ~ C(gender) + coverage_ratio + distress_z', sim_df).fit()
            if f.pvalues['distress_z'] < 0.05:
                rejects += 1
        except Exception:
            pass
    return rejects / N_SIM

for label, beta in scenarios.items():
    power = simulate_power(beta)
    print(f"  [{label}] 검정력={power*100:.1f}% (시뮬레이션 {N_SIM}회 중 p<.05로 검출된 비율)")

# 해석용 MDE: 관측된 SE 기반 근사(2.8*SE ~ 80%검정력, 정규근사)
se_distress_z_equiv = fit_tib.bse['psychological_distress_between'] * d_sim['psychological_distress_between'].std()
mde_80 = 2.8 * se_distress_z_equiv
print(f"\n  (근사) 80% 검정력에 필요한 최소검출효과(MDE, 표준화 1SD 단위) ≈ {mde_80:.3f}")
# 0.259는 삼성헬스 4지표 통합모형(삼성헬스가 가진 지표 전체)의 원점수 계수이며, 표준화하면 0.161.
# Fitbit은 수면시간만 비교 가능해서 0.107~0.142(수면시간단독)가 Fitbit과 같은 지표범위의 참고치, 0.161(통합)은 삼성헬스 자체의 상한 참고치.
print(f"  (참고) 삼성헬스의 수면시간단독 효과크기는 0.107~0.142, 4지표 통합모형(삼성헬스가 가진 전체)은 0.161 "
      f"-> 이 모형(n={len(d_sim)})의 MDE({mde_80:.3f})보다 {'작음(검출 어려움)' if mde_80 > 0.161 else '비슷하거나 큼'}")

# ============================================================
# 4) 시뮬레이션 기반 검정력/MDE (D-1B 수면질 모형 기준, n=29) - Table 19 D-1B 검정력 재산출
#    D-1A와 동일한 원리로, 삼성헬스 표준화 효과크기(coef_z 0.107~0.161)를 D-1B(R_person)
#    모형에 동일하게 적용해 검출확률을 계산한다.
# ============================================================
print("\n" + "=" * 70)
print("[시뮬레이션 기반 검정력/MDE] D-1B 수면질 모형 기준, n=29")
print("=" * 70)

d_sim_b = d_b.copy()
n_b = len(d_sim_b)
resid_sd_b = np.sqrt(fit_b.mse_resid)
X_b = pd.DataFrame({
    'const': 1.0,
    'gender2': (d_sim_b['gender'].astype(int) == 2).astype(float).values,
    'coverage_ratio': d_sim_b['coverage_ratio'].values,
    'distress_z': ((d_sim_b['psychological_distress_between'] - d_sim_b['psychological_distress_between'].mean())
                    / d_sim_b['psychological_distress_between'].std()).values,
})
base_params_b = np.array([fit_b.params['Intercept'],
                           fit_b.params['C(gender)[T.2]'],
                           fit_b.params['coverage_ratio'],
                           0.0])  # true distress effect를 시나리오별로 대입

scenarios_b = {
    '삼성헬스 통합모형 크기(coef_z=0.161)': 0.161,
    '삼성헬스 수면시간단독(coef_z=0.107)': 0.107,
    '삼성헬스 수면시간단독(C(indicator)유지, coef_z=0.142)': 0.142,
}

def simulate_power_b(true_beta_z):
    rejects = 0
    for _ in range(N_SIM):
        params = base_params_b.copy()
        params[3] = true_beta_z
        y_true = X_b.values @ params
        y_sim = y_true + RNG.normal(0, resid_sd_b, size=n_b)
        sim_df = d_sim_b.copy()
        sim_df['y_sim'] = y_sim
        sim_df['distress_z'] = X_b['distress_z'].values
        try:
            f = smf.ols('y_sim ~ C(gender) + coverage_ratio + distress_z', sim_df).fit()
            if f.pvalues['distress_z'] < 0.05:
                rejects += 1
        except Exception:
            pass
    return rejects / N_SIM

for label, beta in scenarios_b.items():
    power = simulate_power_b(beta)
    print(f"  [{label}] 검정력={power*100:.1f}% (시뮬레이션 {N_SIM}회 중 p<.05로 검출된 비율)")

se_distress_z_equiv_b = fit_b.bse['psychological_distress_between'] * d_sim_b['psychological_distress_between'].std()
mde_80_b = 2.8 * se_distress_z_equiv_b
print(f"\n  (근사) 80% 검정력에 필요한 최소검출효과(MDE, 표준화 1SD 단위) ≈ {mde_80_b:.3f}")

print("\n저장 완료: d2_person_sleepquality_d1b.csv")


[D-1B 수면질 person-level 재구축] n=29 (삼성헬스의 build_person_SW와 동일 논리)

[D-1B, 삼성헬스와 동일 사양] R_person(signed) ~ coverage_ratio + C(gender) + psychological_distress_between
(삼성헬스 원본 D-1B: OLS+HC3 강건표준오차, 지표별 개별회귀 - base_code.ipynb 셀66 ols_hc3와 동일 추정방식)
(삼성헬스 계수: coef=-0.735, 95%CI[-1.255,-0.215], p=0.0056, D-1B M1-S)
n=29
                                     coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
Intercept                         -0.1861      1.557     -0.120      0.905      -3.238       2.866
C(gender)[T.2]                    -0.1139      0.559     -0.204      0.839      -1.210       0.982
coverage_ratio                     0.2639      1.529      0.173      0.863      -2.733       3.261
psychological_distress_between    -0.7251      0.600     -1.209      0.227      -1.901       0.451
-> psychological_distress_between(HC3): coef=-0.7251, 95%CI=[-1.9007,0.4505], p=0.2267

[영향

## 13단계: mq3 제외 Zs_qual/R_qual 민감도 + 참여자내 순열검정

"스트레스가 높으면 자기보고 수면질 자체가 나빠지는가"(간극이 아닌 직접효과)를 mq3 문항중첩을 통제하며 확인. 전체합성/mq3제외/mq3단독 세 구성으로 `Zs_qual`(자기보고)과 `R_qual`(signed 간극방향)을 나란히 검정하고, mq3제외 효과에 대해 참여자내 순열검정(5000회)으로 재확인한다.

In [15]:
# mq3(수면 후 개운함 없음) 문항이 자기보고 수면질(Zs_qual) 및 간극방향(R_qual) 결과를 얼마나
# 설명하는지 분리: 전체합성 vs mq3제외 합성 vs mq3단독으로 Zs_qual/R_qual 각각 재검증 +
# mq3제외 효과의 참여자내 순열검정
import statsmodels.formula.api as smf

RNG_SEED = 7777

m = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))
qual = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))[['id', 'survey_date', 'Zs_qual', 'Zw_qual', 'R_qual']]
m = m.merge(qual, on=['id', 'survey_date'], how='left')

def zscore(s):
    return (s - s.mean()) / s.std()

m['distress_full'] = pd.DataFrame({
    'mq1_z': zscore(m['mq1']), 'mq2_z': zscore(m['mq2_rev']),
    'mq3_z': zscore(m['mq3']), 'mq4_z': zscore(m['mq4']),
}).mean(axis=1)
m['distress_excl_mq3'] = pd.DataFrame({
    'mq1_z': zscore(m['mq1']), 'mq2_z': zscore(m['mq2_rev']), 'mq4_z': zscore(m['mq4']),
}).mean(axis=1)
m['mq3_only'] = zscore(m['mq3'])

def add_between_within(df, col):
    pm = df.groupby('id')[col].transform('mean')
    df[f'{col}_between'] = pm
    df[f'{col}_within'] = df[col] - pm
    return df

for col in ['distress_full', 'distress_excl_mq3', 'mq3_only']:
    m = add_between_within(m, col)

print("=" * 70)
print("[1] Zs_qual(자기보고 수면질) ~ 디스트레스(between+within), 문항구성별 비교")
print("=" * 70)
rows_zs = []
for label, col in [('전체합성(mq1-4)', 'distress_full'), ('mq3제외(mq1,2,4)', 'distress_excl_mq3'), ('mq3단독', 'mq3_only')]:
    dd = m.dropna(subset=['Zs_qual', f'{col}_between', f'{col}_within', 'wearable_coverage']).copy()
    fit = smf.mixedlm(f'Zs_qual ~ {col}_between + {col}_within + wearable_coverage', dd, groups=dd['id']).fit(reml=True)
    rows_zs.append({'구성': label, 'n': len(dd), 'n_id': dd['id'].nunique(),
                     'coef_within': fit.params[f'{col}_within'], 'p_within': fit.pvalues[f'{col}_within'],
                     'coef_between': fit.params[f'{col}_between'], 'p_between': fit.pvalues[f'{col}_between']})
    print(f"[{label}] n={len(dd)}({dd['id'].nunique()}명) "
          f"within: coef={fit.params[f'{col}_within']:.4f}, p={fit.pvalues[f'{col}_within']:.4f} | "
          f"between: coef={fit.params[f'{col}_between']:.4f}, p={fit.pvalues[f'{col}_between']:.4f}")
pd.DataFrame(rows_zs).to_csv(os.path.join(RESULT, 'd2_zs_qual_mq3_sensitivity.csv'), index=False)

print("\n" + "=" * 70)
print("[2] R_qual(=Zs-Zw, signed 간극방향) ~ 디스트레스(between+within), 문항구성별 비교")
print("(삼성헬스 D-1B M1-S: coef=-0.735, p=0.006, between-only 사양이었음 - 여기선 Fitbit 고유 within/between 분해로 참고)")
print("=" * 70)
rows_r = []
for label, col in [('전체합성(mq1-4)', 'distress_full'), ('mq3제외(mq1,2,4)', 'distress_excl_mq3'), ('mq3단독', 'mq3_only')]:
    dd = m.dropna(subset=['R_qual', f'{col}_between', f'{col}_within', 'wearable_coverage']).copy()
    fit = smf.mixedlm(f'R_qual ~ {col}_between + {col}_within + wearable_coverage', dd, groups=dd['id']).fit(reml=True)
    rows_r.append({'구성': label, 'n': len(dd), 'n_id': dd['id'].nunique(),
                    'coef_within': fit.params[f'{col}_within'], 'p_within': fit.pvalues[f'{col}_within'],
                    'coef_between': fit.params[f'{col}_between'], 'p_between': fit.pvalues[f'{col}_between']})
    print(f"[{label}] n={len(dd)}({dd['id'].nunique()}명) "
          f"within: coef={fit.params[f'{col}_within']:.4f}, p={fit.pvalues[f'{col}_within']:.4f} | "
          f"between: coef={fit.params[f'{col}_between']:.4f}, p={fit.pvalues[f'{col}_between']:.4f}")
pd.DataFrame(rows_r).to_csv(os.path.join(RESULT, 'd2_r_qual_mq3_sensitivity.csv'), index=False)

# ============================================================
# [3] mq3제외 효과의 참여자내 순열검정 (Zs_qual, R_qual 둘 다, within-effect)
# ============================================================
print("\n" + "=" * 70)
print("[3] mq3제외(distress_excl_mq3) within효과 참여자내 순열검정 (5000회)")
print("=" * 70)

N_PERM = 5000

def perm_test(outcome_col, label):
    dd = m.dropna(subset=[outcome_col, 'distress_excl_mq3_between', 'distress_excl_mq3_within', 'wearable_coverage']).copy()
    obs_fit = smf.mixedlm(f'{outcome_col} ~ distress_excl_mq3_between + distress_excl_mq3_within + wearable_coverage',
                           dd, groups=dd['id']).fit(reml=True)
    obs_coef = obs_fit.params['distress_excl_mq3_within']
    rng = np.random.default_rng(RNG_SEED)
    perm_coefs = np.zeros(N_PERM)
    for i in range(N_PERM):
        dp = dd.copy()
        dp['distress_score_perm'] = dp.groupby('id')['distress_excl_mq3'].transform(lambda s: rng.permutation(s.values))
        dp['distress_within_perm'] = dp['distress_score_perm'] - dp['distress_excl_mq3_between']
        try:
            fit_p = smf.mixedlm(f'{outcome_col} ~ distress_excl_mq3_between + distress_within_perm + wearable_coverage',
                                 dp, groups=dp['id']).fit(reml=True)
            perm_coefs[i] = fit_p.params['distress_within_perm']
        except Exception:
            perm_coefs[i] = np.nan
    valid = ~np.isnan(perm_coefs)
    perm_p = (np.abs(perm_coefs[valid]) >= np.abs(obs_coef)).mean()
    print(f"[{label}] n={len(dd)}, 관측 within coef={obs_coef:.4f}, p(parametric)={obs_fit.pvalues['distress_excl_mq3_within']:.4f}, "
          f"순열성공={valid.sum()}/{N_PERM}, p(순열)={perm_p:.4f}")
    return obs_coef, perm_p

zs_obs, zs_perm_p = perm_test('Zs_qual', 'Zs_qual ~ distress_excl_mq3(within)')
r_obs, r_perm_p = perm_test('R_qual', 'R_qual ~ distress_excl_mq3(within)')

# ============================================================
# [4] 종합 판정
# ============================================================
print("\n" + "=" * 70)
print("[종합 판정]")
print("=" * 70)
zs_full = rows_zs[0]
zs_excl = rows_zs[1]
r_full = rows_r[0]
r_excl = rows_r[1]
print(f"Zs_qual: 전체합성 within p={zs_full['p_within']:.4f} -> mq3제외 within p={zs_excl['p_within']:.4f} (순열 p={zs_perm_p:.4f})")
print(f"R_qual : 전체합성 within p={r_full['p_within']:.4f} -> mq3제외 within p={r_excl['p_within']:.4f} (순열 p={r_perm_p:.4f})")
print("""
해석 기준:
  - mq3 제외 후에도 Zs_qual 효과크기·방향 유지 -> 수면문항 내용중첩만으로 설명되지 않음
  - mq3 제외 후 Zs_qual 효과가 크게 약화 -> 결과가 문항중첩에 상당히 의존
  - Zs_qual은 유지되나 R_qual은 비유의 -> "디스트레스-자기보고 수면질" 관계는 있으나
    "자기보고-웨어러블 간극이 커진다"는 결과로는 아직 확인 안 됨(구분 필요)
""")


[1] Zs_qual(자기보고 수면질) ~ 디스트레스(between+within), 문항구성별 비교
[전체합성(mq1-4)] n=386(42명) within: coef=-0.3655, p=0.0005 | between: coef=-1.2703, p=0.0000
[mq3제외(mq1,2,4)] n=386(42명) within: coef=-0.2434, p=0.0106 | between: coef=-1.2848, p=0.0000
[mq3단독] n=386(42명) within: coef=-0.1935, p=0.0019 | between: coef=-0.6906, p=0.0160

[2] R_qual(=Zs-Zw, signed 간극방향) ~ 디스트레스(between+within), 문항구성별 비교
(삼성헬스 D-1B M1-S: coef=-0.735, p=0.006, between-only 사양이었음 - 여기선 Fitbit 고유 within/between 분해로 참고)
[전체합성(mq1-4)] n=242(34명) within: coef=-0.2579, p=0.1020 | between: coef=-0.8931, p=0.0598
[mq3제외(mq1,2,4)] n=242(34명) within: coef=-0.1293, p=0.3732 | between: coef=-0.8543, p=0.0593
[mq3단독] n=242(34명) within: coef=-0.2330, p=0.0209 | between: coef=-0.5555, p=0.1842

[3] mq3제외(distress_excl_mq3) within효과 참여자내 순열검정 (5000회)
[Zs_qual ~ distress_excl_mq3(within)] n=386, 관측 within coef=-0.2434, p(parametric)=0.0106, 순열성공=5000/5000, p(순열)=0.0254
[R_qual ~ distress_excl_mq3(within)] n=242, 관측 within coef=-0.1293, p

## 14단계: 합성점수 재표준화 + 웨어러블 매칭 동일표본 Zs·Zw·R 성분분해

전체합성과 mq3제외 합성점수는 문항 수·문항간 상관이 달라 원분산이 다르므로(SD 0.742 vs 0.792), 재표준화(SD=1) 후에만 계수를 직접 비교할 수 있다. 또한 `Zs_qual`(n=386, 웨어러블 불필요)과 `R_qual`(n=242, 웨어러블 매칭 필요)의 표본 차이가 결론에 미치는 영향을 분리하기 위해, 웨어러블 매칭 동일표본(n=242, 34명)에서 `Zs_qual`/`Zw_qual`/`R_qual` 세 모형을 동일 covariate로 나란히 재적합한다.

**최종 결론**: mq3를 제외하고 재표준화해도 전체표본(n=386, 42명)에서는 심리적 디스트레스와 자기보고 수면질(Zs_qual)의 동시적 음의 관련성이 유지됐다(within coef=-0.193, 95%CI[-0.341,-0.045], p=.0106). wearable_coverage 포함 여부로도 결론이 바뀌지 않았다(제외 coef=-0.193/p=.010, 포함 coef=-0.193/p=.011). 다만 웨어러블 매칭 동일표본(n=242, 34명)으로 좁히면 세 성분(Zs_qual/Zw_qual/R_qual) 모두 비유의했다(within p=.101/.790/.373). 즉 전체표본에서 관찰된 디스트레스-자기보고 수면질 관련성이, 웨어러블 매칭 표본으로 좁히면 통계적으로 뚜렷하지 않다 — 표본 축소에 따른 검정력 저하인지 실제 관련성이 약한 것인지는 이 분석만으로 구분하기 어렵다. "디스트레스에 따라 자기보고-웨어러블 간극이 확대된다"(R_qual)는 근거는 어느 표본에서도 확인되지 않았다.

**참고**: Zs_qual/Zw_qual/R_qual 3개 결과변수(매칭표본 n=242)는 7c단계의 8개 검정과 같은 원칙으로 family-wise BH-FDR을 적용한다. 결과: Zs_qual(p=.101→p_fdr=.302), Zw_qual(p=.790→p_fdr=.790), R_qual(p=.373→p_fdr=.560) — 매칭표본에서는 FDR 보정 전부터 이미 세 성분 모두 비유의하며, 보정 후에도 동일하다.

In [16]:
# 1) 합성점수 재표준화(SD=1) 후 계수 비교 - 전체합성 vs mq3제외를 "1SD당 효과"로 직접비교 가능하게
# 2) 웨어러블 매칭 동일표본에서 Zs_qual/Zw_qual/R_qual 세 모형을 나란히 적합
# 3) Zs_qual 모형에서 wearable_coverage 포함/제외 민감도 비교
import statsmodels.formula.api as smf


m = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))
qual = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))[['id', 'survey_date', 'Zs_qual', 'Zw_qual', 'R_qual']]
m = m.merge(qual, on=['id', 'survey_date'], how='left')

def zscore(s):
    return (s - s.mean()) / s.std()

m['distress_full_raw'] = pd.DataFrame({
    'a': zscore(m['mq1']), 'b': zscore(m['mq2_rev']), 'c': zscore(m['mq3']), 'd': zscore(m['mq4']),
}).mean(axis=1)
m['distress_excl_mq3_raw'] = pd.DataFrame({
    'a': zscore(m['mq1']), 'b': zscore(m['mq2_rev']), 'd': zscore(m['mq4']),
}).mean(axis=1)

# ============================================================
# 합성점수를 다시 SD=1로 표준화한다 (문항수·문항간 상관 차이로 인한
# 합성점수 분산 차이를 제거해야 "1단위 효과"가 두 구성 간 비교 가능해짐)
# ============================================================
m['distress_full_std'] = zscore(m['distress_full_raw'])
m['distress_excl_mq3_std'] = zscore(m['distress_excl_mq3_raw'])
print(f"재표준화 전 SD - 전체합성: {m['distress_full_raw'].std():.4f}, mq3제외: {m['distress_excl_mq3_raw'].std():.4f}")
print("(서로 다르므로 재표준화 없이 원계수를 직접 비교하면 안 됨)")

def add_bw(df, col):
    pm = df.groupby('id')[col].transform('mean')
    df[f'{col}_between'] = pm
    df[f'{col}_within'] = df[col] - pm
    return df

for col in ['distress_full_std', 'distress_excl_mq3_std']:
    m = add_bw(m, col)

print("\n" + "=" * 70)
print("[재표준화 후] Zs_qual ~ 디스트레스(1SD 단위로 통일)")
print("=" * 70)
for label, col in [('전체합성(표준화)', 'distress_full_std'), ('mq3제외(표준화)', 'distress_excl_mq3_std')]:
    dd = m.dropna(subset=['Zs_qual', f'{col}_between', f'{col}_within', 'wearable_coverage']).copy()
    fit = smf.mixedlm(f'Zs_qual ~ {col}_between + {col}_within + wearable_coverage', dd, groups=dd['id']).fit(reml=True)
    ci_w = fit.conf_int().loc[f'{col}_within'].tolist()
    print(f"[{label}] n={len(dd)} within: coef={fit.params[f'{col}_within']:.4f} 95%CI=[{ci_w[0]:.4f},{ci_w[1]:.4f}] p={fit.pvalues[f'{col}_within']:.4f}")

# ============================================================
# coverage 포함/제외 민감도 (n=386, Zs_qual은 웨어러블 불필요한데 coverage를
# 공변량으로 넣는 것이 불필요한 선택경로를 만드는지 확인)
# ============================================================
print("\n" + "=" * 70)
print("[민감도] Zs_qual 모형에서 wearable_coverage 포함/제외 비교 (mq3제외 표준화 척도)")
print("=" * 70)
dd_full270 = m.dropna(subset=['Zs_qual', 'distress_excl_mq3_std_between', 'distress_excl_mq3_std_within']).copy()
fit_nocov = smf.mixedlm('Zs_qual ~ distress_excl_mq3_std_between + distress_excl_mq3_std_within',
                         dd_full270, groups=dd_full270['id']).fit(reml=True)
print(f"[coverage 제외] n={len(dd_full270)} within: coef={fit_nocov.params['distress_excl_mq3_std_within']:.4f}, "
      f"p={fit_nocov.pvalues['distress_excl_mq3_std_within']:.4f}")
dd_full270_cov = dd_full270.dropna(subset=['wearable_coverage'])
fit_cov = smf.mixedlm('Zs_qual ~ distress_excl_mq3_std_between + distress_excl_mq3_std_within + wearable_coverage',
                       dd_full270_cov, groups=dd_full270_cov['id']).fit(reml=True)
print(f"[coverage 포함] n={len(dd_full270_cov)} within: coef={fit_cov.params['distress_excl_mq3_std_within']:.4f}, "
      f"p={fit_cov.pvalues['distress_excl_mq3_std_within']:.4f}")
print("-> 포함/제외로 결론이 달라지는지 확인용(달라지면 coverage가 선택경로를 만들 가능성 시사)")

# ============================================================
# 웨어러블 매칭 동일표본(n=242, 34명)에서 Zs_qual/Zw_qual/R_qual 나란히 비교
# 모두 동일 mq3제외 표준화 척도 + 동일 covariate(wearable_coverage) + 동일 표본
# ============================================================
print("\n" + "=" * 70)
print("[성분분해, 매칭표본 동일화] Zs_qual / Zw_qual / R_qual ~ distress_excl_mq3(표준화, between+within)")
print("모두 동일 표본(웨어러블 매칭), 동일 covariate(wearable_coverage)")
print("=" * 70)

matched = m.dropna(subset=['R_qual', 'Zs_qual', 'Zw_qual', 'distress_excl_mq3_std_between',
                            'distress_excl_mq3_std_within', 'wearable_coverage']).copy()
print(f"매칭표본: n={len(matched)}, n_id={matched['id'].nunique()}")

results = {}
for outcome in ['Zs_qual', 'Zw_qual', 'R_qual']:
    fit = smf.mixedlm(f'{outcome} ~ distress_excl_mq3_std_between + distress_excl_mq3_std_within + wearable_coverage',
                       matched, groups=matched['id']).fit(reml=True)
    ci_w = fit.conf_int().loc['distress_excl_mq3_std_within'].tolist()
    ci_b = fit.conf_int().loc['distress_excl_mq3_std_between'].tolist()
    results[outcome] = fit
    print(f"\n[{outcome}] within: coef={fit.params['distress_excl_mq3_std_within']:.4f}, "
          f"95%CI=[{ci_w[0]:.4f},{ci_w[1]:.4f}], p={fit.pvalues['distress_excl_mq3_std_within']:.4f}")
    print(f"          between: coef={fit.params['distress_excl_mq3_std_between']:.4f}, "
          f"95%CI=[{ci_b[0]:.4f},{ci_b[1]:.4f}], p={fit.pvalues['distress_excl_mq3_std_between']:.4f}")

print("\n" + "=" * 70)
print("[매칭표본 요약표]")
print("=" * 70)
summary = pd.DataFrame([
    {'결과변수': k, 'n': len(matched), 'n_id': matched['id'].nunique(),
     'within_coef': v.params['distress_excl_mq3_std_within'], 'within_p': v.pvalues['distress_excl_mq3_std_within'],
     'between_coef': v.params['distress_excl_mq3_std_between'], 'between_p': v.pvalues['distress_excl_mq3_std_between']}
    for k, v in results.items()
])
print(summary.to_string(index=False))
summary.to_csv(os.path.join(RESULT, 'd2_matched_component_decomp.csv'), index=False)

# ============================================================
# Zs_qual/Zw_qual/R_qual 3개 결과변수(동일표본)에 family-wise BH-FDR을 적용한다
# (7c단계의 8개 검정 family FDR과 같은 원칙 - 같은 표본·같은 시점에 나란히 비교하는
# 3개 결과변수는 하나의 검정군으로 보정해야 함)
# ============================================================
from statsmodels.stats.multitest import multipletests
within_ps = [v.pvalues['distress_excl_mq3_std_within'] for v in results.values()]
rej, p_fdr, _, _ = multipletests(within_ps, method='fdr_bh')
summary['within_p_fdr'] = p_fdr
summary['within_reject_fdr'] = rej
print("\n" + "=" * 70)
print("[매칭표본 3개 결과변수 family-wise BH-FDR (within효과 기준)]")
print("=" * 70)
print(summary[['결과변수', 'within_coef', 'within_p', 'within_p_fdr', 'within_reject_fdr']].to_string(index=False))
summary.to_csv(os.path.join(RESULT, 'd2_matched_component_decomp.csv'), index=False)
print("\n저장 완료: d2_matched_component_decomp.csv (FDR 열 추가)")


재표준화 전 SD - 전체합성: 0.7415, mq3제외: 0.7921
(서로 다르므로 재표준화 없이 원계수를 직접 비교하면 안 됨)

[재표준화 후] Zs_qual ~ 디스트레스(1SD 단위로 통일)
[전체합성(표준화)] n=386 within: coef=-0.2710 95%CI=[-0.4238,-0.1182] p=0.0005
[mq3제외(표준화)] n=386 within: coef=-0.1928 95%CI=[-0.3406,-0.0450] p=0.0106

[민감도] Zs_qual 모형에서 wearable_coverage 포함/제외 비교 (mq3제외 표준화 척도)
[coverage 제외] n=386 within: coef=-0.1931, p=0.0103
[coverage 포함] n=386 within: coef=-0.1928, p=0.0106
-> 포함/제외로 결론이 달라지는지 확인용(달라지면 coverage가 선택경로를 만들 가능성 시사)

[성분분해, 매칭표본 동일화] Zs_qual / Zw_qual / R_qual ~ distress_excl_mq3(표준화, between+within)
모두 동일 표본(웨어러블 매칭), 동일 covariate(wearable_coverage)
매칭표본: n=242, n_id=34

[Zs_qual] within: coef=-0.1234, 95%CI=[-0.2709,0.0240], p=0.1008
          between: coef=-0.6508, 95%CI=[-1.0672,-0.2345], p=0.0022

[Zw_qual] within: coef=-0.0232, 95%CI=[-0.1943,0.1478], p=0.7901
          between: coef=0.0219, 95%CI=[-0.4532,0.4970], p=0.9280

[R_qual] within: coef=-0.1024, 95%CI=[-0.3279,0.1230], p=0.3732
          between: coef=-0.6768, 95

## 15단계: 삼성헬스 시간분리 파이프라인의 수면시간 한정 재적용

삼성헬스의 Analysis2(초기 1-2주 스트레스가 후기 3-4주 간극을 예측하는지, LOPO+순열검정 p=0.002)를 Fitbit의 timeInBed 기반 수면시간 간극 하나로 동일 절차 재적용. "삼성헬스 통합효과의 완전한 재현"이 아니라 "수면시간 한정 재적용"으로만 명명한다. Fitbit은 참여자별 응답 주차수가 불규칙해 그 사람 응답의 시간순 전반부/후반부로 초기/후기를 대체(각 반부 최소 2주 이상, n=27명).

In [17]:
# "삼성헬스 시간분리 파이프라인의 수면시간 한정 재적용" (삼성헬스 통합효과의 완전한 재현이 아님)
# 삼성헬스의 시간분리 분석(초기 스트레스가 후기 간극을 예측하는지, LOPO+순열검정)을
# Fitbit의 timeInBed 기반 수면시간 간극 하나만으로 동일 절차 재적용.
# - 삼성헬스는 4주 고정설계라 week 1-2=초기/3-4=후기로 나눴지만, Fitbit은 참여자마다 응답 주차수가
#   3~15주로 불규칙해 "그 사람 응답을 시간순 절반으로 나눈 전반부/후반부"로 대체함(명시적 한계).
# - match_quality 가중평균 대신 Fitbit의 wearable_coverage로 가중(동일 역할의 대응 변수).
# - canonical SD(D)는 8단계 D-1A에서 이미 확립한 값(전체기간 union-pooled 기준)을 재사용.
import statsmodels.formula.api as smf

MIN_PER_HALF = 2  # 삼성헬스의 "2주씩 고정"에 대응하는 최소 관측 기준(전반부/후반부 각각)

# ============================================================
# 1) 데이터 준비: person-week 수면시간 D(timeInBed 기반) + 스트레스(mq1/mq2/mq4 3문항 합성, mq3 제외 - d2_stress_gap_pw.csv의 stress_score를 그대로 사용)
# ============================================================
m = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_tib_pw.csv'))
m['survey_date'] = pd.to_datetime(m['survey_date'])
stress = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))[['id', 'survey_date', 'stress_score']]
stress['survey_date'] = pd.to_datetime(stress['survey_date'])
m = m.merge(stress, on=['id', 'survey_date'], how='left')

# 8단계 D-1A에서 이미 확립한 canonical SD(D_tib) 재사용(같은 표준화 상수, 삼성헬스의 "정본 SD" 원칙과 동일)
person_tib = pd.read_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib_unionpooled.csv'))
# person-level 평균 D의 표준편차로 SD_CANON을 계산(11단계의 sd_new와 같은 값)
valid_full = m.dropna(subset=['D_sleep_hr_tib'])
person_mean_D_full = valid_full.groupby('id')['D_sleep_hr_tib'].mean()
SD_CANON = person_mean_D_full.std()
print(f"canonical SD(D_tib) 재사용 = {SD_CANON:.4f} (8단계 D-1A와 동일 상수)")

# ============================================================
# 2) 참여자별 전반부/후반부 분리 (시간순 응답의 절반) - 삼성헬스의 week1-2/3-4 대응
# ============================================================
def split_early_late(g):
    g = g.sort_values('survey_date').reset_index(drop=True)
    n = len(g)
    half = (n + 1) // 2  # 홀수면 전반부에 1개 더
    return g.iloc[:half], g.iloc[half:]

rows = []
for pid, g in m.groupby('id'):
    g_valid = g.dropna(subset=['D_sleep_hr_tib'])
    if len(g_valid) < MIN_PER_HALF * 2:
        continue
    early, late = split_early_late(g_valid)
    if len(early) < MIN_PER_HALF or len(late) < MIN_PER_HALF:
        continue

    def weighted_D(sub):
        w = sub['wearable_coverage']
        if w.sum() <= 0:
            return np.nan
        return (sub['D_sleep_hr_tib'] * w).sum() / w.sum()

    D_early = weighted_D(early)
    D_late = weighted_D(late)
    stress_early = early['stress_score'].mean()
    coverage_late = late['wearable_coverage'].mean()
    n_early, n_late = len(early), len(late)

    rows.append({
        'id': pid, 'n_early': n_early, 'n_late': n_late,
        'gap_early': abs(D_early) / SD_CANON, 'gap_late': abs(D_late) / SD_CANON,
        'stress_early': stress_early, 'coverage_late': coverage_late,
    })

frame = pd.DataFrame(rows).dropna(subset=['gap_late', 'gap_early', 'stress_early', 'coverage_late'])
print(f"\n최종 분석표본: n={len(frame)}명 (전반부/후반부 각 {MIN_PER_HALF}주 이상 관측)")
print(frame[['id', 'n_early', 'n_late', 'gap_early', 'gap_late', 'stress_early', 'coverage_late']].to_string(index=False))

if len(frame) < 8:
    print("\n표본이 너무 작아(< 8명) 회귀모형 적합이 불안정할 수 있음 - 결과 해석에 유의")

# ============================================================
# 3) 주모형: gap_late ~ stress_early + coverage_late + gap_early (OLS+HC3, 삼성헬스와 동일 사양)
# ============================================================
print("\n" + "=" * 70)
print("gap_late(timeInBed) ~ stress_early + coverage_late + gap_early")
print("(삼성헬스: 4지표 통합 gap, LOPO+순열p=0.002 - 여기선 수면시간 단독)")
print("=" * 70)
model_full = smf.ols("gap_late ~ stress_early + coverage_late + gap_early", frame).fit(cov_type='HC3')
print(model_full.summary().tables[1])
coef = model_full.params['stress_early']
ci = model_full.conf_int().loc['stress_early'].tolist()
p = model_full.pvalues['stress_early']
print(f"-> stress_early: coef={coef:.4f}, 95%CI=[{ci[0]:.4f},{ci[1]:.4f}], p={p:.4f}")

model_base = smf.ols("gap_late ~ coverage_late + gap_early", frame).fit(cov_type='HC3')

# ============================================================
# 4) Leave-one-person-out 교차검증: stress_early 추가가 예측력을 높이는지
# ============================================================
def loo_predict(formula, data):
    preds = np.full(len(data), np.nan)
    for i, idx in enumerate(data.index):
        train = data.drop(idx)
        test = data.loc[[idx]]
        try:
            r = smf.ols(formula, train).fit()
            preds[i] = r.predict(test).values[0]
        except Exception:
            preds[i] = np.nan
    return preds

frame = frame.reset_index(drop=True)
pred_full = loo_predict("gap_late ~ stress_early + coverage_late + gap_early", frame)
pred_base = loo_predict("gap_late ~ coverage_late + gap_early", frame)

valid = ~np.isnan(pred_full) & ~np.isnan(pred_base)
actual = frame.loc[valid, 'gap_late'].values
mse_full = np.mean((actual - pred_full[valid]) ** 2)
mse_base = np.mean((actual - pred_base[valid]) ** 2)
corr_full = np.corrcoef(actual, pred_full[valid])[0, 1] if valid.sum() > 2 else np.nan
corr_base = np.corrcoef(actual, pred_base[valid])[0, 1] if valid.sum() > 2 else np.nan

print("\n" + "=" * 70)
print("Leave-one-person-out 교차검증 (stress_early 추가 여부 비교)")
print("=" * 70)
print(f"[Full: +stress_early] OOF MSE={mse_full:.4f}, corr(actual,pred)={corr_full:.4f}")
print(f"[Base: -stress_early] OOF MSE={mse_base:.4f}, corr(actual,pred)={corr_base:.4f}")
print(f"MSE 개선(음수=개선): {mse_full - mse_base:.4f}")

# ============================================================
# 5) 참여자단위 순열검정 (삼성헬스와 동일: stress_early를 참여자 간 재배정, 2000회)
# ============================================================
rng = np.random.default_rng(3)
n_perm = 2000
obs_coef = model_full.params['stress_early']
perm_coefs = []
for _ in range(n_perm):
    shuffled = frame.copy()
    shuffled['stress_early'] = rng.permutation(shuffled['stress_early'].values)
    try:
        r = smf.ols("gap_late ~ stress_early + coverage_late + gap_early", shuffled).fit()
        perm_coefs.append(r.params['stress_early'])
    except Exception:
        continue
perm_coefs = np.array(perm_coefs)
pctile = (perm_coefs < obs_coef).mean() * 100
perm_p = 2 * min(pctile, 100 - pctile) / 100
print(f"\n순열검정({len(perm_coefs)}회): 실제 stress_early 계수={obs_coef:.4f}, "
      f"순열분포 내 위치={pctile:.1f}%ile, 양측 순열p={perm_p:.4f}")

# ============================================================
# 저장
# ============================================================
out = pd.DataFrame([{
    '분석명': '삼성헬스 시간분리 파이프라인의 수면시간 한정 재적용',
    'n': len(frame), 'coef_stress_early': coef, 'ci_lo': ci[0], 'ci_hi': ci[1], 'p': p,
    'perm_p': perm_p, 'mse_full': mse_full, 'mse_base': mse_base,
    'corr_full': corr_full, 'corr_base': corr_base,
}])
out.to_csv(os.path.join(RESULT, 'd2_time_separated_sleeptime.csv'), index=False)
frame.to_csv(os.path.join(RESULT, 'd2_time_separated_sleeptime_frame.csv'), index=False)
print("\n저장 완료: d2_time_separated_sleeptime.csv, d2_time_separated_sleeptime_frame.csv")


canonical SD(D_tib) 재사용 = 1.2802 (8단계 D-1A와 동일 상수)

최종 분석표본: n=27명 (전반부/후반부 각 2주 이상 관측)
  id  n_early  n_late  gap_early  gap_late  stress_early  coverage_late
S001        5       5   1.919789  2.883378      0.300747       1.000000
S008        3       3   0.361475  0.055328     -0.509480       0.761905
S012        3       2   0.543228  0.455640     -0.520732       0.857143
S013        6       5   0.306961  0.220319     -0.971786       1.000000
S016        3       3   0.218095  0.916134      0.859505       0.904762
S019        5       5   0.080775  0.023619     -0.200071       1.000000
S030        3       2   0.104038  0.188765      0.060578       1.000000
S040        5       4   0.345419  0.411541      1.406273       1.000000
S041        7       7   0.156575  1.056244     -0.600498       1.000000
S042        4       3   0.024247  0.950335     -0.539231       0.809524
S069        3       3   0.630664  0.381870     -0.219882       1.000000
S119        5       5   0.153721  0.589967     -

## 16단계: Fitbit within/between 모형 적용

삼성헬스의 within/between 스트레스 분해 + 시간효과 모형을 Fitbit(불규칙 3~15주 응답)에 적용한다.
달력주차(week 1~4)가 없으므로 참여자별 **순차 응답 순서**(`rel_week`, 7주차 이상은 "7+"로 묶음)를
삼성헬스의 week 인덱스 대용으로 사용한다.

**D-1A(수면시간 간극) 최소요건은 11단계와 정확히 동일하게 맞춘다**:
- `n_weeks`는 `survey_sleep_hr`가 유효한 주차 수만 카운트(웨어러블 존재 여부와 무관) → `n_weeks>=3`
- 웨어러블 적정성은 별도로 canonical `coverage_ratio>=0.5`(참여자 union-day 기준)로만 체크
- 두 조건을 모두 만족하는 사람만 포함(11단계와 동일한 2단계 필터, n=26)

In [18]:
MIN_WEEKS = 3
COVERAGE_FLOOR = 0.5

df_tib = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_tib_pw.csv'))
df_tib['survey_date'] = pd.to_datetime(df_tib['survey_date'])

# 11단계와 동일한 n_weeks 정의: survey_sleep_hr notna 주차 수(웨어러블 무관)
n_weeks_survey = df_tib.dropna(subset=['survey_sleep_hr']).groupby('id').size()
eligible_by_nweeks = n_weeks_survey[n_weeks_survey >= MIN_WEEKS].index
has_any_wearable = df_tib.dropna(subset=['D_sleep_hr_tib']).groupby('id').size()
eligible_by_wearable = has_any_wearable[has_any_wearable > 0].index
person_main_nofloor_ids = set(eligible_by_nweeks) & set(eligible_by_wearable)

canon_cov = pd.read_csv(os.path.join(RESULT, 'd2_canonical_coverage.csv'))[['id', 'coverage_ratio']]
cov_map = canon_cov.set_index('id')['coverage_ratio'].to_dict()
final_ids_d1a = sorted([i for i in person_main_nofloor_ids if cov_map.get(i, 0) >= COVERAGE_FLOOR])

print(f"n_weeks>=3: {len(eligible_by_nweeks)}명, 웨어러블 매칭 존재: {len(eligible_by_wearable)}명")
print(f"coverage_ratio>=0.5 최종: {len(final_ids_d1a)}명 (11단계와 동일해야 함)")

d_d1a = df_tib[df_tib['id'].isin(final_ids_d1a)].copy()

# rel_week: 참여자별 전체 설문응답(survey_sleep_hr notna) 순서, 7+ 상한
d_survey = df_tib[df_tib['id'].isin(final_ids_d1a) & df_tib['survey_sleep_hr'].notna()].sort_values(['id', 'survey_date']).copy()
d_survey['rel_week_raw'] = d_survey.groupby('id').cumcount() + 1
rel_map = d_survey.set_index(['id', 'survey_date'])['rel_week_raw'].to_dict()
d_d1a['rel_week_raw'] = [rel_map.get((i, sd), np.nan) for i, sd in zip(d_d1a['id'], d_d1a['survey_date'])]
d_d1a['rel_week'] = d_d1a['rel_week_raw'].apply(lambda x: '7+' if pd.notna(x) and x >= 7 else (str(int(x)) if pd.notna(x) else np.nan))

# 모형용 관측치: D_sleep_hr_tib 유효 주차만
dm_d1a = d_d1a.dropna(subset=['D_sleep_hr_tib', 'rel_week']).copy()
dm_d1a['SD_t'] = dm_d1a.groupby('rel_week')['D_sleep_hr_tib'].transform('std')
dm_d1a['D_star_t'] = dm_d1a['D_sleep_hr_tib'] / dm_d1a['SD_t']
dm_d1a['absD_star_t'] = dm_d1a['D_star_t'].abs()

stress_pw02 = pd.read_csv(os.path.join(RESULT, 'd2_stress_gap_pw.csv'))
stress_pw02['survey_date'] = pd.to_datetime(stress_pw02['survey_date'])
dm_d1a = dm_d1a.merge(stress_pw02[['id', 'survey_date', 'stress_between', 'stress_within']], on=['id', 'survey_date'], how='left')

dm_d1a['coverage_between'] = dm_d1a.groupby('id')['wearable_coverage'].transform('mean')
dm_d1a['coverage_within'] = dm_d1a['wearable_coverage'] - dm_d1a['coverage_between']

survey_pw_g36 = pd.read_csv(os.path.join(DATA, 'fitbit_survey_person_week.csv'))
gender_df02 = survey_pw_g36[['id', 'gender']].dropna().drop_duplicates(subset=['id'])
dm_d1a = dm_d1a.merge(gender_df02, on='id', how='left')

dm_d1a_m = dm_d1a.dropna(subset=['absD_star_t', 'stress_between', 'stress_within', 'coverage_between', 'coverage_within', 'gender']).copy()
dm_d1a_m['gender'] = dm_d1a_m['gender'].astype(int).astype('category')

print(f"\n모형 표본: {len(dm_d1a_m)}행, {dm_d1a_m['id'].nunique()}명")
print(dm_d1a_m['rel_week'].value_counts())

formula_fb_d1a = 'absD_star_t ~ stress_within + stress_between + C(gender) + coverage_within + coverage_between + C(rel_week)'
fit_fb_d1a = smf.mixedlm(formula_fb_d1a, dm_d1a_m, groups=dm_d1a_m['id']).fit(reml=True)
print(fit_fb_d1a.summary())

dm_d1a_m.to_csv(os.path.join(RESULT, 'd3_fitbit_d1a_model_data.csv'), index=False)

n_weeks>=3: 42명, 웨어러블 매칭 존재: 34명
coverage_ratio>=0.5 최종: 26명 (11단계와 동일해야 함)

모형 표본: 223행, 26명
rel_week
7+    84
2     25
3     25
4     24
5     24
6     22
1     19
Name: count, dtype: int64
           Mixed Linear Model Regression Results
Model:             MixedLM  Dependent Variable:  absD_star_t
No. Observations:  223      Method:              REML       
No. Groups:        26       Scale:               0.2960     
Min. group size:   3        Log-Likelihood:      -207.6023  
Max. group size:   14       Converged:           Yes        
Mean group size:   8.6                                      
------------------------------------------------------------
                  Coef.  Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept         -0.885    1.932 -0.458 0.647 -4.672  2.902
C(gender)[T.2]     0.105    0.196  0.534 0.594 -0.279  0.489
C(rel_week)[T.2]   0.085    0.183  0.463 0.644 -0.273  0.443
C(rel_week)[T.3]   0.021   

## 16-보완: daily mood person-week 집계 (Table 10 재현 원천, D-1B mood 경로용)

일별 daily mood 원문항(`data/fitbit_daily_mood.csv`, 1~5점, 높을수록 긍정)을 스트레스 방향에 맞춰 `6-mood`로 역채점한 뒤, 각 person-week의 [D-7,D-1] 윈도우 내 응답을 평균/표준편차로 집계한다(mood_mean/mood_sd).

In [19]:
mw_mood = pd.read_csv(os.path.join(RESULT, 'd2_merged_person_week.csv'))
mw_mood['survey_date'] = pd.to_datetime(mw_mood['survey_date'])

daily_mood_raw = pd.read_csv(os.path.join(DATA, 'fitbit_daily_mood.csv'))
daily_mood_raw['date'] = pd.to_datetime(daily_mood_raw['date'])
daily_mood_raw['stress_proxy'] = 6 - daily_mood_raw['mood']

mood_rows = []
for sid, daily in daily_mood_raw.groupby('id'):
    daily_idx = daily.set_index('date')['stress_proxy']
    sub = mw_mood[mw_mood['id'] == sid]
    for _, r in sub.iterrows():
        window = [(r['survey_date'] - pd.Timedelta(days=d)).normalize() for d in range(1, 8)]
        present = daily_idx.reindex(window).dropna()
        if len(present) == 0:
            mood_rows.append({'id': sid, 'survey_date': r['survey_date'],
                               'mood_n': 0, 'mood_mean': np.nan, 'mood_sd': np.nan,
                               'mood_min': np.nan, 'mood_max': np.nan})
            continue
        mood_rows.append({
            'id': sid, 'survey_date': r['survey_date'],
            'mood_n': len(present),
            'mood_mean': present.mean(),
            'mood_sd': present.std() if len(present) > 1 else np.nan,
            'mood_min': present.min(),
            'mood_max': present.max(),
        })

mood_pw = pd.DataFrame(mood_rows)
n_zero = (mood_pw['mood_n'] == 0).sum()
print(f"person-week {len(mood_pw)}건 중 그 주 mood 응답 0건: {n_zero}건({n_zero/len(mood_pw)*100:.1f}%)")
print(f"주당 mood 응답일수 분포:\n{mood_pw['mood_n'].describe()}")

mood_pw.to_csv(os.path.join(RESULT, 'd2_mood_stress_pw.csv'), index=False)
print("저장 완료: d2_mood_stress_pw.csv")


person-week 386건 중 그 주 mood 응답 0건: 5건(1.3%)
주당 mood 응답일수 분포:
count    386.000000
mean       6.018135
std        1.710831
min        0.000000
25%        6.000000
50%        7.000000
75%        7.000000
max        7.000000
Name: mood_n, dtype: float64
저장 완료: d2_mood_stress_pw.csv


## 16-2단계: Fitbit D-1B(수면의 질) within/between - 스트레스, 데일리무드 두 경로

D-1B는 12단계와 동일하게
`Squal`과 `w_mean_sleep_score`가 모두 유효한 주차 수 `n_weeks>=3`만 포함(별도 coverage floor 없음).
$Z$표준화는 `rel_week`(참여자별 순차 응답순서, 7+ 상한) 그룹 내 횡단면 기준으로 계산한다.

In [20]:
df_qual = pd.read_csv(os.path.join(RESULT, 'd2_sleep_gap_pw.csv'))
df_qual['survey_date'] = pd.to_datetime(df_qual['survey_date'])

valid_qual = df_qual.dropna(subset=['Squal', 'w_mean_sleep_score'])
pcount_qual = valid_qual.groupby('id').size()
final_ids_d1b = sorted(pcount_qual[pcount_qual >= MIN_WEEKS].index)
print(f"D-1B 최소요건(Squal & wearable 매칭 주차>=3) 충족: {len(final_ids_d1b)}명 (12단계와 동일해야 함, n=29)")

d_d1b = df_qual[df_qual['id'].isin(final_ids_d1b) & df_qual['Squal'].notna() & df_qual['w_mean_sleep_score'].notna()].copy()
d_d1b = d_d1b.sort_values(['id', 'survey_date'])
d_d1b['rel_week_raw'] = d_d1b.groupby('id').cumcount() + 1
d_d1b['rel_week_cap'] = d_d1b['rel_week_raw'].apply(lambda x: '7+' if x >= 7 else str(int(x)))

d_d1b['Zs_t'] = d_d1b.groupby('rel_week_cap')['Squal'].transform(lambda x: (x - x.mean()) / x.std())
d_d1b['Zw_t'] = d_d1b.groupby('rel_week_cap')['w_mean_sleep_score'].transform(lambda x: (x - x.mean()) / x.std())
d_d1b['R_t'] = d_d1b['Zs_t'] - d_d1b['Zw_t']
d_d1b['absR_t'] = d_d1b['R_t'].abs()

d_d1b['coverage_between'] = d_d1b.groupby('id')['wearable_coverage'].transform('mean')
d_d1b['coverage_within'] = d_d1b['wearable_coverage'] - d_d1b['coverage_between']

survey_pw_g38 = pd.read_csv(os.path.join(DATA, 'fitbit_survey_person_week.csv'))
gender_df_d1b = survey_pw_g38[['id', 'gender']].dropna().drop_duplicates(subset=['id'])
d_d1b = d_d1b.merge(gender_df_d1b, on='id', how='left')

# (a) 스트레스 within/between 경로
d_d1b_stress = d_d1b.merge(stress_pw02[['id', 'survey_date', 'stress_between', 'stress_within']], on=['id', 'survey_date'], how='left')
dm_d1b_stress = d_d1b_stress.dropna(subset=['R_t', 'stress_between', 'stress_within', 'coverage_between', 'coverage_within', 'gender']).copy()
dm_d1b_stress['gender'] = dm_d1b_stress['gender'].astype(int).astype('category')
print(f"\n[스트레스 경로] 모형 표본: {len(dm_d1b_stress)}행, {dm_d1b_stress['id'].nunique()}명")

formula_fb_d1b_stress = 'R_t ~ stress_within + stress_between + C(gender) + coverage_within + coverage_between + C(rel_week_cap)'
fit_fb_d1b_stress_R = smf.mixedlm(formula_fb_d1b_stress, dm_d1b_stress, groups=dm_d1b_stress['id']).fit(reml=True)
print('--- R_t(방향, 스트레스) ---'); print(fit_fb_d1b_stress_R.summary())
fit_fb_d1b_stress_abs = smf.mixedlm(formula_fb_d1b_stress.replace('R_t', 'absR_t'), dm_d1b_stress, groups=dm_d1b_stress['id']).fit(reml=True)
print('--- absR_t(크기, 스트레스) ---'); print(fit_fb_d1b_stress_abs.summary())

# (b) 데일리무드 within/between 경로
mood_pw = pd.read_csv(os.path.join(RESULT, 'd2_mood_stress_pw.csv'))
mood_pw['survey_date'] = pd.to_datetime(mood_pw['survey_date'])
mood_pw['mood_between'] = mood_pw.groupby('id')['mood_mean'].transform('mean')
mood_pw['mood_within'] = mood_pw['mood_mean'] - mood_pw['mood_between']
d_d1b_mood = d_d1b.merge(mood_pw[['id', 'survey_date', 'mood_between', 'mood_within']], on=['id', 'survey_date'], how='left')
dm_d1b_mood = d_d1b_mood.dropna(subset=['R_t', 'mood_between', 'mood_within', 'coverage_between', 'coverage_within', 'gender']).copy()
dm_d1b_mood['gender'] = dm_d1b_mood['gender'].astype(int).astype('category')
print(f"\n[데일리무드 경로] 모형 표본: {len(dm_d1b_mood)}행, {dm_d1b_mood['id'].nunique()}명")

formula_fb_d1b_mood = 'R_t ~ mood_within + mood_between + C(gender) + coverage_within + coverage_between + C(rel_week_cap)'
fit_fb_d1b_mood_R = smf.mixedlm(formula_fb_d1b_mood, dm_d1b_mood, groups=dm_d1b_mood['id']).fit(reml=True)
print('--- R_t(방향, 무드) ---'); print(fit_fb_d1b_mood_R.summary())
fit_fb_d1b_mood_abs = smf.mixedlm(formula_fb_d1b_mood.replace('R_t', 'absR_t'), dm_d1b_mood, groups=dm_d1b_mood['id']).fit(reml=True)
print('--- absR_t(크기, 무드) ---'); print(fit_fb_d1b_mood_abs.summary())

dm_d1b_stress.to_csv(os.path.join(RESULT, 'd3_fitbit_d1b_stress_model_data.csv'), index=False)
dm_d1b_mood.to_csv(os.path.join(RESULT, 'd3_fitbit_d1b_mood_model_data.csv'), index=False)

D-1B 최소요건(Squal & wearable 매칭 주차>=3) 충족: 29명 (12단계와 동일해야 함, n=29)

[스트레스 경로] 모형 표본: 237행, 29명
--- R_t(방향, 스트레스) ---
             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    R_t      
No. Observations:    237        Method:                REML     
No. Groups:          29         Scale:                 0.7679   
Min. group size:     3          Log-Likelihood:        -345.2877
Max. group size:     14         Converged:             Yes      
Mean group size:     8.2                                        
----------------------------------------------------------------
                      Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------
Intercept              1.285    3.035  0.423 0.672 -4.663  7.233
C(gender)[T.2]        -0.114    0.550 -0.208 0.835 -1.191  0.963
C(rel_week_cap)[T.2]  -0.114    0.248 -0.460 0.645 -0.601  0.372
C(rel_week_cap)[T.3]  -0.104    0.242 -0.432 0.665 -0

## 16-2-보완: daily mood 변동성(mood_sd) within/between 관계 (Table 18)

위 셀에서 만든 `d_d1b`(D-1B 수면질 방향성 간극 R_t, n=29)에 daily mood의 person-week 내 **변동성**(mood_sd, cell 16-보완에서 이미 계산됨)을 within/between으로 분해해 결합한다. 수준(mood_mean) 경로는 바로 위 셀에서 이미 확인했으므로, 여기서는 변동성 경로만 추가로 본다.

In [21]:
mood_pw_vol = pd.read_csv(os.path.join(RESULT, 'd2_mood_stress_pw.csv'))
mood_pw_vol['survey_date'] = pd.to_datetime(mood_pw_vol['survey_date'])
mood_pw_vol['mood_sd_between'] = mood_pw_vol.groupby('id')['mood_sd'].transform('mean')
mood_pw_vol['mood_sd_within'] = mood_pw_vol['mood_sd'] - mood_pw_vol['mood_sd_between']

d_d1b_vol = d_d1b.merge(mood_pw_vol[['id', 'survey_date', 'mood_sd_between', 'mood_sd_within']], on=['id', 'survey_date'], how='left')
dm_d1b_vol = d_d1b_vol.dropna(subset=['R_t', 'mood_sd_between', 'mood_sd_within', 'coverage_between', 'coverage_within', 'gender']).copy()
dm_d1b_vol['gender'] = dm_d1b_vol['gender'].astype(int).astype('category')
print(f"[데일리무드 변동성 경로] 모형 표본: {len(dm_d1b_vol)}행, {dm_d1b_vol['id'].nunique()}명")

formula_fb_d1b_vol = 'R_t ~ mood_sd_within + mood_sd_between + C(gender) + coverage_within + coverage_between + C(rel_week_cap)'
fit_fb_d1b_vol_R = smf.mixedlm(formula_fb_d1b_vol, dm_d1b_vol, groups=dm_d1b_vol['id']).fit(reml=True)
print('--- R_t(방향, 무드 변동성) ---'); print(fit_fb_d1b_vol_R.summary())

dm_d1b_vol.to_csv(os.path.join(RESULT, 'd3_fitbit_d1b_moodsd_model_data.csv'), index=False)


[데일리무드 변동성 경로] 모형 표본: 231행, 29명
--- R_t(방향, 무드 변동성) ---
             Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    R_t      
No. Observations:    231        Method:                REML     
No. Groups:          29         Scale:                 0.7710   
Min. group size:     3          Log-Likelihood:        -334.5360
Max. group size:     14         Converged:             Yes      
Mean group size:     8.0                                        
----------------------------------------------------------------
                      Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------------
Intercept              0.900    2.843  0.317 0.751 -4.672  6.472
C(gender)[T.2]        -0.130    0.498 -0.261 0.794 -1.106  0.846
C(rel_week_cap)[T.2]  -0.037    0.257 -0.144 0.886 -0.540  0.466
C(rel_week_cap)[T.3]  -0.095    0.249 -0.383 0.702 -0.583  0.392
C(rel_week_cap)[T.4]   0.071    0.252  0.281 0.7

## 16-2-보완 2: daily mood 분포(Table 10) + 참여자별 평균 기반 mood 수준·변동성 분석 (3.4.2 추가 탐색)

일별 응답 분포와 person-week 집계 산출 건수를 확인하고(Table 10), 참여자별 평균 mood 수준(mood_mean)과 변동성(mood_sd)을 표준화하여 D-1A(n=26)·D-1B(n=29) 간극에 각각 회귀한다(OLS, HC3, 공변량 coverage_ratio·gender). 이 분석은 참여자별 평균값을 사용하므로 개인 간 차이(between-person)를 반영한다. 2(mood 수준/변동성)×2(D-1A/D-1B) 4개 검정을 하나의 검정군으로 BH-FDR 보정한다.

In [24]:
from statsmodels.stats.multitest import multipletests

# ---- Table 10: 일별 mood 응답 분포 및 person-week 집계 산출 건수 ----
daily_raw = pd.read_csv(os.path.join(DATA, 'fitbit_daily_mood.csv'))
n_daily = len(daily_raw)
dist = {k: (daily_raw['mood'] == k).sum() / n_daily * 100 for k in [1, 2, 3, 4, 5]}
print(f"mood: n={n_daily}(일별 응답, {daily_raw['id'].nunique()}명) | " + " / ".join(f"{k}점 {v:.1f}" for k, v in dist.items()))
n_frac = (daily_raw['mood'] % 1 != 0).sum()
print(f"(정수 1~5점 외 소수값 응답 {n_frac}건은 위 분포에서 제외)")

mood_pw_t10 = pd.read_csv(os.path.join(RESULT, 'd2_mood_stress_pw.csv'))
print(f"person-week {len(mood_pw_t10)}건 중 mood_mean 산출 {mood_pw_t10['mood_mean'].notna().sum()}건, mood_sd 산출 {mood_pw_t10['mood_sd'].notna().sum()}건")

# ---- 참여자별 평균 mood 수준/변동성 -> D-1A, D-1B (between-person) ----
per_mood = mood_pw_t10.groupby('id').agg(mood_level=('mood_mean', 'mean'), mood_var=('mood_sd', 'mean')).reset_index()

d1a_m = pd.read_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib_unionpooled.csv')).drop(columns=['psychological_distress_between'], errors='ignore').merge(per_mood, on='id')
d1b_m = pd.read_csv(os.path.join(RESULT, 'd2_person_sleepquality_d1b.csv')).drop(columns=['psychological_distress_between'], errors='ignore').merge(per_mood, on='id')

tests = [('D-1A(간극 크기)', d1a_m, 'absD_star', 'mood_level'), ('D-1B(간극 방향)', d1b_m, 'R_person', 'mood_level'),
         ('D-1A(간극 크기)', d1a_m, 'absD_star', 'mood_var'),   ('D-1B(간극 방향)', d1b_m, 'R_person', 'mood_var')]
rows = []
for label, df, dv, x in tests:
    d = df.dropna(subset=[dv, x, 'coverage_ratio', 'gender']).copy()
    d['gender'] = d['gender'].astype(int).astype('category')
    d['x_z'] = (d[x] - d[x].mean()) / d[x].std()
    fit = smf.ols(f'{dv} ~ coverage_ratio + C(gender) + x_z', d).fit(cov_type='HC3')
    ci = fit.conf_int().loc['x_z']
    rows.append({'분석': label, 'mood 지표': '수준(평균)' if x == 'mood_level' else '변동성(SD)', 'n': len(d),
                 'coef': fit.params['x_z'], 'ci_lo': ci[0], 'ci_hi': ci[1], 'p': fit.pvalues['x_z']})
res_mood = pd.DataFrame(rows)
res_mood['q_fdr'] = multipletests(res_mood['p'], method='fdr_bh')[1]
print("\n[참여자별 평균 mood -> 간극 (1SD당, OLS+HC3), 4개 검정 BH-FDR]")
print(res_mood.round(4).to_string(index=False))


mood: n=2165(일별 응답, 42명) | 1점 2.8 / 2점 8.0 / 3점 19.1 / 4점 38.7 / 5점 29.9
(정수 1~5점 외 소수값 응답 33건은 위 분포에서 제외)
person-week 386건 중 mood_mean 산출 381건, mood_sd 산출 375건

[참여자별 평균 mood -> 간극 (1SD당, OLS+HC3), 4개 검정 BH-FDR]
         분석 mood 지표  n    coef   ci_lo   ci_hi      p  q_fdr
D-1A(간극 크기)  수준(평균) 26 -0.0097 -0.2230  0.2037 0.9291 0.9291
D-1B(간극 방향)  수준(평균) 29 -0.2733 -1.4176  0.8709 0.6397 0.8529
D-1A(간극 크기) 변동성(SD) 26  0.1895 -0.2139  0.5929 0.3573 0.7145
D-1B(간극 방향) 변동성(SD) 29 -0.7785 -1.3600 -0.1971 0.0087 0.0347


## 16-3단계: 참여자단위 부트스트랩 강건성 확인 (Fitbit)

삼성헬스/Fitbit 전반에 걸쳐 동일한 관례(300회 목표, 성공률<50%면 cluster-robust 대체)를 적용한다.

In [22]:
def boot_fb(data, formula, groups_col, target_terms, n_target=300, max_attempts=500, seed=1):
    ids = data[groups_col].unique()
    rng = np.random.default_rng(seed)
    results = {t: [] for t in target_terms}
    attempts = 0
    while len(results[target_terms[0]]) < n_target and attempts < max_attempts:
        attempts += 1
        samp_ids = rng.choice(ids, size=len(ids), replace=True)
        parts = []
        for i, sid in enumerate(samp_ids):
            sub = data[data[groups_col] == sid].copy()
            sub['_bid'] = f'{sid}_{i}'
            parts.append(sub)
        dboot = pd.concat(parts, ignore_index=True)
        try:
            f = smf.mixedlm(formula, dboot, groups=dboot['_bid']).fit(reml=True)
            for t in target_terms:
                results[t].append(f.params[t])
        except Exception:
            continue
    sr = len(results[target_terms[0]]) / attempts if attempts > 0 else 0
    print(f'  시도{attempts}회, 성공{len(results[target_terms[0]])}회 (성공률 {sr*100:.1f}%)')
    if sr < 0.5:
        print('  성공률<50% -> cluster-robust 대체 필요')
        return None
    for t in target_terms:
        print(f'  {t} 95%CI:', np.percentile(results[t], [2.5, 97.5]))
    return results

print('=== D-1A(수면시간 간극, n=26) 부트스트랩 ===')
_ = boot_fb(dm_d1a_m, formula_fb_d1a, 'id', ['stress_within', 'stress_between'])

print('\n=== D-1B 스트레스 경로(R_t, n=29) 부트스트랩 ===')
_ = boot_fb(dm_d1b_stress, formula_fb_d1b_stress, 'id', ['stress_within', 'stress_between'])

print('\n=== D-1B 무드 경로(R_t, n=29) 부트스트랩 ===')
_ = boot_fb(dm_d1b_mood, formula_fb_d1b_mood, 'id', ['mood_within', 'mood_between'])

print('\n=== D-1B 무드 변동성 경로(R_t, n=29) 부트스트랩 ===')
_ = boot_fb(dm_d1b_vol, formula_fb_d1b_vol, 'id', ['mood_sd_within', 'mood_sd_between'])

=== D-1A(수면시간 간극, n=26) 부트스트랩 ===


  시도302회, 성공300회 (성공률 99.3%)
  stress_within 95%CI: [-0.038632    0.41779797]
  stress_between 95%CI: [-0.14848066  0.4212357 ]

=== D-1B 스트레스 경로(R_t, n=29) 부트스트랩 ===


  시도300회, 성공300회 (성공률 100.0%)
  stress_within 95%CI: [-0.58471221  0.19894383]
  stress_between 95%CI: [-1.47820675  0.32350275]

=== D-1B 무드 경로(R_t, n=29) 부트스트랩 ===


  시도300회, 성공300회 (성공률 100.0%)
  mood_within 95%CI: [-0.6811374  -0.02055444]
  mood_between 95%CI: [-1.50500056  0.77210992]

=== D-1B 무드 변동성 경로(R_t, n=29) 부트스트랩 ===
  시도300회, 성공300회 (성공률 100.0%)
  mood_sd_within 95%CI: [-0.47174226  0.11563626]
  mood_sd_between 95%CI: [-3.29147543 -0.47050824]


### 16단계 계열 종합 해석 (논문 3.4.2/Table 18)

수면의 질(D-1B) 모형(n=29)에서는 GHQ 기반 디스트레스의 within-person 효과(βW=−0.195, 95% CI [−0.585, 0.199], P=.108)와 between-person 효과(βB=−0.624, 95% CI [−1.478, 0.324], P=.147)가 삼성헬스와 같은 방향을 보였으나 유의하지 않았다. 수면시간(D-1A) 모형(n=26)에서는 between-person 효과가 삼성헬스와 같은 양의 방향이었으나 유의하지 않았고(Fitbit βB=0.110, P=.497; 삼성헬스 수면시간 2지표 모형 βB=0.161, P=.017), within-person 효과는 Fitbit에서만 양의 방향으로 모형 기반 검정에서 유의했다(βW=0.160, P=.034; 참여자단위 부트스트랩 95% CI [−0.039, 0.418]). 삼성헬스의 수면시간 within-person 효과는 음의 방향이며 비유의(βW=−0.058, P=.521; 삼성헬스 노트북 D-3 4단계)여서 방향이 달라 재현으로 해석하지 않는다. P값은 혼합모형의 모형 기반 값이고 95% CI는 참여자단위 부트스트랩 백분위수이며 다중검정 보정 전 값이다. 또한 **보조지표인 daily mood 수준(평균)의 within-person 관계는 참여자단위 부트스트랩에서도 유지되었다(βW=−0.324, 95% CI[−0.681,−0.021], P=.045)** — 같은 참여자라도 평소보다 기분이 나쁜(스트레스 방향) 주에는 자기보고 수면질이 웨어러블 대응지표보다 상대적으로 더 낮게 보고되는 경향으로, 삼성헬스의 within-person 발견(D-1B, βW=−0.319)의 재현이 아니라 Fitbit에서만 수행한 사후 보조분석의 신호로서 개인 내 관련성의 가능성을 시사한다.

**대조**: 바로 위 16-2-보완 셀에서 확인했듯, 같은 within/between 분해를 daily mood **변동성**(mood_sd)에 적용하면 양상이 반대로 나타난다 — between-person 관계는 유의했으나(βB=−1.990, 95%CI[−3.291,−0.471], P=.014) within-person 관계는 비유의했다(βW=−0.223, 95%CI[−0.472,0.116], P=.253). 즉 기분 **수준**의 신호는 개인 내 시간적 변동(state)에서, 기분 **변동성**의 신호는 개인 간 차이(trait)에서 나타나는 대조적 패턴이다.

## 최종: Table 16/17 (삼성헬스-Fitbit 표준화 스트레스-간극 계수 직접비교)

**참고**: Table 15는 Step4~6의 Fitbit 측정방법 일치도 표를 가리키는 별개의 표이며, 이 셀은 Table 16(삼성헬스-Fitbit 표준화 계수 비교)·Table 17(D-1B 성분분해)의 원천이다.

`d2_person_sleeptime_tib_unionpooled.csv`(n=26, D-1A)와 `d2_person_sleepquality_d1b.csv`(n=29, D-1B) — 표본 수가 논문과 정확히 일치하는 person-level 데이터 —를 사용해 삼성헬스와 동일한 방식(HC3 강건표준오차, distress를 Z-score로 표준화)으로 재적합한다.

In [23]:
import pandas as pd
import statsmodels.formula.api as smf

d1a_final = pd.read_csv(os.path.join(RESULT, 'd2_person_sleeptime_tib_unionpooled.csv'))
d1a_final = d1a_final.dropna(subset=['absD_star', 'coverage_ratio', 'gender', 'psychological_distress_between']).copy()
d1a_final['gender'] = d1a_final['gender'].astype(int).astype('category')
d1a_final['distress_z'] = (d1a_final['psychological_distress_between'] - d1a_final['psychological_distress_between'].mean()) / d1a_final['psychological_distress_between'].std()

res_d1a = smf.ols('absD_star ~ coverage_ratio + C(gender) + distress_z', d1a_final).fit(cov_type='HC3')
ci_d1a = res_d1a.conf_int().loc['distress_z']
print(f"[D-1A, n={len(d1a_final)}] distress_z: coef={res_d1a.params['distress_z']:.4f}, "
      f"95%CI=[{ci_d1a[0]:.4f},{ci_d1a[1]:.4f}], p={res_d1a.pvalues['distress_z']:.4f}")

d1b_final = pd.read_csv(os.path.join(RESULT, 'd2_person_sleepquality_d1b.csv'))
d1b_final = d1b_final.dropna(subset=['R_person', 'S', 'W', 'coverage_ratio', 'gender', 'psychological_distress_between']).copy()
d1b_final['gender'] = d1b_final['gender'].astype(int).astype('category')
d1b_final['distress_z'] = (d1b_final['psychological_distress_between'] - d1b_final['psychological_distress_between'].mean()) / d1b_final['psychological_distress_between'].std()

res_d1b = smf.ols('R_person ~ coverage_ratio + C(gender) + distress_z', d1b_final).fit(cov_type='HC3')
ci_d1b = res_d1b.conf_int().loc['distress_z']
print(f"[D-1B, n={len(d1b_final)}] distress_z: coef={res_d1b.params['distress_z']:.4f}, "
      f"95%CI=[{ci_d1b[0]:.4f},{ci_d1b[1]:.4f}], p={res_d1b.pvalues['distress_z']:.4f}")

# Table 17: D-1B 간극의 자기보고(Zs)·웨어러블(Zw) 성분분해
d1b_final['Zs'] = (d1b_final['S'] - d1b_final['S'].mean()) / d1b_final['S'].std()
d1b_final['Zw'] = (d1b_final['W'] - d1b_final['W'].mean()) / d1b_final['W'].std()
for dv in ['Zs', 'Zw']:
    res = smf.ols(f'{dv} ~ coverage_ratio + C(gender) + distress_z', d1b_final).fit(cov_type='HC3')
    ci = res.conf_int().loc['distress_z']
    print(f"[{dv} 성분, n={len(d1b_final)}] distress_z: coef={res.params['distress_z']:.4f}, "
          f"95%CI=[{ci[0]:.4f},{ci[1]:.4f}], p={res.pvalues['distress_z']:.4f}")


[D-1A, n=26] distress_z: coef=0.0671, 95%CI=[-0.3746,0.5089], p=0.7658
[D-1B, n=29] distress_z: coef=-0.4646, 95%CI=[-1.2179,0.2887], p=0.2267
[Zs 성분, n=29] distress_z: coef=-0.4646, 95%CI=[-0.8436,-0.0857], p=0.0162
[Zw 성분, n=29] distress_z: coef=-0.0000, 95%CI=[-0.4744,0.4743], p=0.9999
